# Colab-Free Fall Detection Research Pipeline

**Objective:** Build a scientifically valid, reproducible fall/non-fall classifier from frozen pose estimation while staying within a strict Google Colab Free/T4 budget.

This notebook is designed to get as much scientific value as possible from each GPU minute. It does not assume that large backbones, exhaustive hyperparameter searches, or repeated video inference are worth the extra cost.

## Default experiment

**Train:** MCFD + GMDCSA24 v2.1

**External test:** CAUCAFall v5 / Zenodo repack cache

**Pose:** YOLO11n-Pose, frozen, COCO 17 keypoints

**Representation:** `[T=32, V=17, C=8]` = position, confidence, velocity, acceleration, speed

**Model:** lightweight confidence-aware ST-GCN + TCN + 2-head temporal attention

**Baselines:** tiny MLP and GRU

**Evaluation:** within-domain subject-aware validation, MCFD cross-view shift, external CAUCAFall, and hard-negative analysis.

> **Important:** this notebook never fabricates performance or runtime. Every runtime figure is measured on the actual Colab session and stored in `metadata/run_metadata.json`.

## 1. Scientific audit and design decisions

### MCFD
The Multiple Cameras Fall Dataset contains 24 scenarios captured from eight cameras. The literature consistently describes one subject and 24 scenarios, with confounding daily activities. The dataset is therefore useful for **cross-view/view-shift evaluation**, but it cannot support a strong subject-independent generalization claim because the subject diversity is extremely limited. The implementation labels MCFD results explicitly as **cross-view / view-shift**, never as subject-independent generalization.

### GMDCSA24
The verified v2.1 Zenodo release contains 81 falls and 79 ADL clips from four actors across three home environments. Its CSV files contain class intervals, and the v2.1 release specifically added class start/end timing to the CSV metadata. This makes it useful for subject-aware splitting and hard-negative analysis.

### CAUCAFall
CAUCAFall stays completely external in the main experiment. Version 5 was published on Mendeley Data in 2025, contains 10 subjects and 5 fall + 5 ADL activities, and was recorded under uncontrolled lighting/occlusion/environmental variation. For Colab portability, the notebook caches a versioned Zenodo binary repack while documenting the original Mendeley provenance separately. A repack is **not** treated as equivalent to the original source or as a change to its license.

### Removed from the default run
EDF/OCCU/UP-Fall are not downloaded by default. OOPS-1k is not part of the default training run; it remains an optional OOD stress-test stage. This keeps the first Colab-Free run from turning into a storage, download, and pose-extraction exercise with little extra evidence for the main scientific question.

### Architecture audit
A large RGB video model or video Transformer is rejected on compute and experimental-design grounds. A frozen lightweight pose estimator removes most trainable visual compute, while the proposed model focuses on motion and body-graph structure. ST-GCN and TCN are treated as established components, not as novelty by themselves. Confidence is kept as an explicit signal because low-confidence pose observations should not be treated as equally reliable geometry.

### Provenance used by the default notebook

| Dataset | Binary source used by notebook | Exact version / record | Archive checksum recorded | Scientific source |
|---|---|---|---|---|
| MCFD | Zenodo versioned repack | Zenodo `17170592`, `MCFD.zip` | MD5 `b365b5a4b16691fce3fc09ccd554df79` | Auvinet et al.; Université de Montréal original source |
| GMDCSA24 | Official Zenodo release | v2.1, DOI `10.5281/zenodo.13354453` | MD5 `3d36f2c5c1a666b99639e4e9fd843efb` | Alam et al., *Data in Brief* |
| CAUCAFall | Zenodo versioned repack | Zenodo `17170592`, `Cauca_fall.zip` | MD5 `5a1899fe0022c9c4c0d37ce64ac0c986` | Original Mendeley Data v5, DOI `10.17632/7w7fccy7ky.5` |

The checksum and file size are checked from the metadata/source where available. The notebook **does not hard-code a universal “dataset size” claim**; it records the actual archive size in bytes observed in the Colab cache.

External source records:
- MCFD original laboratory dataset: `https://www.iro.umontreal.ca/~labimage/Dataset/`
- GMDCSA24 v2.1: `https://doi.org/10.5281/zenodo.13354453`
- CAUCAFall v5: `https://data.mendeley.com/datasets/7w7fccy7ky/5`
- Colab-friendly MCFD/CAUCAFall binary repack: `https://doi.org/10.5281/zenodo.17170592`

In [14]:
# ============================================================
# Fast, Colab-safe setup and imports
#
# Note:
# - Do not downgrade/upgrade NumPy, pandas, PyTorch, torchvision,
#   OpenCV, or other core Colab packages.
# - Colab already ships a working scientific/PyTorch stack.
# - Only install Ultralytics when it is actually missing or too old.
# - YOLO11n-Pose remains the same research component.
# ============================================================

import sys
import subprocess
import importlib
import importlib.metadata as importlib_metadata
import warnings
import random
import os
import re
import gc
import io
import json
import csv
import math
import time
import hashlib
import shutil
import zipfile
import platform

from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict


# ------------------------------------------------------------
# Core-stack policy
# ------------------------------------------------------------

REQUIRED_ULTRALYTICS = "8.3.176"
POSE_MODEL_NAME = "yolo11n-pose.pt"


def _version_tuple(v):
    m = re.match(r"^(\d+)\.(\d+)\.(\d+)", str(v))
    return tuple(map(int, m.groups())) if m else (0, 0, 0)


def _pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def _pip_install(requirements):
    """
    Install only the requested package(s).
    Do not force-reinstall the Colab scientific/CUDA stack.
    """
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "--prefer-binary",
        "--no-input",
    ]
    cmd.extend(requirements)
    subprocess.check_call(cmd)


# ------------------------------------------------------------
# Install Ultralytics only when needed
# ------------------------------------------------------------

ultra_ver = _pkg_version("ultralytics")

if ultra_ver is None:
    print(
        f"[INSTALL] ultralytics=={REQUIRED_ULTRALYTICS} "
        "(missing; core Colab stack untouched)"
    )
    _pip_install([
        f"ultralytics=={REQUIRED_ULTRALYTICS}"
    ])

elif _version_tuple(ultra_ver) < _version_tuple(REQUIRED_ULTRALYTICS):
    print(
        f"[REPAIR] ultralytics {ultra_ver} is older than "
        f"{REQUIRED_ULTRALYTICS}; updating only Ultralytics."
    )
    _pip_install([
        f"ultralytics=={REQUIRED_ULTRALYTICS}"
    ])

else:
    print(
        f"[CACHE HIT] ultralytics {ultra_ver} satisfies the minimum; "
        "no pip reinstall."
    )


# ------------------------------------------------------------
# Safe import helper
# ------------------------------------------------------------

def _import_or_fail(module_name, package_name=None):
    try:
        return importlib.import_module(module_name)
    except Exception as e:
        package_name = package_name or module_name.split(".")[0]

        raise RuntimeError(
            f"Failed to import {module_name!r} after the Colab-safe setup.\n"
            f"The notebook did not modify NumPy/pandas/PyTorch.\n"
            f"Package={package_name!r}\n"
            f"Error={type(e).__name__}: {e}"
        ) from e


# ------------------------------------------------------------
# Scientific stack
# Note: import this as-is from Colab
# ------------------------------------------------------------

np = _import_or_fail("numpy")
pd = _import_or_fail("pandas")


# ------------------------------------------------------------
# Common runtime libraries
# ------------------------------------------------------------

plt = _import_or_fail("matplotlib.pyplot", "matplotlib")

_import_or_fail("tqdm.auto", "tqdm")
from tqdm.auto import tqdm

cv2 = _import_or_fail("cv2", "opencv-python")
requests = _import_or_fail("requests")
psutil = _import_or_fail("psutil")

torch = _import_or_fail("torch")

import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)


# ------------------------------------------------------------
# sklearn metrics
# ------------------------------------------------------------

sklearn_metrics = _import_or_fail(
    "sklearn.metrics",
    "scikit-learn",
)

accuracy_score = sklearn_metrics.accuracy_score
precision_score = sklearn_metrics.precision_score
recall_score = sklearn_metrics.recall_score
f1_score = sklearn_metrics.f1_score
confusion_matrix = sklearn_metrics.confusion_matrix
roc_auc_score = sklearn_metrics.roc_auc_score
average_precision_score = sklearn_metrics.average_precision_score


# ------------------------------------------------------------
# Ultralytics
# ------------------------------------------------------------

ultralytics = _import_or_fail("ultralytics")

from ultralytics import YOLO


# ------------------------------------------------------------
# huggingface_hub
# Used later for OmniFall metadata retrieval
# ------------------------------------------------------------

hf_version = _pkg_version("huggingface-hub")

if hf_version is None:
    print(
        "[INSTALL] huggingface_hub missing; "
        "installing only this lightweight dependency."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "huggingface_hub>=0.24,<1",
    ])


# ------------------------------------------------------------
# Warnings
# ------------------------------------------------------------

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 20260903

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Fixed-shape workload:
# benchmarked cuDNN kernels improve throughput.
# This does not claim bitwise determinism.

torch.backends.cudnn.benchmark = True


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

torch_device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# Hard operational VRAM budget
# ------------------------------------------------------------

MAX_OPERATIONAL_VRAM_GIB = 13.5


def assert_vram_budget(stage="stage"):
    if not torch.cuda.is_available():
        return

    peak = (
        torch.cuda.max_memory_allocated()
        / 2**30
    )

    if peak > MAX_OPERATIONAL_VRAM_GIB:
        raise RuntimeError(
            f"GPU memory budget exceeded during {stage}: "
            f"peak allocated {peak:.2f} GiB > "
            f"{MAX_OPERATIONAL_VRAM_GIB:.2f} GiB."
        )


# ------------------------------------------------------------
# Environment report
# ------------------------------------------------------------

print("=" * 70)
print("COLAB-SAFE RESEARCH ENVIRONMENT")
print("=" * 70)

print(
    "Python      :",
    sys.version.split()[0],
)

print(
    "NumPy       :",
    np.__version__,
)

print(
    "Pandas      :",
    pd.__version__,
)

print(
    "OpenCV      :",
    cv2.__version__,
)

print(
    "PyTorch     :",
    torch.__version__,
)

print(
    "Ultralytics :",
    ultralytics.__version__,
)

print(
    "Device      :",
    torch_device,
)


if torch.cuda.is_available():

    props = torch.cuda.get_device_properties(0)

    print(
        "GPU         :",
        props.name,
    )

    print(
        "VRAM total  :",
        round(
            props.total_memory / 2**30,
            2,
        ),
        "GiB",
    )

    print(
        "VRAM budget :",
        MAX_OPERATIONAL_VRAM_GIB,
        "GiB allocated peak",
    )

else:

    print(
        "WARNING     : CUDA GPU is unavailable; "
        "CPU execution may be substantially slower."
    )


# ------------------------------------------------------------
# Final compatibility check
# ------------------------------------------------------------

if _version_tuple(ultralytics.__version__) < _version_tuple(
    REQUIRED_ULTRALYTICS
):
    raise RuntimeError(
        f"Ultralytics {ultralytics.__version__} is too old "
        f"for the YOLO11n-Pose experiment. "
        f"Required minimum: {REQUIRED_ULTRALYTICS}."
    )


print("=" * 70)
print(
    "Runtime ready — NumPy/pandas/PyTorch/OpenCV "
    "were NOT force-reinstalled."
)
print("=" * 70)

[CACHE HIT] ultralytics 8.3.176 satisfies the minimum; no pip reinstall.
COLAB-SAFE RESEARCH ENVIRONMENT
Python      : 3.13.15
NumPy       : 2.1.3
Pandas      : 2.2.3
OpenCV      : 5.0.0
PyTorch     : 2.11.0+cu128
Ultralytics : 8.3.176
Device      : cuda
GPU         : Tesla T4
VRAM total  : 14.56 GiB
VRAM budget : 13.5 GiB allocated peak
Runtime ready — NumPy/pandas/PyTorch/OpenCV were NOT force-reinstalled.


In [15]:
# Mount Google Drive and create the persistent project structure.
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/drive/MyDrive/fall_detection_project')
DIRS = {
    'datasets_zips': PROJECT_ROOT/'datasets_zips',
    'pose_cache': PROJECT_ROOT/'pose_cache',
    'metadata': PROJECT_ROOT/'metadata',
    'results': PROJECT_ROOT/'results',
    'models': PROJECT_ROOT/'models',
    'logs': PROJECT_ROOT/'logs',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
TEMP_ROOT = Path('/content/fall_detection_work')
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/fall_detection_project


## 2. Runtime controller: 3-hour target, 4-hour hard ceiling

The budget controller is cumulative within the persistent project directory. It is checked before and during expensive stages. Optional experiments are skipped once the 3-hour target is exceeded; essential stages may continue until the **4-hour hard ceiling**. Pose extraction checkpoints are written video-by-video so an interrupted session can resume from cached NPZ files.

In [16]:
# Persistent cumulative runtime manager.
class BudgetStop(RuntimeError):
    pass

@dataclass
class BudgetState:
    seed: int
    hard_limit_s: float = 3.83*3600
    target_limit_s: float = 3*3600
    cumulative_completed_s: float = 0.0
    session_start_epoch: float = 0.0
    last_checkpoint_epoch: float = 0.0
    stopped_reason: str = ""

BUDGET_FILE = DIRS['metadata']/'budget_state.json'
RUN_COMPLETE_FILE = DIRS['metadata']/'run_complete.json'

class RuntimeBudget:
    def __init__(self):
        # A completed run should not consume the next run's wall-clock budget.
        # Cached datasets, poses, and checkpoints stay reusable; only the cumulative timer resets.
        if RUN_COMPLETE_FILE.exists():
            try:
                d=json.loads(RUN_COMPLETE_FILE.read_text())
                archived=BUDGET_FILE.with_name(f"budget_state_completed_{d.get('timestamp_utc','unknown').replace(':','').replace('-','')}.json")
                if BUDGET_FILE.exists():
                    shutil.copy2(BUDGET_FILE, archived)
                RUN_COMPLETE_FILE.unlink(missing_ok=True)
            except Exception as e:
                print(f"[WARN] Could not reset completed-run budget state cleanly: {e!r}")
        if BUDGET_FILE.exists():
            try:
                d=json.loads(BUDGET_FILE.read_text())
                self.state=BudgetState(**d)
            except Exception:
                self.state=BudgetState(seed=SEED)
        else:
            self.state=BudgetState(seed=SEED)
        self.session_start=time.time()
        self.state.session_start_epoch=self.session_start
        self.state.last_checkpoint_epoch=self.session_start
        self.save()

    @property
    def elapsed_total(self):
        return self.state.cumulative_completed_s + (time.time()-self.session_start)
    @property
    def remaining_hard(self):
        return self.state.hard_limit_s-self.elapsed_total
    @property
    def target_exceeded(self):
        return self.elapsed_total >= self.state.target_limit_s

    def save(self, reason=""):
        d=asdict(self.state)
        d['elapsed_total_s']=self.elapsed_total
        d['target_exceeded']=self.target_exceeded
        d['remaining_hard_s']=self.remaining_hard
        if reason: d['last_reason']=reason
        BUDGET_FILE.write_text(json.dumps(d, indent=2))

    def checkpoint(self, reason=""):
        self.state.cumulative_completed_s += max(0.0, time.time()-self.session_start)
        self.session_start=time.time()
        self.state.session_start_epoch=self.session_start
        self.state.last_checkpoint_epoch=self.session_start
        self.save(reason)

    def log_decision(self, stage, action, details=None):
        rec={'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
             'stage':stage,'action':action,'essential':bool(getattr(self,'_last_essential',False))}
        if details: rec.update(details)
        with open(BUDGET_DECISIONS_FILE,'a',encoding='utf-8') as f:
            f.write(json.dumps(rec,default=str)+'\n')

    def guard(self, stage="stage", essential=True):
        self._last_essential=essential
        if self.remaining_hard <= 0:
            self.state.stopped_reason=f"Hard 4-hour ceiling reached during {stage}."
            self.log_decision(stage,'hard_stop',{'remaining_hard_s':self.remaining_hard})
            self.save(stage)
            raise BudgetStop(self.state.stopped_reason)
        if (not essential) and self.target_exceeded:
            self.log_decision(stage,'optional_skip',{'elapsed_total_s':self.elapsed_total,'target_limit_s':self.state.target_limit_s})
            return False
        return True

    def estimate_and_gate(self, stage, estimated_s, essential=True):
        if self.remaining_hard < estimated_s + 60:
            if essential:
                self.log_decision(stage,'estimated_hard_stop',{'estimated_s':estimated_s,'remaining_hard_s':self.remaining_hard})
                raise BudgetStop(f"Estimated {estimated_s/60:.1f} min for essential stage '{stage}' does not fit remaining hard budget ({self.remaining_hard/60:.1f} min).")
            self.log_decision(stage,'estimated_optional_skip',{'estimated_s':estimated_s,'remaining_hard_s':self.remaining_hard})
            print(f"[BUDGET] Skipping optional stage '{stage}': estimate={estimated_s/60:.1f} min, remaining={self.remaining_hard/60:.1f} min")
            return False
        return self.guard(stage, essential=essential)

BUDGET=RuntimeBudget()
TIMING_FILE=DIRS['metadata']/'stage_timings.csv'
BUDGET_DECISIONS_FILE=DIRS['metadata']/'budget_decisions.jsonl'

class StageTimer:
    def __init__(self,name,estimated_s=None,essential=True):
        self.name=name; self.estimated_s=estimated_s; self.essential=essential; self.t0=None
    def __enter__(self):
        BUDGET.guard(self.name, essential=self.essential)
        self.t0=time.time(); return self
    def __exit__(self, exc_type, exc, tb):
        elapsed=time.time()-self.t0
        row=pd.DataFrame([{'stage':self.name,'elapsed_seconds':elapsed,'estimated_seconds':self.estimated_s,'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'essential':self.essential,'exception':repr(exc) if exc else ''}])
        row.to_csv(TIMING_FILE,mode='a',header=not TIMING_FILE.exists(),index=False)
        BUDGET.checkpoint(self.name)
        return False

print(f"Cumulative budget used: {BUDGET.elapsed_total/60:.1f} min | target: 180 min | hard ceiling: 230 min")

Cumulative budget used: 0.0 min | target: 180 min | hard ceiling: 230 min


## 3. Dataset configuration and verified-download helpers

The notebook resolves Zenodo file metadata at runtime, compares the remote checksum/name with the expected values below, and stores the exact ZIP under `datasets_zips/`. A matching ZIP is never downloaded again. A partial `.part` file can be resumed.

In [17]:
DATASETS = {
    'MCFD': {
        'record_id': '17170592',
        'filename': 'MCFD.zip',
        'expected_md5': 'b365b5a4b16691fce3fc09ccd554df79',
        'binary_provenance': 'Zenodo repack of MCFD',
        'original_source': 'https://www.iro.umontreal.ca/~labimage/Dataset/',
        'citation': 'Auvinet et al., Multiple cameras fall dataset, DIRO, Université de Montréal.',
        'license_note': 'Check the original MCFD distribution terms. The Zenodo repack is a distribution convenience, not a new license grant.',
    },
    'GMDCSA24': {
        'record_id': '13354453',
        'filename': 'ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip',
        'expected_md5': '3d36f2c5c1a666b99639e4e9fd843efb',
        'binary_provenance': 'Official Zenodo v2.1 release',
        'original_source': 'https://github.com/ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos',
        'citation': 'Alam, Sufian, Dutta, Leo & Hameed, GMDCSA-24, Data in Brief, 2024.',
        'license_note': 'Zenodo/OpenAIRE metadata indicate CC BY for the dataset; verify archive-level rights before redistribution.',
    },
    'CAUCAFall': {
        'record_id': '17170592',
        'filename': 'Cauca_fall.zip',
        'expected_md5': '5a1899fe0022c9c4c0d37ce64ac0c986',
        'binary_provenance': 'Versioned Zenodo repack used only for Colab portability',
        'original_source': 'https://data.mendeley.com/datasets/7w7fccy7ky/5',
        'citation': 'Eraso, Muñoz, Muñoz & Pinto, Dataset CAUCAFall, Mendeley Data v5, 2025.',
        'license_note': 'Original Mendeley v5 is CC BY 4.0. Repack provenance must still be cited separately.',
    },
}

def md5_file(path, chunk_size=8*1024*1024):
    h=hashlib.md5()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def resolve_zenodo_file(record_id, filename):
    url=f'https://zenodo.org/api/records/{record_id}'
    r=requests.get(url, timeout=30)
    r.raise_for_status()
    rec=r.json()
    candidates=rec.get('files', [])
    wanted=filename.split('/')[-1]
    for item in candidates:
        key=item.get('key','')
        if key==filename or Path(key).name==wanted:
            links=item.get('links',{})
            return {
                'record_id': record_id,
                'key': key,
                'size': item.get('size'),
                'checksum': item.get('checksum'),
                'download': links.get('self') or links.get('content'),
                'record_title': rec.get('metadata',{}).get('title'),
                'record_version': rec.get('metadata',{}).get('version'),
            }
    raise FileNotFoundError(f"File {filename} not found in Zenodo record {record_id}.")

def safe_download(url, dst, expected_md5=None, max_minutes=70):
    dst=Path(dst); part=dst.with_suffix(dst.suffix+'.part')
    started=time.time()
    existing=part.stat().st_size if part.exists() else 0
    headers={'Range':f'bytes={existing}-'} if existing else {}
    with requests.get(url, headers=headers, stream=True, timeout=(30,60)) as r:
        if existing and r.status_code==200:
            # The server ignored Range; restart safely.
            existing=0; part.unlink(missing_ok=True)
            r.close()
            with requests.get(url, stream=True, timeout=(30,60)) as rr:
                rr.raise_for_status()
                mode='wb'
                with open(part,mode) as f:
                    for chunk in rr.iter_content(chunk_size=8*1024*1024):
                        if chunk:
                            f.write(chunk)
                        if time.time()-started > max_minutes*60:
                            raise BudgetStop(f"Download exceeded per-stage safety limit ({max_minutes} min). Partial file kept at {part}")
        else:
            r.raise_for_status(); mode='ab' if existing else 'wb'
            with open(part,mode) as f:
                for chunk in r.iter_content(chunk_size=8*1024*1024):
                    if chunk:
                        f.write(chunk)
                    if time.time()-started > max_minutes*60:
                        raise BudgetStop(f"Download exceeded per-stage safety limit ({max_minutes} min). Partial file kept at {part}")
    os.replace(part, dst)
    if expected_md5:
        got=md5_file(dst)
        if got.lower()!=expected_md5.lower():
            dst.unlink(missing_ok=True)
            raise IOError(f"Checksum mismatch for {dst.name}: got {got}, expected {expected_md5}")
    return dst

def ensure_dataset_zip(name, max_minutes=70):
    cfg=DATASETS[name]
    dst=DIRS['datasets_zips']/Path(cfg['filename']).name
    meta_path=DIRS['metadata']/f'{name}_source_metadata.json'
    remote=resolve_zenodo_file(cfg['record_id'], cfg['filename'])
    remote_md5=(remote.get('checksum') or '').split(':')[-1]
    expected=cfg['expected_md5']
    if remote_md5 and remote_md5.lower()!=expected.lower():
        raise RuntimeError(f"Source integrity audit failed for {name}: Zenodo reports {remote_md5}, notebook expects {expected}.")
    if dst.exists():
        got=md5_file(dst)
        if got.lower()==expected.lower():
            info={**cfg, **remote, 'local_path':str(dst), 'local_bytes':dst.stat().st_size, 'local_md5':got, 'verified':True}
            meta_path.write_text(json.dumps(info, indent=2, default=str))
            print(f"[CACHE HIT] {name}: {dst.name} ({dst.stat().st_size/2**30:.2f} GiB)")
            return dst
        print(f"[CACHE INVALID] {name}: removing checksum-mismatched archive.")
        dst.unlink()
    print(f"[DOWNLOAD] {name}: {remote['key']} ({(remote.get('size') or 0)/2**30:.2f} GiB)")
    path=safe_download(remote['download'], dst, expected_md5=expected, max_minutes=max_minutes)
    info={**cfg, **remote, 'local_path':str(path), 'local_bytes':path.stat().st_size, 'local_md5':expected, 'verified':True}
    meta_path.write_text(json.dumps(info, indent=2, default=str))
    return path

In [18]:
# ============================================================
# Dataset selection and real download progress
# ============================================================
#
# Features:
#   - One real progress bar per dataset
#   - Progress measured in BYTES, not fake step counts
#   - Download speed
#   - ETA
#   - Total file size
#   - Resume interrupted downloads using .part files
#   - Detect servers that ignore HTTP Range
#   - MD5 verification after download
#   - Existing verified cache is reused
#   - Does not load the whole archive into RAM
#   - Preserves the original scientific train/external-test policy
#   - Respects the runtime budget
# ============================================================

import os
import time
import json
import hashlib
from pathlib import Path

import requests
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Dataset policy
# ------------------------------------------------------------

# Default data list.
# Set OPTIONAL_OOPS=False unless explicitly running the OOD stress test.

ENABLE_MCFD = True
ENABLE_GMDCSA24 = True
ENABLE_CAUCAFALL = True
ENABLE_OPTIONAL_OOPS = False


# Training datasets
train_dataset_names = [
    name
    for name, enabled in [
        ("MCFD", ENABLE_MCFD),
        ("GMDCSA24", ENABLE_GMDCSA24),
    ]
    if enabled
]


# External datasets are never added to training.
external_dataset_names = (
    ["CAUCAFall"]
    if ENABLE_CAUCAFALL
    else []
)


all_download_names = (
    train_dataset_names +
    external_dataset_names
)


print("=" * 70)
print("DATASET DOWNLOAD PLAN")
print("=" * 70)

print("Training datasets :", train_dataset_names)
print("External datasets :", external_dataset_names)
print("Optional OOPS     :", ENABLE_OPTIONAL_OOPS)

print("=" * 70)


# ------------------------------------------------------------
# Download settings
# ------------------------------------------------------------

DOWNLOAD_CHUNK_SIZE = 8 * 1024 * 1024       # 8 MiB
DOWNLOAD_TIMEOUT = (30, 120)                 # connect, read
DOWNLOAD_MAX_MINUTES = 70                    # per dataset
PROGRESS_MIN_INTERVAL = 0.5                  # tqdm refresh interval


# ------------------------------------------------------------
# File hashing
# ------------------------------------------------------------

def md5_file(path, chunk_size=DOWNLOAD_CHUNK_SIZE):
    """
    Calculate MD5 without loading the complete file into RAM.
    """
    h = hashlib.md5()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Human-readable size
# ------------------------------------------------------------

def human_bytes(value):
    """
    Convert bytes to readable binary units.
    """
    value = float(value)

    units = [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]

    for unit in units:

        if value < 1024.0 or unit == units[-1]:
            return f"{value:.2f} {unit}"

        value /= 1024.0

    return f"{value:.2f} TiB"


# ------------------------------------------------------------
# Progress-aware safe download
# ------------------------------------------------------------

def safe_download_with_progress(
    url,
    dst,
    expected_md5=None,
    expected_size=None,
    max_minutes=DOWNLOAD_MAX_MINUTES,
):
    """
    Download a large file safely with:

        - tqdm byte progress bar
        - resume support
        - Range/206 detection
        - bandwidth reporting
        - ETA
        - checksum validation
        - partial-file preservation
    """

    dst = Path(dst)

    # Example:
    # MCFD.zip
    #
    # partial:
    # MCFD.zip.part

    part = dst.with_suffix(dst.suffix + ".part")

    started = time.time()

    existing = (
        part.stat().st_size
        if part.exists()
        else 0
    )


    # --------------------------------------------------------
    # First request
    # --------------------------------------------------------

    headers = {}

    if existing > 0:
        headers["Range"] = f"bytes={existing}-"

        print(
            f"[RESUME] {dst.name}: "
            f"continuing from {human_bytes(existing)}"
        )

    else:
        print(
            f"[DOWNLOAD] {dst.name}: "
            f"starting from 0 B"
        )


    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=DOWNLOAD_TIMEOUT,
    )


    # --------------------------------------------------------
    # Detect HTTP Range support
    # --------------------------------------------------------

    if existing > 0 and response.status_code == 206:

        # Proper partial-content response.
        mode = "ab"

        content_range = response.headers.get(
            "Content-Range"
        )

        print(
            f"[RANGE OK] {dst.name}: "
            f"{content_range or 'partial response'}"
        )


    elif existing > 0 and response.status_code == 200:

        # Server ignored Range.
        #
        # Restart safely instead of corrupting the archive.

        print(
            f"[RANGE UNSUPPORTED] {dst.name}: "
            "server ignored resume request; restarting safely."
        )

        response.close()

        existing = 0

        part.unlink(missing_ok=True)

        response = requests.get(
            url,
            stream=True,
            timeout=DOWNLOAD_TIMEOUT,
        )

        response.raise_for_status()

        mode = "wb"


    else:

        response.raise_for_status()

        mode = "ab" if existing > 0 else "wb"


    # --------------------------------------------------------
    # Determine total remote size
    # --------------------------------------------------------

    content_length = response.headers.get(
        "Content-Length"
    )

    if content_length is not None:

        content_length = int(content_length)

    else:

        content_length = None


    if response.status_code == 206:

        total_size = (
            existing + content_length
            if content_length is not None
            else expected_size
        )

    else:

        total_size = (
            content_length
            if content_length is not None
            else expected_size
        )


    # --------------------------------------------------------
    # Progress bar
    # --------------------------------------------------------

    progress = tqdm(
        total=total_size,
        initial=existing,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=dst.stem,
        dynamic_ncols=True,
        mininterval=PROGRESS_MIN_INTERVAL,
    )


    bytes_written = existing


    try:

        with response:

            with open(part, mode) as f:

                for chunk in response.iter_content(
                    chunk_size=DOWNLOAD_CHUNK_SIZE
                ):

                    if not chunk:
                        continue

                    f.write(chunk)

                    bytes_written += len(chunk)

                    progress.update(len(chunk))


                    # ------------------------------------------------
                    # Hard per-download safety timeout
                    # ------------------------------------------------

                    elapsed = time.time() - started

                    if elapsed > max_minutes * 60:

                        raise RuntimeError(
                            f"Download exceeded the per-dataset "
                            f"safety limit of {max_minutes} minutes. "
                            f"Partial file preserved at: {part}"
                        )


    except Exception:

        progress.close()

        print(
            f"\n[INTERRUPTED] {dst.name}"
        )

        print(
            f"Partial file preserved at: {part}"
        )

        raise


    finally:

        progress.close()


    # --------------------------------------------------------
    # Verify expected size when known
    # --------------------------------------------------------

    if expected_size is not None:

        actual_size = part.stat().st_size

        if actual_size != expected_size:

            raise IOError(
                f"Size verification failed for {dst.name}: "
                f"got {human_bytes(actual_size)}, "
                f"expected {human_bytes(expected_size)}"
            )


    # --------------------------------------------------------
    # Move completed .part -> final archive
    # --------------------------------------------------------

    os.replace(part, dst)


    # --------------------------------------------------------
    # MD5 verification
    # --------------------------------------------------------

    if expected_md5:

        print(
            f"[VERIFY] {dst.name}: calculating MD5..."
        )

        got_md5 = md5_file(dst)

        if got_md5.lower() != expected_md5.lower():

            dst.unlink(missing_ok=True)

            raise IOError(
                f"Checksum mismatch for {dst.name}: "
                f"got {got_md5}, "
                f"expected {expected_md5}"
            )

    else:

        got_md5 = None


    # --------------------------------------------------------
    # Final statistics
    # --------------------------------------------------------

    elapsed = max(time.time() - started, 1e-6)

    final_size = dst.stat().st_size

    avg_speed = final_size / elapsed

    print()

    print(
        f"[DONE] {dst.name}: "
        f"{human_bytes(final_size)} "
        f"in {elapsed / 60:.2f} min "
        f"({human_bytes(avg_speed)}/s)"
    )

    if got_md5:

        print(
            f"[MD5 OK] {got_md5}"
        )

    print()

    return dst


# ------------------------------------------------------------
# Verified Zenodo download
# ------------------------------------------------------------

def ensure_dataset_zip_with_progress(
    name,
    max_minutes=DOWNLOAD_MAX_MINUTES,
):
    """
    Resolve the current Zenodo record, verify expected metadata,
    reuse a verified local archive, otherwise download it with
    a real byte-level progress bar.
    """

    if name not in DATASETS:

        raise KeyError(
            f"Unknown dataset: {name}"
        )


    cfg = DATASETS[name]


    # --------------------------------------------------------
    # Local destination
    # --------------------------------------------------------

    dst = (
        DIRS["datasets_zips"]
        / Path(cfg["filename"]).name
    )


    meta_path = (
        DIRS["metadata"]
        / f"{name}_source_metadata.json"
    )


    # --------------------------------------------------------
    # Resolve official/versioned Zenodo metadata
    # --------------------------------------------------------

    remote = resolve_zenodo_file(
        cfg["record_id"],
        cfg["filename"],
    )


    remote_md5 = (
        remote.get("checksum") or ""
    ).split(":")[-1]


    expected_md5 = cfg["expected_md5"]


    # --------------------------------------------------------
    # Remote integrity check
    # --------------------------------------------------------

    if (
        remote_md5
        and remote_md5.lower()
        != expected_md5.lower()
    ):

        raise RuntimeError(
            f"Source integrity audit failed for {name}: "
            f"Zenodo reports {remote_md5}, "
            f"but notebook expects {expected_md5}."
        )


    remote_size = remote.get("size")


    # --------------------------------------------------------
    # Existing verified cache
    # --------------------------------------------------------

    if dst.exists():

        print(
            f"\n[CACHE CHECK] {name}"
        )

        local_size = dst.stat().st_size

        local_md5 = md5_file(dst)


        if (
            local_md5.lower()
            == expected_md5.lower()
            and (
                remote_size is None
                or local_size == remote_size
            )
        ):

            print(
                f"[CACHE HIT] {name}: "
                f"{dst.name} "
                f"({human_bytes(local_size)})"
            )

            info = {
                **cfg,
                **remote,
                "local_path": str(dst),
                "local_bytes": local_size,
                "local_md5": local_md5,
                "verified": True,
                "source": "verified_local_cache",
            }

            meta_path.write_text(
                json.dumps(
                    info,
                    indent=2,
                    default=str,
                )
            )

            return dst


        print(
            f"[CACHE INVALID] {name}: "
            "local archive failed integrity check."
        )

        dst.unlink(missing_ok=True)


    # --------------------------------------------------------
    # Download
    # --------------------------------------------------------

    print()
    print("=" * 70)

    print(
        f"[DOWNLOAD] {name}"
    )

    print(
        f"Remote file : {remote['key']}"
    )

    print(
        f"Remote size : "
        f"{human_bytes(remote_size) if remote_size else 'unknown'}"
    )

    print(
        f"Expected MD5: {expected_md5}"
    )

    print("=" * 70)


    path = safe_download_with_progress(
        url=remote["download"],
        dst=dst,
        expected_md5=expected_md5,
        expected_size=remote_size,
        max_minutes=max_minutes,
    )


    # --------------------------------------------------------
    # Save source metadata
    # --------------------------------------------------------

    info = {
        **cfg,
        **remote,
        "local_path": str(path),
        "local_bytes": path.stat().st_size,
        "local_md5": expected_md5,
        "verified": True,
        "source": "fresh_verified_download",
    }


    meta_path.write_text(
        json.dumps(
            info,
            indent=2,
            default=str,
        )
    )


    return path


# ------------------------------------------------------------
# Runtime budget gate
# ------------------------------------------------------------

if not all_download_names:

    print(
        "[INFO] No datasets are enabled for download."
    )

else:

    # Conservative estimate for the complete download stage.
    #
    # This is only the gate estimate; actual timing is recorded
    # by RuntimeBudget and each download reports its own progress.

    estimated_download_seconds = (
        20 * 60
    )


    BUDGET.estimate_and_gate(
        "dataset_downloads",
        estimated_s=estimated_download_seconds,
        essential=True,
    )


    # --------------------------------------------------------
    # Download datasets one-by-one
    # --------------------------------------------------------

    downloaded_paths = {}


    for name in all_download_names:

        print()
        print(
            "#" * 70
        )

        print(
            f"DATASET: {name}"
        )

        print(
            "#" * 70
        )


        BUDGET.guard(
            f"download_{name}",
            essential=True,
        )


        t0 = time.time()


        path = ensure_dataset_zip_with_progress(
            name=name,
            max_minutes=DOWNLOAD_MAX_MINUTES,
        )


        elapsed = time.time() - t0


        downloaded_paths[name] = path


        print(
            f"[COMPLETED] {name}: "
            f"{path}"
        )

        print(
            f"[STAGE TIME] "
            f"{elapsed / 60:.2f} minutes"
        )


        # ----------------------------------------------------
        # Persist cumulative runtime checkpoint
        # ----------------------------------------------------

        BUDGET.checkpoint(
            f"download_{name}"
        )


        # ----------------------------------------------------
        # Immediate VRAM sanity
        # ----------------------------------------------------

        if torch.cuda.is_available():

            try:
                assert_vram_budget(
                    f"dataset_download_{name}"
                )
            except NameError:
                # Function may not exist if this cell is run
                # independently of the earlier runtime cell.
                pass


# ------------------------------------------------------------
# Final download summary
# ------------------------------------------------------------

print()
print("=" * 70)
print("DATASET DOWNLOAD SUMMARY")
print("=" * 70)


for name in all_download_names:

    path = downloaded_paths.get(name)

    if path is None:

        print(
            f"[NOT COMPLETED] {name}"
        )

        continue


    size = path.stat().st_size

    print(
        f"{name:15s} | "
        f"{human_bytes(size):>12s} | "
        f"{path}"
    )


print("=" * 70)

print(
    "Training datasets:",
    train_dataset_names
)

print(
    "External test datasets:",
    external_dataset_names
)

print(
    "CAUCAFall is external-only and is NOT added to training."
)

print("=" * 70)

DATASET DOWNLOAD PLAN
Training datasets : ['MCFD', 'GMDCSA24']
External datasets : ['CAUCAFall']
Optional OOPS     : False

######################################################################
DATASET: MCFD
######################################################################

[CACHE CHECK] MCFD
[CACHE HIT] MCFD: MCFD.zip (2.61 GiB)
[COMPLETED] MCFD: /content/drive/MyDrive/fall_detection_project/datasets_zips/MCFD.zip
[STAGE TIME] 0.22 minutes

######################################################################
DATASET: GMDCSA24
######################################################################

[CACHE CHECK] GMDCSA24
[CACHE HIT] GMDCSA24: GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip (1.03 GiB)
[COMPLETED] GMDCSA24: /content/drive/MyDrive/fall_detection_project/datasets_zips/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip
[STAGE TIME] 0.11 minutes

######################################################################
DATASET: CAUCAFall
######

## 4. Safe extraction and dataset discovery

Raw extraction is temporary under `/content/fall_detection_work`. ZIP members are checked to prevent Zip Slip. The persistent Google Drive copy is the ZIP plus processed pose caches/manifests, not a second permanent raw-data copy.

In [19]:
def safe_extract_zip(zip_path, extract_root):
    zip_path=Path(zip_path); extract_root=Path(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path,'r') as z:
        root=extract_root.resolve()
        bad=[]
        for info in z.infolist():
            target=(extract_root/info.filename).resolve()
            if os.path.commonpath([str(root),str(target)])!=str(root):
                bad.append(info.filename)
        if bad:
            raise RuntimeError(f"Unsafe ZIP members detected: {bad[:3]}")
        z.extractall(extract_root)
    return extract_root

def find_videos(root):
    exts={'.mp4','.avi','.mov','.mkv','.webm'}
    return sorted([p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in exts])

EXTRACTED={}
for name in train_dataset_names+external_dataset_names:
    zip_path=DIRS['datasets_zips']/Path(DATASETS[name]['filename']).name
    out=TEMP_ROOT/name
    if out.exists():
        print(f"[EXTRACT CACHE] {name}: {out}")
    else:
        BUDGET.guard(f'extract_{name}', essential=True)
        safe_extract_zip(zip_path,out)
    EXTRACTED[name]=out
    vids=find_videos(out)
    print(name, 'video files:', len(vids))
    BUDGET.checkpoint(f'extract_{name}')

[EXTRACT CACHE] MCFD: /content/fall_detection_work/MCFD
MCFD video files: 1352
[EXTRACT CACHE] GMDCSA24: /content/fall_detection_work/GMDCSA24
GMDCSA24 video files: 160
[EXTRACT CACHE] CAUCAFall: /content/fall_detection_work/CAUCAFall
CAUCAFall video files: 516


## 5. Manifest construction

The manifest defines the scientific boundary of the pipeline. It records dataset, video path, subject, camera, scenario, activity, label, and segment times. Segment-level sampling happens **after** the split fields are determined, so overlapping temporal windows cannot accidentally leak across splits.

**GMDCSA24:** class intervals are parsed from the supplied CSVs. A positive segment is any interval whose class is fall-like; all other annotated intervals remain non-fall hard negatives.

**MCFD:** the raw/repacked MCFD is treated as scenario-level segments because the dataset itself consists of short scenario videos observed from multiple synchronized cameras. Camera IDs are parsed from path/name patterns where possible. The notebook stops rather than silently guessing when camera parsing is ambiguous.

**CAUCAFall:** subject/activity labels come from the subject/activity directory organization described by the source; it is never used for train/validation tuning.

In [20]:

# ================================================================
# 5. Manifest construction — self-contained and repaired
# ================================================================
#
# This cell defines EVERY manifest builder before any
# builder is called:
#   - make_mcfd_manifest
#   - make_gmd_manifest
#   - make_caucafall_manifest
#
# It also defines the missing OmniFall metadata helper
# `download_omnifall_file`.
#
# It uses the published OmniFall staged-label schema:
# path, label, start, end, subject, cam, dataset
# and the published MCFD CV split files.
# ================================================================

import json
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# OmniFall repository
# ----------------------------------------------------------------

OMNIFALL_REPO_ID = "simplexsigil2/omnifall"
OMNIFALL_REPO_TYPE = "dataset"

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    hf_hub_download = None
    _HF_IMPORT_ERROR = exc
else:
    _HF_IMPORT_ERROR = None


def download_omnifall_file(repo_path, local_path):
    """
    Download one file from the public OmniFall Hugging Face dataset
    repository and copy it into the notebook's persistent metadata area.

    The helper is intentionally idempotent: an existing non-empty file
    is reused.
    """
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path

    if hf_hub_download is None:
        raise RuntimeError(
            "huggingface_hub is required to retrieve OmniFall metadata. "
            f"Import error: {_HF_IMPORT_ERROR!r}"
        )

    try:
        cached = hf_hub_download(
            repo_id=OMNIFALL_REPO_ID,
            repo_type=OMNIFALL_REPO_TYPE,
            filename=repo_path,
        )
    except Exception as exc:
        raise RuntimeError(
            "Failed to download OmniFall metadata file "
            f"{repo_path!r} from {OMNIFALL_REPO_ID!r}. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc

    shutil.copy2(cached, local_path)
    return local_path


# ----------------------------------------------------------------
# Shared helpers
# ----------------------------------------------------------------

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


CLASS_NAMES = {
    0: "walk",
    1: "fall",
    2: "fallen",
    3: "sit_down",
    4: "sitting",
    5: "lie_down",
    6: "lying",
    7: "stand_up",
    8: "standing",
    9: "other",
    10: "kneel_down",
    11: "kneeling",
    12: "squat_down",
    13: "squatting",
    14: "crawl",
    15: "jump",
}


POSITIVE_IDS = {1, 2}


def _safe_int(value, default=None):
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    try:
        return int(value)
    except Exception:
        return default


def _safe_float(value, default=None):
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    try:
        return float(value)
    except Exception:
        return default


def _normalize_path_key(value):
    """
    Normalize a relative/video label path for deterministic matching.

    Examples:
        chute24/cam4
        chute24\\cam4.avi

    become the same key:
        chute24/cam4
    """
    s = str(value).strip()

    if s.lower() in {"", "nan", "none", "null"}:
        return ""

    s = s.replace("\\", "/")
    s = re.sub(r"/+", "/", s)
    s = s.strip("/").lower()

    parts = []
    for part in s.split("/"):
        if not part:
            continue

        suffix = Path(part).suffix.lower()
        if suffix in VIDEO_EXTS:
            part = Path(part).stem

        parts.append(part.lower())

    return "/".join(parts)


def _compact(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def _parse_subject(value):
    """
    Parse subject/person/participant/S<n> identifiers conservatively.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = "/".join(Path(str(value)).parts).lower()

    patterns = [
        r"(?:^|[^a-z0-9])subject[_ -]?0*(\d+)(?:[^0-9]|$)",
        r"(?:^|[^a-z0-9])person[_ -]?0*(\d+)(?:[^0-9]|$)",
        r"(?:^|[^a-z0-9])participant[_ -]?0*(\d+)(?:[^0-9]|$)",
        r"(?:^|[^a-z0-9])s[_ -]?0*(\d+)(?:[^0-9]|$)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            parsed = _safe_int(match.group(1))
            if parsed is not None:
                return parsed

    stem = _compact(Path(str(value)).stem)
    match = re.search(r"s0*(\d+)$", stem, flags=re.I)
    if match:
        return _safe_int(match.group(1))

    return None


def _relative_video_path(vp, root):
    vp = Path(vp)
    root = Path(root)

    try:
        rel = vp.relative_to(root)
    except ValueError:
        rel = vp

    return str(rel).replace("\\", "/")


def _index_videos(root):
    """
    Build a deterministic index from normalized relative video path to
    physical Path.

    Duplicate normalized keys are retained in `duplicates` rather than
    silently overwritten.
    """
    root = Path(root)
    videos = find_videos(root)

    if not videos:
        raise RuntimeError(
            f"No video files found under extracted root: {root}"
        )

    mapping = {}
    duplicates = {}

    for vp in sorted(videos, key=lambda p: str(p).lower()):
        rel = _relative_video_path(vp, root)
        key = _normalize_path_key(rel)

        if not key:
            continue

        if key in mapping:
            duplicates.setdefault(key, [mapping[key]])
            duplicates[key].append(vp)
        else:
            mapping[key] = vp

    if not mapping:
        raise RuntimeError(
            f"Video indexing produced no usable paths under {root}."
        )

    return {
        "videos": videos,
        "mapping": mapping,
        "duplicates": duplicates,
    }


def _resolve_video_from_key(key, video_index, subject=None):
    """
    Resolve a normalized OmniFall label key to one physical video.

    Matching policy:
      1. exact normalized relative-path match;
      2. unique suffix match;
      3. deterministic subject-compatible suffix match.

    Ambiguous matches are rejected rather than guessed.
    """
    key = _normalize_path_key(key)
    if not key:
        return None, "empty_key"

    mapping = video_index["mapping"]

    exact = mapping.get(key)
    if exact is not None:
        return exact, "exact"

    suffix_matches = []
    for candidate_key, vp in mapping.items():
        if candidate_key.endswith("/" + key) or candidate_key == key:
            suffix_matches.append(vp)

    if subject is not None:
        subject_compatible = []
        for vp in suffix_matches:
            parsed_subject = _parse_subject(
                _relative_video_path(vp, vp.parents[-1])
            )
            if parsed_subject is None or parsed_subject == subject:
                subject_compatible.append(vp)

        if subject_compatible:
            suffix_matches = subject_compatible

    suffix_matches = sorted(
        set(suffix_matches),
        key=lambda p: str(p).lower(),
    )

    if len(suffix_matches) == 1:
        return suffix_matches[0], "suffix"

    if len(suffix_matches) > 1:
        return None, f"ambiguous:{len(suffix_matches)}"

    return None, "not_found"


def _read_omnifall_labels(repo_path, local_path, dataset_name):
    local_path = download_omnifall_file(repo_path, local_path)
    df = pd.read_csv(local_path)

    required = {"path", "label", "start", "end"}
    missing = sorted(required - set(df.columns))

    if missing:
        raise RuntimeError(
            f"{dataset_name} OmniFall labels are missing required columns: "
            f"{missing}. Found columns: {list(df.columns)}"
        )

    return df


def _activity_name(label_id):
    label_id = _safe_int(label_id)
    if label_id is None:
        return "unknown"
    return CLASS_NAMES.get(label_id, str(label_id))


def _make_common_manifest_row(
    dataset,
    video_path,
    subject,
    camera,
    scenario,
    source_label,
    start,
    end,
):
    source_label = int(source_label)

    return {
        "dataset": dataset,
        "video_path": str(video_path),
        "subject": subject,
        "camera": int(camera),
        "scenario": str(scenario),
        "activity": _activity_name(source_label),
        "source_label": source_label,
        "label": int(source_label in POSITIVE_IDS),
        "start_s": float(start),
        "end_s": float(end),
    }


# ================================================================
# MCFD
# ================================================================

def make_mcfd_manifest(root):
    """
    Build MCFD from OmniFall segment labels and the published cross-view split.

    IMPORTANT MCFD path rule:
        OmniFall identifies an MCFD camera stream by the canonical key
        ``chute<N>/cam<M>``.  The physical ZIP may contain additional parent
        directories and/or suffixes on the AVI filename.  Therefore the
        resolver MUST NOT compare the whole relative path literally.

    A physical video is indexed by the same canonical key extracted from:
        1. the first path component beginning with ``chute``; and
        2. the camera identifier at the beginning of the file stem.

    This supports both the original 192-camera-view layout and repack layouts
    in which the extracted archive contains extra directories or segment-like
    filename suffixes.
    """
    root = Path(root)

    meta_dir = DIRS["metadata"] / "omnifall"
    split_dir = meta_dir / "splits" / "cv" / "mcfd"
    meta_dir.mkdir(parents=True, exist_ok=True)
    split_dir.mkdir(parents=True, exist_ok=True)

    labels_path = meta_dir / "mcfd.csv"
    labels = _read_omnifall_labels(
        "labels/mcfd.csv",
        labels_path,
        "MCFD",
    )

    videos = sorted(
        find_videos(root),
        key=lambda p: str(p).lower(),
    )

    if not videos:
        raise RuntimeError(
            f"MCFD contains no video files under extracted root: {root}"
        )

    # ------------------------------------------------------------
    # Canonical MCFD identity: chute<N>/cam<M>
    # ------------------------------------------------------------

    def canonical_mcfd_key(value):
        """
        Convert label paths and physical video paths to the same identity.

        Examples:
            chute24/cam4       -> chute24/cam4
            MCFD/chute24/cam4.avi
                                -> chute24/cam4
            foo/chute24/cam4_seg17.avi
                                -> chute24/cam4
        """
        raw = str(value).replace("\\", "/")
        parts = [
            p.strip()
            for p in raw.split("/")
            if p.strip()
        ]

        chute = None
        for part in parts:
            m = re.search(
                r"^(chute[-_ ]?0*\d+)$",
                Path(part).stem,
                flags=re.I,
            )
            if m:
                digits = re.search(r"(\d+)$", m.group(1)).group(1)
                chute = f"chute{int(digits)}"
                break

        if chute is None:
            # Also allow chute identifier to appear inside a compound folder.
            for part in parts:
                m = re.search(
                    r"(?:^|[^a-z0-9])chute[-_ ]?0*(\d+)(?:[^0-9]|$)",
                    part,
                    flags=re.I,
                )
                if m:
                    chute = f"chute{int(m.group(1))}"
                    break

        stem = Path(parts[-1]).stem if parts else ""
        stem_l = stem.lower()

        # Camera token is normally cam1 ... cam8.  Accept camera/cam variants.
        cam_match = re.search(
            r"(?:^|[^a-z0-9])(?:cam(?:era)?)[-_ ]?0*(\d+)",
            stem_l,
            flags=re.I,
        )

        if cam_match is None:
            # Some layouts put the camera token in a directory rather than
            # the terminal filename.
            for part in reversed(parts[:-1]):
                cam_match = re.search(
                    r"(?:^|[^a-z0-9])(?:cam(?:era)?)[-_ ]?0*(\d+)(?:[^0-9]|$)",
                    part,
                    flags=re.I,
                )
                if cam_match:
                    break

        if chute is None or cam_match is None:
            return ""

        cam = int(cam_match.group(1))
        return f"{chute}/cam{cam}"

    # ------------------------------------------------------------
    # Build the physical-video index by canonical MCFD identity.
    # ------------------------------------------------------------

    video_groups = {}
    unkeyed_videos = []

    for vp in videos:
        key = canonical_mcfd_key(
            _relative_video_path(vp, root)
        )

        if not key:
            unkeyed_videos.append(vp)
            continue

        video_groups.setdefault(
            key,
            [],
        ).append(vp)

    if not video_groups:
        preview = [
            _relative_video_path(v, root)
            for v in videos[:10]
        ]
        raise RuntimeError(
            "MCFD physical-video indexing found zero canonical chute/camera "
            "keys. The extracted archive layout is incompatible with the "
            "expected MCFD layout. First physical paths: "
            f"{preview}"
        )

    # Deterministic representative for each camera stream.
    # Prefer a file whose stem is exactly cam<N>; otherwise use the shortest,
    # lexicographically first filename. This handles repacks with suffixes.
    video_map = {}
    video_duplicates = {}

    for key, paths in video_groups.items():

        def rank_path(p):
            stem = Path(p).stem.lower()
            cam_token = key.split("/")[-1]
            exact = int(stem == cam_token)
            return (
                -exact,
                len(str(p)),
                str(p).lower(),
            )

        ordered = sorted(
            paths,
            key=rank_path,
        )

        video_map[key] = ordered[0]

        if len(ordered) > 1:
            video_duplicates[key] = ordered

    # ------------------------------------------------------------
    # Read the published MCFD CV split files using the SAME canonical key.
    # ------------------------------------------------------------

    split_map = {}
    split_file_counts = {}

    for split_name in ("train", "val", "test"):

        split_path = split_dir / f"{split_name}.csv"

        if not split_path.exists():
            download_omnifall_file(
                f"splits/cv/mcfd/{split_name}.csv",
                split_path,
            )

        sdf = pd.read_csv(split_path)

        if "path" not in sdf.columns:
            raise RuntimeError(
                f"MCFD OmniFall split file {split_name}.csv lacks 'path'. "
                f"Found columns: {list(sdf.columns)}"
            )

        split_file_counts[split_name] = len(sdf)

        for raw_path in sdf["path"].astype(str):

            key = canonical_mcfd_key(
                raw_path
            )

            if not key:
                raise RuntimeError(
                    "Could not canonicalize an MCFD split path: "
                    f"{raw_path!r} in {split_name}.csv"
                )

            prior = split_map.get(key)

            if prior is not None and prior != split_name:
                raise RuntimeError(
                    "MCFD CV split map assigns the same canonical camera key "
                    f"to multiple splits: {key!r} -> {prior!r}, {split_name!r}"
                )

            split_map[key] = split_name

    # ------------------------------------------------------------
    # Build one manifest row per OmniFall temporal annotation.
    # ------------------------------------------------------------

    rows = []
    missing_video = []
    missing_split = []
    malformed = []

    for row_idx, row in labels.iterrows():

        raw_path = str(row["path"]).strip()

        source_label = _safe_int(
            row["label"]
        )

        start = _safe_float(
            row["start"]
        )

        end = _safe_float(
            row["end"]
        )

        if (
            source_label is None
            or start is None
            or end is None
        ):
            malformed.append({
                "row": int(row_idx),
                "path": raw_path,
                "reason": "non_numeric_annotation_fields",
            })
            continue

        if (
            not np.isfinite(start)
            or not np.isfinite(end)
            or end <= start
        ):
            malformed.append({
                "row": int(row_idx),
                "path": raw_path,
                "start": start,
                "end": end,
                "reason": "invalid_annotation_interval",
            })
            continue

        key = canonical_mcfd_key(
            raw_path
        )

        if not key:
            missing_video.append({
                "row": int(row_idx),
                "path": raw_path,
                "reason": "cannot_canonicalize_label_path",
            })
            continue

        video_path = video_map.get(
            key
        )

        if video_path is None:
            missing_video.append({
                "row": int(row_idx),
                "path": raw_path,
                "canonical_key": key,
                "reason": "canonical_video_key_not_found",
            })
            continue

        split = split_map.get(
            key
        )

        if split is None:
            missing_split.append({
                "row": int(row_idx),
                "path": raw_path,
                "canonical_key": key,
                "reason": "canonical_split_key_not_found",
            })
            continue

        subject = _safe_int(
            row["subject"]
            if "subject" in labels.columns
            else None,
            default=-1,
        )

        camera = _safe_int(
            row["cam"]
            if "cam" in labels.columns
            else None,
            default=-1,
        )

        scenario = key.split("/", 1)[0].replace(
            "chute",
            "",
        )

        # Keep the downstream schema stable.
        rows.append({
            "dataset": "MCFD",
            "video_path": str(video_path),
            "subject": subject,
            "camera": camera,
            "scenario": scenario,
            "activity": _activity_name(source_label),
            "source_label": int(source_label),
            "label": int(source_label in POSITIVE_IDS),
            "start_s": float(start),
            "end_s": float(end),
            "split": split,
        })

    out = pd.DataFrame(rows)

    diagnostics = {
        "extracted_root": str(root),
        "physical_video_files_found": len(videos),
        "canonical_camera_keys_found": len(video_map),
        "unkeyed_physical_videos": len(unkeyed_videos),
        "physical_duplicate_keys": {
            k: [str(p) for p in v[:10]]
            for k, v in video_duplicates.items()
        },
        "omnifall_label_rows": len(labels),
        "resolved_manifest_rows": len(out),
        "missing_video_rows": len(missing_video),
        "missing_split_rows": len(missing_split),
        "malformed_rows": len(malformed),
        "split_file_row_counts": split_file_counts,
        "split_map_entries": len(split_map),
        "split_counts": (
            out["split"].value_counts().to_dict()
            if not out.empty
            else {}
        ),
        "label_counts": (
            out["label"].value_counts().to_dict()
            if not out.empty
            else {}
        ),
        "camera_counts": (
            out["camera"].value_counts().sort_index().to_dict()
            if not out.empty
            else {}
        ),
        "missing_video_examples": missing_video[:50],
        "missing_split_examples": missing_split[:50],
        "malformed_examples": malformed[:50],
        "resolution_policy": (
            "canonical MCFD key chute<N>/cam<M>; representative is exact cam<N> "
            "filename when available, otherwise deterministic shortest path"
        ),
        "protocol": "cross-view / view-shift",
        "time_policy": "use OmniFall annotation start/end directly",
    }

    (
        meta_dir / "mcfd_manifest_resolution_diagnostics.json"
    ).write_text(
        json.dumps(
            diagnostics,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    if malformed:
        raise RuntimeError(
            f"MCFD contains {len(malformed)} malformed OmniFall rows. "
            "See mcfd_manifest_resolution_diagnostics.json."
        )

    if missing_video:
        sample = missing_video[:5]
        raise RuntimeError(
            "MCFD physical-video resolution failed. "
            f"Unresolved rows: {len(missing_video)} / {len(labels)}. "
            f"Examples: {sample}. "
            "The resolver now canonicalizes paths as chute<N>/cam<M>."
        )

    if missing_split:
        sample = missing_split[:5]
        raise RuntimeError(
            "MCFD cross-view split resolution failed. "
            f"Unresolved rows: {len(missing_split)} / {len(labels)}. "
            f"Examples: {sample}."
        )

    if out.empty:
        raise RuntimeError(
            "MCFD manifest is empty after canonical path resolution. "
            f"Videos={len(videos)}, labels={len(labels)}."
        )

    if len(out) != len(labels):
        raise RuntimeError(
            "MCFD annotation mapping is incomplete: "
            f"resolved {len(out)} of {len(labels)} label rows."
        )

    expected_split_rows = {
        "train": 169,
        "val": 169,
        "test": 1014,
    }

    observed_split_rows = (
        out["split"].value_counts().to_dict()
    )

    if observed_split_rows != expected_split_rows:
        raise RuntimeError(
            "MCFD cross-view split counts differ from the published "
            "OmniFall CV protocol. "
            f"Observed={observed_split_rows}, "
            f"Expected={expected_split_rows}."
        )

    camera_ids = sorted(
        out["camera"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if camera_ids != list(range(1, 9)):
        raise RuntimeError(
            "MCFD camera coverage is unexpected. "
            f"Observed cameras={camera_ids}, expected 1..8."
        )

    if out["label"].nunique() < 2:
        raise RuntimeError(
            "MCFD binary label diversity is insufficient: "
            f"{out['label'].value_counts().to_dict()}"
        )

    print("=" * 70)
    print("MCFD PRE-SEGMENTED CLIP MANIFEST")
    print("=" * 70)
    print("Archive video files          :", len(videos))
    print("Canonical camera keys        :", len(video_map))
    print("OmniFall label rows          :", len(labels))
    print("Resolved manifest rows       :", len(out))
    print("Missing video mappings       :", len(missing_video))
    print("Missing split mappings       :", len(missing_split))
    print("Malformed annotations        :", len(malformed))
    print("Split map entries            :", len(split_map))
    print("Label counts                 :", out["label"].value_counts().to_dict())
    print("Split counts                 :", observed_split_rows)
    print("Camera counts                :", out["camera"].value_counts().sort_index().to_dict())
    print("Physical duplicate keys     :", len(video_duplicates))
    print("Time policy                  :", "OmniFall [start,end]")
    print("Protocol                    :", "cross-view / view-shift")
    print("=" * 70)

    return out

# ================================================================
# GMDCSA24
# ================================================================


def make_gmd_manifest(root):
    """
    Build GMDCSA24 from OmniFall temporal annotations and the physical
    GMDCSA24 archive.

    IMPORTANT GMDCSA24 identity rule
    --------------------------------
    The public dataset is organized as:

        Subject 1/ADL/<file>.mp4
        Subject 1/Fall/<file>.mp4
        ...
        Subject 4/ADL/<file>.mp4
        Subject 4/Fall/<file>.mp4

    The OmniFall label paths can use a slightly different spelling of
    "Subject N", separators, or an archive prefix. Therefore the physical
    file must NOT be matched by literal whole-path equality.

    The canonical identity is:

        subject=<1..4> + partition={adl,fall} + filename stem

    This is sufficient because the upstream dataset has one camera and
    the filename numbering is unique within each Subject/partition pair.
    """
    root = Path(root)

    meta_dir = DIRS["metadata"] / "omnifall"
    meta_dir.mkdir(parents=True, exist_ok=True)

    labels_path = meta_dir / "GMDCSA24.csv"

    labels = _read_omnifall_labels(
        "labels/GMDCSA24.csv",
        labels_path,
        "GMDCSA24",
    )

    videos = sorted(
        find_videos(root),
        key=lambda p: str(p).lower(),
    )

    if not videos:
        raise RuntimeError(
            f"GMDCSA24 contains no video files under extracted root: {root}"
        )

    # ------------------------------------------------------------
    # Canonical GMDCSA24 key
    # ------------------------------------------------------------

    def canonical_gmd_key(value):
        """
        Return subject/partition/filename identity.

        Examples:
            Subject 1/ADL/01.mp4
                -> subject1/adl/01

            GMDCSA24/Subject_1/Fall/07.mp4
                -> subject1/fall/07

            Subject_4/ADL/23
                -> subject4/adl/23
        """
        raw = str(value).replace("\\", "/")

        parts = [
            p.strip()
            for p in raw.split("/")
            if p.strip()
        ]

        if not parts:
            return ""

        subject = None
        partition = None

        # Search every path component, not just one fixed depth.
        for part in parts:
            stem = Path(part).stem

            m = re.search(
                r"subject[\s_-]*0*(\d+)",
                stem,
                flags=re.I,
            )

            if m:
                subject = int(m.group(1))

            low = _compact(stem)

            if low in {"adl", "activities", "activity"}:
                partition = "adl"

            elif low in {"fall", "falls"}:
                partition = "fall"

        # Filename = last path component.
        filename_stem = _compact(
            Path(parts[-1]).stem
        )

        if subject is None:
            # Allow a compact subject token such as subject1.
            compact_all = _compact(raw)

            m = re.search(
                r"subject0*(\d+)",
                compact_all,
                flags=re.I,
            )

            if m:
                subject = int(m.group(1))

        # Last-resort partition detection from the complete path.
        if partition is None:
            low_raw = raw.lower()

            if re.search(
                r"(?:^|[/ _-])adl(?:[/ _-]|$)",
                low_raw,
            ):
                partition = "adl"

            elif re.search(
                r"(?:^|[/ _-])falls?(?:[/ _-]|$)",
                low_raw,
            ):
                partition = "fall"

        if subject is None or partition is None or not filename_stem:
            return ""

        if subject not in {1, 2, 3, 4}:
            return ""

        return f"subject{subject}/{partition}/{filename_stem}"

    # ------------------------------------------------------------
    # Build one canonical physical-file index.
    # ------------------------------------------------------------

    video_groups = {}
    unkeyed_videos = []

    for vp in videos:

        rel = _relative_video_path(
            vp,
            root,
        )

        key = canonical_gmd_key(
            rel
        )

        if not key:
            unkeyed_videos.append(
                str(vp)
            )
            continue

        video_groups.setdefault(
            key,
            [],
        ).append(
            vp
        )

    if not video_groups:

        preview = [
            _relative_video_path(
                v,
                root,
            )
            for v in videos[:20]
        ]

        raise RuntimeError(
            "GMDCSA24 physical-video indexing found zero canonical "
            "subject/partition/filename keys. "
            "Expected paths resembling "
            "'Subject 1/ADL/01.mp4' or 'Subject_1/Fall/01.mp4'. "
            f"First physical paths: {preview}"
        )

    # ------------------------------------------------------------
    # Duplicate policy:
    # Keep exact canonical identities, but do NOT silently choose
    # between genuinely duplicated files.
    # ------------------------------------------------------------

    video_map = {}
    video_duplicates = {}

    for key, paths in video_groups.items():

        ordered = sorted(
            paths,
            key=lambda p: (
                len(str(p)),
                str(p).lower(),
            ),
        )

        video_map[key] = ordered[0]

        if len(ordered) > 1:
            video_duplicates[key] = [
                str(p)
                for p in ordered
            ]

    # The source paper describes exactly 160 MP4 clips:
    # S1=32, S2=48, S3=43, S4=37.
    expected_video_count = 160

    # ------------------------------------------------------------
    # Build manifest rows.
    # ------------------------------------------------------------

    rows = []
    missing_video = []
    malformed = []

    for row_idx, row in labels.iterrows():

        raw_path = str(
            row["path"]
        ).strip()

        source_label = _safe_int(
            row["label"]
        )

        start = _safe_float(
            row["start"]
        )

        end = _safe_float(
            row["end"]
        )

        if (
            source_label is None
            or start is None
            or end is None
        ):

            malformed.append({
                "row":
                    int(row_idx),
                "path":
                    raw_path,
                "reason":
                    "non_numeric_annotation_fields",
            })

            continue

        if (
            not np.isfinite(start)
            or not np.isfinite(end)
            or end <= start
        ):

            malformed.append({
                "row":
                    int(row_idx),
                "path":
                    raw_path,
                "start":
                    start,
                "end":
                    end,
                "reason":
                    "invalid_annotation_interval",
            })

            continue

        # Subject comes from OmniFall metadata when available,
        # otherwise from the path.
        subject = _safe_int(
            row["subject"]
            if "subject" in labels.columns
            else None,
            default=None,
        )

        if subject is None:
            subject = _parse_subject(
                raw_path
            )

        if subject not in {1, 2, 3, 4}:

            missing_video.append({
                "row":
                    int(row_idx),
                "path":
                    raw_path,
                "subject":
                    subject,
                "reason":
                    "invalid_or_missing_subject",
            })

            continue

        camera = _safe_int(
            row["cam"]
            if "cam" in labels.columns
            else None,
            default=1,
        )

        if camera is None or camera < 0:
            camera = 1

        # Canonical label identity.
        key = canonical_gmd_key(
            raw_path
        )

        # OmniFall labels may occasionally expose subject separately while
        # the path omits/changes the exact spelling. Rebuild a canonical key
        # using subject + partition + filename when possible.
        if not key:

            raw_parts = [
                p.strip()
                for p in raw_path.replace(
                    "\\",
                    "/",
                ).split("/")
                if p.strip()
            ]

            partition = None

            for part in raw_parts:
                low = _compact(
                    Path(part).stem
                )

                if low == "adl":
                    partition = "adl"
                    break

                if low in {"fall", "falls"}:
                    partition = "fall"
                    break

            filename_stem = (
                _compact(
                    Path(
                        raw_parts[-1]
                    ).stem
                )
                if raw_parts
                else ""
            )

            if (
                partition
                and filename_stem
            ):

                key = (
                    f"subject{subject}/"
                    f"{partition}/"
                    f"{filename_stem}"
                )

        video_path = video_map.get(
            key
        )

        if video_path is None:

            missing_video.append({
                "row":
                    int(row_idx),
                "path":
                    raw_path,
                "canonical_key":
                    key,
                "subject":
                    subject,
                "reason":
                    "canonical_video_key_not_found",
            })

            continue

        partition = key.split(
            "/",
        )[1]

        scenario = (
            f"subject{subject}_{partition}_"
            f"{Path(raw_path).stem}"
        )

        rows.append({
            "dataset":
                "GMDCSA24",

            "video_path":
                str(video_path),

            "subject":
                f"Subject_{subject}",

            "camera":
                int(camera),

            "scenario":
                scenario,

            "activity":
                _activity_name(
                    source_label
                ),

            "source_label":
                int(source_label),

            "label":
                int(
                    source_label
                    in POSITIVE_IDS
                ),

            "start_s":
                float(start),

            "end_s":
                float(end),
        })

    out = pd.DataFrame(
        rows
    )

    diagnostics = {
        "extracted_root":
            str(root),

        "physical_video_files_found":
            len(videos),

        "canonical_video_keys_found":
            len(video_map),

        "unkeyed_physical_videos":
            len(unkeyed_videos),

        "physical_duplicate_key_count":
            len(video_duplicates),

        "physical_duplicate_keys":
            {
                k: v[:10]
                for k, v
                in video_duplicates.items()
            },

        "expected_physical_video_count":
            expected_video_count,

        "omnifall_label_rows":
            len(labels),

        "resolved_manifest_rows":
            len(out),

        "missing_video_rows":
            len(missing_video),

        "malformed_rows":
            len(malformed),

        "missing_video_examples":
            missing_video[:50],

        "malformed_examples":
            malformed[:50],

        "label_counts":
            (
                out["label"]
                .value_counts()
                .to_dict()
                if not out.empty
                else {}
            ),

        "source_label_counts":
            (
                out["source_label"]
                .value_counts()
                .sort_index()
                .to_dict()
                if not out.empty
                else {}
            ),

        "subject_counts":
            (
                out["subject"]
                .value_counts()
                .sort_index()
                .to_dict()
                if not out.empty
                else {}
            ),

        "time_policy":
            "use OmniFall annotation start/end directly",

        "identity_policy":
            "subject + ADL/Fall partition + filename stem",

        "split_policy":
            "subject-aware split applied later by add_splits()",
    }

    (
        meta_dir
        / "gmdcsa24_manifest_resolution_diagnostics.json"
    ).write_text(
        json.dumps(
            diagnostics,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    if malformed:

        raise RuntimeError(
            "GMDCSA24 contains malformed OmniFall rows: "
            f"{len(malformed)}. "
            "See gmdcsa24_manifest_resolution_diagnostics.json."
        )

    if missing_video:

        raise RuntimeError(
            "GMDCSA24 physical-video resolution failed. "
            f"Unresolved rows: {len(missing_video)} / {len(labels)}. "
            f"Examples: {missing_video[:5]}. "
            "The resolver now uses "
            "subject + ADL/Fall + filename stem."
        )

    if out.empty:

        raise RuntimeError(
            "GMDCSA24 manifest is empty after canonical resolution. "
            f"Videos={len(videos)}, labels={len(labels)}."
        )

    if len(out) != len(labels):

        raise RuntimeError(
            "GMDCSA24 annotation mapping is incomplete: "
            f"resolved {len(out)} of {len(labels)} label rows."
        )

    observed_subjects = set(
        out["subject"].astype(str)
    )

    expected_subjects = {
        "Subject_1",
        "Subject_2",
        "Subject_3",
        "Subject_4",
    }

    missing_subjects = sorted(
        expected_subjects
        - observed_subjects
    )

    if missing_subjects:

        raise RuntimeError(
            "GMDCSA24 subject metadata is incomplete. "
            f"Expected {sorted(expected_subjects)}, "
            f"missing {missing_subjects}."
        )

    # The archive itself should contain the 160 published video clips.
    if len(videos) != expected_video_count:

        raise RuntimeError(
            "GMDCSA24 physical archive size is unexpected. "
            f"Found {len(videos)} videos; expected "
            f"{expected_video_count} MP4 clips for the verified dataset."
        )

    if len(video_map) != expected_video_count:

        raise RuntimeError(
            "GMDCSA24 canonical indexing did not produce one unique "
            f"identity per physical video. "
            f"Canonical keys={len(video_map)}, physical videos={len(videos)}. "
            "Inspect gmdcsa24_manifest_resolution_diagnostics.json."
        )

    if out["label"].nunique() < 2:

        raise RuntimeError(
            "GMDCSA24 binary label diversity is insufficient: "
            f"{out['label'].value_counts().to_dict()}"
        )

    print(
        "=" * 70
    )

    print(
        "GMDCSA24 MANIFEST"
    )

    print(
        "=" * 70
    )

    print(
        "Archive video files          :",
        len(videos),
    )

    print(
        "Canonical video keys         :",
        len(video_map),
    )

    print(
        "OmniFall label rows          :",
        len(labels),
    )

    print(
        "Resolved manifest rows       :",
        len(out),
    )

    print(
        "Missing video mappings       :",
        len(missing_video),
    )

    print(
        "Malformed annotations        :",
        len(malformed),
    )

    print(
        "Physical duplicate keys      :",
        len(video_duplicates),
    )

    print(
        "Unique physical videos used  :",
        out["video_path"].nunique(),
    )

    print(
        "Label counts                 :",
        out["label"]
        .value_counts()
        .to_dict(),
    )

    print(
        "Source-label counts          :",
        out["source_label"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )

    print(
        "Subject counts               :",
        out["subject"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )

    print(
        "Camera IDs                   :",
        sorted(
            out["camera"]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        ),
    )

    print(
        "Time policy                  :",
        "OmniFall annotation [start,end]",
    )

    print(
        "Identity policy              :",
        "subject + ADL/Fall + filename stem",
    )

    print(
        "=" * 70
    )

    return out

def make_caucafall_manifest(root):
    """
    Build the CAUCAFall manifest from OmniFall temporal annotations.

    Design:
      - OmniFall supplies 258 temporal segments for CAUCAFall.
      - CAUCAFall contains 100 logical videos arranged as
        10 subjects x 10 activities.
      - Multiple annotation rows may reference one physical video.
      - One deterministic physical representative is selected per
        (subject, activity) logical video key.
      - Explicit subject boundaries are never crossed.
      - The OmniFall temporal interval [start,end] is preserved directly.
    """
    root = Path(root)

    meta_dir = DIRS["metadata"] / "omnifall"
    meta_dir.mkdir(parents=True, exist_ok=True)

    labels_path = meta_dir / "caucafall.csv"

    labels = _read_omnifall_labels(
        "labels/caucafall.csv",
        labels_path,
        "CAUCAFall",
    )

    video_index = _index_videos(root)
    videos = video_index["videos"]

    logical_keys = []
    malformed = []

    def clean_activity(value):
        c = _compact(value)
        return re.sub(r"s0*\d+$", "", c)

    def annotation_identity(raw_path):
        key = _normalize_path_key(raw_path)

        if not key:
            return None, None, ""

        parts = [x for x in key.split("/") if x]
        base = parts[-1]

        base_compact = _compact(base)

        subject = _parse_subject(base)
        if subject is None:
            subject = _parse_subject(raw_path)

        activity = clean_activity(base_compact)

        category = parts[0] if len(parts) > 1 else ""

        return activity, subject, category

    def row_subject(row, raw_path):
        if "subject" in labels.columns:
            value = _safe_int(row["subject"], default=None)
            if value is not None and value >= 0:
                return value
        return _parse_subject(raw_path)

    def row_camera(row):
        if "cam" in labels.columns:
            value = _safe_int(row["cam"], default=None)
            if value is not None and value >= 0:
                return value
        return 1

    for row_idx, row in labels.iterrows():
        raw_path = str(row["path"]).strip()

        source_label = _safe_int(row["label"])
        start = _safe_float(row["start"])
        end = _safe_float(row["end"])

        if source_label is None or start is None or end is None:
            malformed.append({
                "row": int(row_idx),
                "path": raw_path,
                "reason": "non_numeric_annotation_fields",
            })
            continue

        if (
            not np.isfinite(start)
            or not np.isfinite(end)
            or end <= start
        ):
            malformed.append({
                "row": int(row_idx),
                "path": raw_path,
                "start": start,
                "end": end,
                "reason": "invalid_annotation_interval",
            })
            continue

        activity_key, path_subject, category = annotation_identity(raw_path)

        subject = row_subject(row, raw_path)
        camera = row_camera(row)

        if not activity_key:
            malformed.append({
                "row": int(row_idx),
                "path": raw_path,
                "reason": "invalid_logical_activity",
            })
            continue

        logical_keys.append({
            "row": int(row_idx),
            "raw_path": raw_path,
            "activity_key": activity_key,
            "path_subject": path_subject,
            "subject": subject,
            "category": category,
            "camera": camera,
            "label": int(source_label),
            "start": float(start),
            "end": float(end),
        })

    if malformed:
        print("[CAUCAFall] malformed annotation rows:", len(malformed))

    if not logical_keys:
        raise RuntimeError(
            "CAUCAFall contains no usable OmniFall annotations."
        )

    # ------------------------------------------------------------
    # Physical-file records
    # ------------------------------------------------------------

    video_records = []

    for vp in videos:
        rel = _relative_video_path(vp, root)
        rel_norm = _normalize_path_key(rel)
        stem_compact = _compact(Path(vp).stem)
        stem_activity = clean_activity(stem_compact)

        video_records.append({
            "path": vp,
            "rel": rel,
            "norm": rel_norm,
            "compact": _compact(rel_norm),
            "stem_compact": stem_compact,
            "stem_activity": stem_activity,
            "subject": _parse_subject(rel),
            "tokens": [
                _compact(x)
                for x in rel_norm.split("/")
                if _compact(x)
            ],
        })

    def candidate_score(
        rec,
        activity_key,
        subject,
        category,
        raw_path,
    ):
        score = 0

        raw_stem = _compact(Path(raw_path).stem)
        raw_activity = clean_activity(raw_stem)

        if subject is not None and rec["subject"] == subject:
            score += 10000
        elif subject is not None and rec["subject"] is None:
            score += 100

        if rec["stem_activity"] == activity_key:
            score += 9000
        elif raw_activity == rec["stem_activity"]:
            score += 8500
        elif rec["stem_compact"] == raw_stem:
            score += 8000
        elif raw_stem in rec["stem_compact"]:
            score += 7000
        elif activity_key in rec["compact"]:
            score += 5000

        if category:
            if _compact(category) in rec["tokens"]:
                score += 1000

        score += max(0, 100 - len(rec["tokens"]) * 5)

        if rec["path"].suffix.lower() == ".avi":
            score += 50

        return score

    representative_cache = {}
    representative_diagnostics = []
    unresolved_logical_keys = []

    for item in logical_keys:
        subject = item["subject"]
        activity_key = item["activity_key"]
        logical_id = (subject, activity_key)

        if logical_id in representative_cache:
            continue

        candidates = []

        for rec in video_records:
            rec_subject = rec["subject"]

            # Never cross an explicit subject boundary.
            if (
                subject is not None
                and rec_subject is not None
                and rec_subject != subject
            ):
                continue

            score = candidate_score(
                rec,
                activity_key,
                subject,
                item["category"],
                item["raw_path"],
            )

            raw_key = _normalize_path_key(item["raw_path"])
            raw_stem = _compact(Path(item["raw_path"]).stem)

            activity_ok = (
                rec["stem_activity"] == activity_key
                or activity_key in rec["compact"]
                or raw_stem in rec["compact"]
                or raw_key in rec["norm"]
                or rec["norm"].endswith("/" + raw_key)
            )

            if not activity_ok:
                continue

            candidates.append((score, rec))

        if not candidates:
            unresolved_logical_keys.append({
                "subject": subject,
                "activity_key": activity_key,
                "raw_path": item["raw_path"],
                "category": item["category"],
            })
            continue

        candidates.sort(
            key=lambda x: (
                -x[0],
                len(x[1]["tokens"]),
                str(x[1]["path"]).lower(),
            )
        )

        best_score = candidates[0][0]
        best = [
            rec
            for score, rec in candidates
            if score == best_score
        ]

        if len(best) > 1 and subject is not None:
            subject_tokens = {
                _compact(f"s{subject}"),
                _compact(f"s{subject:02d}"),
                _compact(f"s{subject:03d}"),
                _compact(f"subject{subject}"),
                _compact(f"subject{subject:02d}"),
                _compact(f"subject{subject:03d}"),
            }

            explicit_subject = [
                rec
                for rec in best
                if any(tok in subject_tokens for tok in rec["tokens"])
            ]

            if explicit_subject:
                best = explicit_subject

        selected = sorted(
            best,
            key=lambda rec: str(rec["path"]).lower(),
        )[0]

        representative_cache[logical_id] = selected

        representative_diagnostics.append({
            "subject": subject,
            "activity_key": activity_key,
            "selected": str(selected["path"]),
            "candidate_count": len(candidates),
            "best_score": best_score,
            "top_equivalent_candidates": [
                str(rec["path"])
                for rec in best[:10]
            ],
        })

    rows = []
    missing = []

    for item in logical_keys:
        logical_id = (item["subject"], item["activity_key"])

        rec = representative_cache.get(logical_id)

        if rec is None:
            missing.append({
                "row": item["row"],
                "path": item["raw_path"],
                "activity_key": item["activity_key"],
                "subject": item["subject"],
                "start": item["start"],
                "end": item["end"],
                "label": item["label"],
            })
            continue

        rows.append({
            "dataset": "CAUCAFall",
            "video_path": str(rec["path"]),
            "subject": (
                item["subject"]
                if item["subject"] is not None
                else (
                    rec["subject"]
                    if rec["subject"] is not None
                    else "unknown_subject"
                )
            ),
            "camera": int(item["camera"]),
            "scenario": Path(item["raw_path"]).stem,
            "activity": _activity_name(item["label"]),
            "source_label": int(item["label"]),
            "label": int(item["label"] in POSITIVE_IDS),
            "start_s": float(item["start"]),
            "end_s": float(item["end"]),
        })

    out = pd.DataFrame(rows)

    diagnostics = {
        "extracted_root": str(root),
        "video_files_found": len(videos),
        "omnifall_label_rows": len(labels),
        "usable_annotation_rows": len(logical_keys),
        "resolved_manifest_rows": len(out),
        "missing_annotations": len(missing),
        "malformed_annotations": len(malformed),
        "unresolved_logical_video_keys": len(unresolved_logical_keys),
        "logical_video_keys": len(representative_cache),
        "unique_physical_videos_used": (
            int(out["video_path"].nunique()) if not out.empty else 0
        ),
        "expected_logical_videos": 100,
        "expected_omnifall_segments": 258,
        "resolution_policy": (
            "one deterministic physical representative per "
            "(subject,activity) logical video key"
        ),
        "resolution_diagnostics": representative_diagnostics[:200],
        "unresolved_logical_keys": unresolved_logical_keys,
        "label_space": "OmniFall staged classes 0-15; binary fall={1:fall,2:fallen}",
        "time_policy": "use OmniFall annotation start/end directly",
    }

    (
        meta_dir / "caucafall_resolution_diagnostics.json"
    ).write_text(
        json.dumps(diagnostics, indent=2, default=str),
        encoding="utf-8",
    )

    if missing:
        (
            meta_dir / "caucafall_missing_annotations.json"
        ).write_text(
            json.dumps(missing, indent=2, default=str),
            encoding="utf-8",
        )

    if unresolved_logical_keys:
        (
            meta_dir / "caucafall_unresolved_logical_keys.json"
        ).write_text(
            json.dumps(unresolved_logical_keys, indent=2, default=str),
            encoding="utf-8",
        )

    if malformed:
        (
            meta_dir / "caucafall_malformed_rows.json"
        ).write_text(
            json.dumps(malformed, indent=2, default=str),
            encoding="utf-8",
        )

    # ------------------------------------------------------------
    # Hard validation
    # ------------------------------------------------------------

    if out.empty:
        raise RuntimeError(
            "CAUCAFall manifest is empty. "
            f"Videos={len(videos)}, labels={len(labels)}, "
            f"usable_annotations={len(logical_keys)}, "
            f"missing={len(missing)}, malformed={len(malformed)}, "
            f"unresolved_logical_keys={len(unresolved_logical_keys)}."
        )

    if len(out) != len(logical_keys):
        raise RuntimeError(
            "CAUCAFall annotation mapping is incomplete. "
            f"Resolved {len(out)} of {len(logical_keys)} usable "
            "OmniFall annotation rows. "
            f"Missing={len(missing)} | "
            f"Malformed={len(malformed)} | "
            f"Unresolved logical keys={len(unresolved_logical_keys)}."
        )

    if malformed:
        raise RuntimeError(
            "CAUCAFall contains malformed OmniFall annotation rows: "
            f"{len(malformed)}."
        )

    if unresolved_logical_keys:
        raise RuntimeError(
            "CAUCAFall contains unresolved logical video keys: "
            f"{len(unresolved_logical_keys)}."
        )

    if out["label"].nunique() < 2:
        raise RuntimeError(
            "CAUCAFall contains fewer than two binary classes. "
            f"Label counts: {out['label'].value_counts().to_dict()}"
        )

    bad_interval = out[out["end_s"] <= out["start_s"]]
    if not bad_interval.empty:
        raise RuntimeError(
            "CAUCAFall contains non-positive annotation intervals: "
            f"{len(bad_interval)}"
        )

    nonexistent = [
        p for p in out["video_path"]
        if not Path(p).exists()
    ]

    if nonexistent:
        raise RuntimeError(
            "CAUCAFall manifest contains nonexistent physical files: "
            f"{len(nonexistent)}"
        )

    observed_subjects = set()
    for value in out["subject"]:
        parsed = _safe_int(value, default=None)
        if parsed is not None:
            observed_subjects.add(parsed)

    expected_subjects = set(range(1, 11))

    if not expected_subjects.issubset(observed_subjects):
        raise RuntimeError(
            "CAUCAFall subject metadata is incomplete. "
            f"Expected subjects 1-10; observed {sorted(observed_subjects)}"
        )

    observed_cameras = sorted(
        out["camera"].dropna().astype(int).unique().tolist()
    )

    if len(observed_cameras) > 1:
        raise RuntimeError(
            "CAUCAFall unexpectedly contains multiple camera IDs. "
            f"Observed cameras: {observed_cameras}"
        )

    unique_physical = out["video_path"].nunique()

    if unique_physical > 110:
        raise RuntimeError(
            "CAUCAFall resolver produced too many distinct physical videos: "
            f"{unique_physical}. Expected approximately 100."
        )

    logical_unique = (
        out[["subject", "activity"]]
        .drop_duplicates()
        .shape[0]
    )

    if logical_unique > 110:
        raise RuntimeError(
            "CAUCAFall produced too many logical "
            f"(subject,activity) keys: {logical_unique}."
        )

    label_counts = out["label"].value_counts().to_dict()
    source_label_counts = (
        out["source_label"].value_counts().sort_index().to_dict()
    )
    subject_counts = (
        out["subject"].value_counts().sort_index().to_dict()
    )

    print("=" * 70)
    print("CAUCAFall OMNIFALL TEMPORAL MANIFEST")
    print("=" * 70)
    print("Archive video files          :", len(videos))
    print("OmniFall label rows          :", len(labels))
    print("Usable annotation rows       :", len(logical_keys))
    print("Resolved manifest rows       :", len(out))
    print("Missing annotations          :", len(missing))
    print("Unresolved logical keys      :", len(unresolved_logical_keys))
    print("Malformed annotations        :", len(malformed))
    print("Logical video keys resolved  :", len(representative_cache))
    print("Unique physical videos used  :", unique_physical)
    print("Expected logical videos      :", 100)
    print("Expected OmniFall segments   :", 258)
    print("Label counts                 :", label_counts)
    print("Source-label counts          :", source_label_counts)
    print("Subject counts               :", subject_counts)
    print("Camera IDs                   :", observed_cameras)
    print("Time policy                  :", "OmniFall annotation [start, end]")
    print("=" * 70)

    return out


# ================================================================
# Build requested manifests
# ================================================================

MANIFESTS = {}

if "MCFD" in train_dataset_names:
    MANIFESTS["MCFD"] = make_mcfd_manifest(
        EXTRACTED["MCFD"]
    )

if "GMDCSA24" in train_dataset_names:
    MANIFESTS["GMDCSA24"] = make_gmd_manifest(
        EXTRACTED["GMDCSA24"]
    )

if "CAUCAFall" in external_dataset_names:
    MANIFESTS["CAUCAFall"] = make_caucafall_manifest(
        EXTRACTED["CAUCAFall"]
    )


# ================================================================
# Global validation
# ================================================================

for k, df in MANIFESTS.items():

    if (
        df.empty
        or df["label"].dropna().nunique() < 2
    ):
        raise RuntimeError(
            f"{k}: manifest has insufficient class diversity "
            "or parsing failed. Inspect source layout before continuing."
        )

    if k == "GMDCSA24":
        expected_subjects = {
            "Subject_1",
            "Subject_2",
            "Subject_3",
            "Subject_4",
        }

        observed = set(df["subject"].astype(str))

        missing = sorted(expected_subjects - observed)

        if missing:
            raise RuntimeError(
                "GMDCSA24 subject metadata is incomplete; "
                f"expected {sorted(expected_subjects)}, missing {missing}. "
                "Refusing to create a potentially invalid subject split."
            )

    print(
        k,
        df.shape,
        df.groupby("label").size().to_dict(),
    )


MCFD PRE-SEGMENTED CLIP MANIFEST
Archive video files          : 1352
Canonical camera keys        : 192
OmniFall label rows          : 1352
Resolved manifest rows       : 1352
Missing video mappings       : 0
Missing split mappings       : 0
Malformed annotations        : 0
Split map entries            : 192
Label counts                 : {0: 912, 1: 440}
Split counts                 : {'test': 1014, 'train': 169, 'val': 169}
Camera counts                : {1: 169, 2: 169, 3: 169, 4: 169, 5: 169, 6: 169, 7: 169, 8: 169}
Physical duplicate keys     : 192
Time policy                  : OmniFall [start,end]
Protocol                    : cross-view / view-shift
GMDCSA24 MANIFEST
Archive video files          : 160
Canonical video keys         : 160
OmniFall label rows          : 458
Resolved manifest rows       : 458
Missing video mappings       : 0
Malformed annotations        : 0
Physical duplicate keys      : 0
Unique physical videos used  : 160
Label counts                 : {0: 303, 1:

In [21]:
# Split protocols. The default is intentionally explicit and deterministic.

def add_splits(manifest):
    df=manifest.copy()
    if df['dataset'].iloc[0]=='GMDCSA24':
        # Four subjects: deterministic subject-aware split.
        df['split']=df['subject'].map({'Subject_1':'train','Subject_2':'train','Subject_3':'val','Subject_4':'test'}).fillna('drop')
    elif df['dataset'].iloc[0]=='MCFD':
        # Split already comes from versioned OmniFall cross-view metadata.
        if 'split' not in df.columns: raise RuntimeError('MCFD split metadata missing.')
    elif df['dataset'].iloc[0]=='CAUCAFall':
        df['split']='external_test'
    return df[df['split']!='drop'].reset_index(drop=True)

FULL_MANIFEST=pd.concat([add_splits(df) for df in MANIFESTS.values()], ignore_index=True)

# Persist a lightweight manifest in the permanent Drive area.
FULL_MANIFEST.to_csv(DIRS['metadata']/'full_manifest.csv', index=False)
print(FULL_MANIFEST.groupby(['dataset','split','label']).size())

dataset    split          label
CAUCAFall  external_test  0        160
                          1         98
GMDCSA24   test           0         59
                          1         34
           train          0        161
                          1         80
           val            0         83
                          1         41
MCFD       test           0        684
                          1        330
           train          0        114
                          1         55
           val            0        114
                          1         55
dtype: int64


### Leakage guard

The training loader uses the manifest's explicit split field. No frame-level split is ever performed. GMDCSA24 uses subject-disjoint train/validation/test groups. MCFD uses camera-disjoint views for train/validation/test, but because the same underlying event can be observed by several cameras, its result is explicitly interpreted as **view shift**, not an independent-event test.

## 6. Runtime estimation before pose extraction

This stage estimates the number of videos, segments, sampled frames, and an actual pose throughput benchmark on the current Colab GPU. If the estimate is unsafe, the notebook can reduce optional experiments before touching the full pose cache. It does **not** silently throw away data; any budget-driven decision is written to `metadata/budget_decisions.json`.

In [22]:
def inspect_video_duration(vp):
    cap=cv2.VideoCapture(str(vp))
    if not cap.isOpened(): return None,None,None
    fps=cap.get(cv2.CAP_PROP_FPS) or 0
    n=cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    dur=(n/fps) if fps>0 else None
    return int(n),float(fps),dur

sample_df=FULL_MANIFEST[~((FULL_MANIFEST.dataset=='CAUCAFall') & (FULL_MANIFEST.split=='external_test'))].copy()
# One row is one labeled segment; only pose-extract each unique video+interval once.
unique_segments=sample_df[['dataset','video_path','start_s','end_s','label','activity','subject','camera','scenario']].drop_duplicates()

stats=[]
for vp in unique_segments.video_path.sample(min(20,len(unique_segments)), random_state=SEED):
    n,fps,dur=inspect_video_duration(vp)
    stats.append({'frames':n,'fps':fps,'duration':dur})
vid_stats=pd.DataFrame(stats)
print('Sampled video stats:', vid_stats.describe(include='all'))
print('Segments:', len(unique_segments), '| sampled frames target:', len(unique_segments)*32)

Sampled video stats:            frames         fps   duration
count   20.000000   20.000000  20.000000
mean   139.250000   95.230924   2.510985
std     57.484895   44.171235   2.485419
min     64.000000   15.001748   0.533333
25%     83.000000   97.455135   0.697917
50%    165.500000  120.000000   1.491667
75%    179.000000  120.000000   3.131292
max    277.000000  120.000000   8.865633
Segments: 1810 | sampled frames target: 57920


In [23]:
# ============================================================
# Realistic YOLO11n-Pose throughput benchmark
# ============================================================
#
# This benchmark measures actual YOLO11n-Pose inference throughput on
# real frames from the current train/validation manifest.
#
# Note:
# The canonical `read_sampled_frames()` helper is defined later in
# the notebook. So this cell uses a benchmark-local reader.
#
# The local reader is built to handle:
#   - OpenCV random-seek failures,
#   - codec/backend seek inconsistencies,
#   - an individual unreadable video,
#   - an individual annotation interval that cannot be decoded.
#
# The benchmark tries several REAL train/validation segments in a
# deterministic order and uses the first one from which >= 8 frames
# can actually be decoded.
#
# No synthetic frames are created.
# No labels or splits are changed.
# No downstream pose-extraction functions are modified.
#
# This is an operational throughput estimate only; it does not alter
# the scientific representation or labels.
# ============================================================

POSE_MODEL_NAME = "yolo11n-pose.pt"
POSE_IMAGE_SIZE = 640
POSE_INFER_BATCH = 32
POSE_FP16 = bool(torch.cuda.is_available())

pose_model = YOLO(POSE_MODEL_NAME)
pose_model.to(torch_device)


# ------------------------------------------------------------
# Benchmark-local robust frame reader.
# ------------------------------------------------------------

def _benchmark_read_frames(
    video_path,
    start_s=0.0,
    end_s=None,
    T=32,
):
    """
    Read up to T real RGB frames from one annotated interval.

    Strategy:
      1. Open video.
      2. Compute valid frame interval from fps/frame count.
      3. Try direct seek to the interval.
      4. If seek/decode is insufficient, reopen and sequentially scan.
      5. Return decoded frames plus a status string.

    This helper is benchmark-only and intentionally does not replace
    the canonical `sample_indices()` / `read_sampled_frames()` later
    in the notebook.
    """

    video_path = str(
        video_path
    )

    cap = cv2.VideoCapture(
        video_path
    )

    if not cap.isOpened():
        return (
            [],
            None,
            "open_failed",
        )

    try:

        fps = float(
            cap.get(
                cv2.CAP_PROP_FPS
            )
            or 0.0
        )

        frame_count = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
            or 0
        )

        if fps <= 0:
            fps = 30.0

        if frame_count <= 0:
            return (
                [],
                fps,
                "no_frames",
            )

        start_s = max(
            0.0,
            float(
                start_s
                if start_s is not None
                else 0.0
            ),
        )

        if (
            end_s is None
            or not np.isfinite(
                float(end_s)
            )
        ):

            end_s = (
                frame_count - 1
            ) / fps

        else:

            end_s = max(
                start_s,
                float(end_s),
            )

        start_idx = min(
            frame_count - 1,
            max(
                0,
                int(
                    round(
                        start_s * fps
                    )
                ),
            ),
        )

        end_idx = min(
            frame_count - 1,
            max(
                start_idx,
                int(
                    round(
                        end_s * fps
                    )
                ),
            ),
        )

        if end_idx < start_idx:
            end_idx = start_idx

        target_indices = (
            np.linspace(
                start_idx,
                end_idx,
                T,
            )
            .round()
            .astype(int)
        )

        target_indices = np.clip(
            target_indices,
            start_idx,
            end_idx,
        )

        unique_targets = {
            int(i)
            for i in np.unique(
                target_indices
            )
        }


        # --------------------------------------------------------
        # Attempt 1:
        # Direct seek to the annotated interval.
        # --------------------------------------------------------

        frame_map = {}

        seek_ok = cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(start_idx),
        )

        if seek_ok:

            current_idx = int(
                cap.get(
                    cv2.CAP_PROP_POS_FRAMES
                )
                or start_idx
            )

            # OpenCV backends can report an imprecise seek position.
            # Trust the actual reported/current frame only if it remains
            # within a sensible range; otherwise restart sequentially.
            if current_idx > end_idx:
                current_idx = start_idx

            while (
                current_idx <= end_idx
                and len(frame_map)
                < len(unique_targets)
            ):

                ok, frame = cap.read()

                if not ok:
                    break

                if current_idx in unique_targets:

                    frame_map[
                        current_idx
                    ] = cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB,
                    )

                current_idx += 1


        frames = [
            frame_map.get(
                int(idx)
            )
            for idx in target_indices
        ]

        valid_count = sum(
            frame is not None
            for frame in frames
        )


        # --------------------------------------------------------
        # Attempt 2:
        # Sequential fallback from frame 0.
        #
        # This is important for codecs where CAP_PROP_POS_FRAMES
        # succeeds syntactically but lands at a bad position.
        # --------------------------------------------------------

        if valid_count < min(8, T):

            cap.release()

            cap = cv2.VideoCapture(
                video_path
            )

            if not cap.isOpened():

                return (
                    [
                        frame
                        for frame
                        in frames
                        if frame is not None
                    ],
                    fps,
                    "reopen_failed",
                )

            frame_map = {}

            current_idx = 0

            while (
                current_idx <= end_idx
                and len(frame_map)
                < len(unique_targets)
            ):

                ok, frame = cap.read()

                if not ok:
                    break

                if (
                    current_idx >= start_idx
                    and current_idx
                    in unique_targets
                ):

                    frame_map[
                        current_idx
                    ] = cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB,
                    )

                current_idx += 1


            frames = [
                frame_map.get(
                    int(idx)
                )
                for idx in target_indices
            ]

            valid_count = sum(
                frame is not None
                for frame in frames
            )


        if valid_count < min(8, T):

            return (
                [
                    frame
                    for frame
                    in frames
                    if frame is not None
                ],
                fps,
                "insufficient_frames",
            )


        # --------------------------------------------------------
        # Fill occasional missing sampled indices from decoded
        # real frames while preserving exactly T inputs.
        # --------------------------------------------------------

        valid_frames = [
            frame
            for frame in frames
            if frame is not None
        ]

        final_frames = []

        for i, frame in enumerate(
            frames
        ):

            if frame is None:

                frame = valid_frames[
                    i % len(valid_frames)
                ]

            final_frames.append(
                frame
            )

        return (
            final_frames,
            fps,
            "ok",
        )

    finally:

        cap.release()


# ------------------------------------------------------------
# Build a realistic 32-frame benchmark sample from an actual
# train/validation segment.
# ------------------------------------------------------------

if (
    "unique_segments" not in globals()
    or unique_segments is None
    or len(unique_segments) == 0
):

    raise RuntimeError(
        "The benchmark requires `unique_segments` from the preceding "
        "runtime-estimation cell. Run the preceding cell first."
    )


# ------------------------------------------------------------
# Deterministic candidate order.
#
# We avoid relying on only the first row because one
# malformed/unreadable video should not invalidate the benchmark.
# ------------------------------------------------------------

required_benchmark_columns = {
    "dataset",
    "video_path",
    "start_s",
    "end_s",
}

missing_benchmark_columns = (
    required_benchmark_columns
    - set(
        unique_segments.columns
    )
)

if missing_benchmark_columns:

    raise RuntimeError(
        "unique_segments is missing required benchmark columns: "
        f"{sorted(missing_benchmark_columns)}"
    )


benchmark_candidates = (
    unique_segments
    .sort_values(
        [
            "dataset",
            "video_path",
            "start_s",
            "end_s",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


bench_row = None
bench_frames = None
bench_status = None
bench_failures = []

max_benchmark_candidates = min(
    32,
    len(
        benchmark_candidates
    ),
)


for candidate_idx in range(
    max_benchmark_candidates
):

    candidate = (
        benchmark_candidates.iloc[
            candidate_idx
        ]
    )

    try:

        candidate_start = float(
            candidate.start_s
        )

    except Exception:

        bench_failures.append({
            "candidate_index":
                int(candidate_idx),
            "video_path":
                str(candidate.video_path),
            "status":
                "invalid_start_time",
        })

        continue


    if pd.isna(
        candidate.end_s
    ):

        candidate_end = None

    else:

        try:

            candidate_end = float(
                candidate.end_s
            )

        except Exception:

            candidate_end = None


    (
        candidate_frames,
        candidate_fps,
        candidate_status,
    ) = _benchmark_read_frames(
        candidate.video_path,
        candidate_start,
        candidate_end,
        T=32,
    )


    if len(candidate_frames) >= 8:

        bench_row = candidate
        bench_frames = candidate_frames
        bench_status = candidate_status

        print(
            "Benchmark source selected:"
        )

        print(
            f"  dataset     : {candidate.dataset}"
        )

        print(
            f"  video       : {candidate.video_path}"
        )

        print(
            f"  interval    : "
            f"[{candidate_start:.4f}, "
            f"{candidate_end if candidate_end is not None else 'END'}]"
        )

        print(
            f"  decoded     : {len(candidate_frames)}/32"
        )

        print(
            f"  source FPS  : {candidate_fps:.3f}"
        )

        print(
            f"  reader mode : {candidate_status}"
        )

        break


    bench_failures.append({
        "candidate_index":
            int(candidate_idx),

        "dataset":
            str(candidate.dataset),

        "video_path":
            str(candidate.video_path),

        "start_s":
            candidate_start,

        "end_s":
            candidate_end,

        "status":
            candidate_status,

        "decoded_frames":
            len(candidate_frames),
    })


if (
    bench_row is None
    or bench_frames is None
):

    raise RuntimeError(
        "Could not obtain enough real benchmark frames from the first "
        f"{max_benchmark_candidates} train/validation candidates. "
        f"Failures: {bench_failures[:8]}"
    )


bench_valid = [
    frame
    for frame in bench_frames
    if frame is not None
]

if len(bench_valid) < 8:

    raise RuntimeError(
        "Could not obtain enough real benchmark frames: "
        f"{len(bench_valid)}/32"
    )


# ------------------------------------------------------------
# Preserve a fixed batch of exactly 32 real images.
# ------------------------------------------------------------

bench_images = []

for i in range(32):

    frame = bench_frames[i]

    if frame is None:

        frame = bench_valid[
            i % len(bench_valid)
        ]

    bench_images.append(
        frame
    )


# ------------------------------------------------------------
# Warm-up passes.
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.synchronize()


for _ in range(2):

    _ = pose_model.predict(
        source=bench_images,
        imgsz=POSE_IMAGE_SIZE,
        batch=POSE_INFER_BATCH,
        conf=0.15,
        iou=0.70,
        max_det=10,
        half=POSE_FP16,
        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),
        verbose=False,
        stream=False,
    )


if torch.cuda.is_available():
    torch.cuda.synchronize()


# ------------------------------------------------------------
# Timed passes.
# ------------------------------------------------------------

timed_passes = 3
times_s = []


for _ in range(
    timed_passes
):

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()


    _ = pose_model.predict(
        source=bench_images,
        imgsz=POSE_IMAGE_SIZE,
        batch=POSE_INFER_BATCH,
        conf=0.15,
        iou=0.70,
        max_det=10,
        half=POSE_FP16,
        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),
        verbose=False,
        stream=False,
    )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = max(
        time.perf_counter() - t0,
        1e-3,
    )

    times_s.append(
        elapsed
    )


median_pass_s = float(
    np.median(
        times_s
    )
)


pose_fps = (
    len(bench_images)
    / max(
        median_pass_s,
        1e-3,
    )
)


print(
    f"Measured batched YOLO11n-Pose throughput: "
    f"{pose_fps:.1f} frames/s "
    f"(batch={POSE_INFER_BATCH}, "
    f"imgsz={POSE_IMAGE_SIZE}, "
    f"FP16={POSE_FP16}, "
    f"median of {timed_passes} timed passes)."
)


# ------------------------------------------------------------
# Estimate the ACTUAL pose workload that will be extracted.
#
# The extraction workload is train+validation only. Test and the
# external dataset are evaluated AFTER the final model is frozen.
# ------------------------------------------------------------

pose_work_df = FULL_MANIFEST[
    FULL_MANIFEST.split.isin(
        ["train", "val"]
    )
].copy()


pose_work_segments = (
    pose_work_df[
        [
            "dataset",
            "video_path",
            "start_s",
            "end_s",
            "label",
            "activity",
            "subject",
            "camera",
            "scenario",
        ]
    ]
    .drop_duplicates()
)


if pose_work_segments.empty:

    raise RuntimeError(
        "Train+validation pose workload is empty. "
        "Cannot estimate pose-extraction time."
    )


total_pose_frames = (
    len(
        pose_work_segments
    )
    * 32
)


# ------------------------------------------------------------
# Conservative scheduling estimate.
#
# The 1.5x factor is only a scheduling guard.
# It is NOT a scientific performance result.
# ------------------------------------------------------------

POSE_TIME_SAFETY_FACTOR = 1.5


estimated_pose_s = (
    total_pose_frames
    / max(
        pose_fps,
        1e-3,
    )
    * POSE_TIME_SAFETY_FACTOR
)


estimated_decode_s = max(
    90.0,
    len(
        pose_work_segments
    )
    * 0.12,
)


estimated_total_s = (
    estimated_pose_s
    + estimated_decode_s
)


print(
    f"Train+val pose segments       : "
    f"{len(pose_work_segments)}"
)


print(
    f"Train+val sampled frames      : "
    f"{total_pose_frames}"
)


print(
    f"Conservative pose estimate    : "
    f"{estimated_total_s / 60:.1f} min"
)


print(
    f"Current runtime remaining     : "
    f"{BUDGET.remaining_hard / 60:.1f} min"
)


# ------------------------------------------------------------
# Do not let an obviously impossible preprocessing stage start.
# ------------------------------------------------------------

BUDGET.estimate_and_gate(
    "pose_extraction_estimate",
    estimated_total_s,
    essential=True,
)

Benchmark source selected:
  dataset     : GMDCSA24
  video       : /content/fall_detection_work/GMDCSA24/ekramalam-GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-5abac76/Subject 1/ADL/01.mp4
  interval    : [0.0000, 0.5]
  decoded     : 32/32
  source FPS  : 29.612
  reader mode : ok
Measured batched YOLO11n-Pose throughput: 236.0 frames/s (batch=32, imgsz=640, FP16=True, median of 3 timed passes).
Train+val pose segments       : 703
Train+val sampled frames      : 22496
Conservative pose estimate    : 3.9 min
Current runtime remaining     : 229.4 min


True

## 7. Frozen pose extraction with persistent NPZ caching

Only 32 uniformly sampled frames are decoded per labeled segment. Each segment is converted to a compressed NPZ and marked complete only after a successful write. If an NPZ exists and passes shape/version checks, pose estimation is skipped.

Primary-person selection uses detection confidence, mean keypoint confidence, box area, and temporal center continuity. Zero-confidence coordinates are treated as missing observations—not as real `(0,0)` points. Coordinates are body-centered using a hip midpoint when reliable, with shoulders/bounding-box center as fallbacks; scale is based on torso/shoulder-hip geometry.

In [24]:
CACHE_VERSION = "pose_v2_yolo11n_T32_C8_batch32_fp16"
POSE_DIR = DIRS["pose_cache"] / CACHE_VERSION
POSE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

POSE_IMAGE_SIZE = 640
POSE_INFER_BATCH = 32
POSE_FP16 = bool(torch.cuda.is_available())

COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (0,5),(0,6),(5,6),
    (5,7),(7,9),
    (6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),
    (12,14),(14,16),
]

KPT_CONF_MIN = 0.20


def sample_indices(cap, start_s, end_s, T=32):
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    if n <= 0:
        raise ValueError("Video has no frames")

    start = max(
        0,
        int(round(start_s * fps))
    )

    end = (
        n - 1
        if end_s is None
        or not np.isfinite(end_s)
        else min(
            n - 1,
            int(round(end_s * fps))
        )
    )

    if end < start:
        end = start

    idx = (
        np.linspace(
            start,
            end,
            T
        )
        .round()
        .astype(int)
    )

    return idx, fps


def read_sampled_frames(
    video_path,
    start_s=0,
    end_s=None,
    T=32,
):
    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise IOError(
            f"Cannot open video: {video_path}"
        )

    idxs, fps = sample_indices(
        cap,
        start_s,
        end_s,
        T
    )

    start_idx = int(
        idxs[0]
    )

    end_idx = int(
        idxs[-1]
    )

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        start_idx
    )

    wanted = {
        int(i)
        for i in idxs
    }

    frame_map = {}

    for frame_idx in range(
        start_idx,
        end_idx + 1
    ):

        ok, frame = cap.read()

        if not ok:
            break

        if frame_idx in wanted:

            frame_map[frame_idx] = (
                cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )
            )

    cap.release()

    frames = [
        frame_map.get(
            int(i)
        )
        for i in idxs
    ]

    times = np.asarray(
        [
            float(i / fps)
            for i in idxs
        ],
        dtype=np.float32,
    )

    return (
        frames,
        times,
        fps,
    )


def choose_primary_person(
    result,
    prev_center=None
):
    if (
        result.boxes is None
        or len(result.boxes) == 0
        or result.keypoints is None
    ):
        return None, None, None

    xy = (
        result.keypoints.xy
        .detach()
        .cpu()
        .numpy()
    )

    kc = (
        result.keypoints.conf
        .detach()
        .cpu()
        .numpy()
        if result.keypoints.conf is not None
        else np.ones(
            xy.shape[:2],
            dtype=np.float32
        )
    )

    bconf = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
        if result.boxes.conf is not None
        else np.ones(
            len(xy),
            dtype=np.float32
        )
    )

    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    H, W = result.orig_shape
    img_area = max(
        float(H * W),
        1.0
    )

    best = None

    for i in range(
        len(xy)
    ):

        conf = float(
            np.mean(kc[i])
        )

        area = max(
            0.0,
            float(
                (boxes[i, 2] - boxes[i, 0])
                * (boxes[i, 3] - boxes[i, 1])
            )
        ) / img_area

        cx = float(
            (boxes[i, 0] + boxes[i, 2]) / 2
        )

        cy = float(
            (boxes[i, 1] + boxes[i, 3]) / 2
        )

        continuity = 0.0

        if prev_center is not None:

            diag = (
                H**2 + W**2
            ) ** 0.5

            continuity = max(
                0.0,
                1.0
                - np.hypot(
                    cx - prev_center[0],
                    cy - prev_center[1],
                ) / max(
                    diag,
                    1.0
                )
            )

        score = (
            0.45 * float(bconf[i])
            + 0.30 * conf
            + 0.15 * min(area * 8, 1.0)
            + 0.10 * continuity
        )

        if (
            best is None
            or score > best[0]
        ):
            best = (
                score,
                i,
                (cx, cy),
                xy[i],
                kc[i],
            )

    _, i, center, kpts, kcps = best

    return (
        kpts.astype(
            np.float32
        ),
        kcps.astype(
            np.float32
        ),
        center,
    )


def normalize_skeleton(
    kpts,
    conf
):
    T, V, _ = kpts.shape

    xy = kpts.copy().astype(
        np.float32
    )

    c = conf.copy().astype(
        np.float32
    )

    xy[
        c < KPT_CONF_MIN
    ] = np.nan

    out = np.zeros_like(
        xy,
        dtype=np.float32
    )

    valid_center = []
    valid_scale = []

    for t in range(T):

        def mean_joint(ids):

            pts = [
                xy[t, i]
                for i in ids
                if np.isfinite(
                    xy[t, i]
                ).all()
            ]

            return (
                np.mean(
                    pts,
                    axis=0
                )
                if pts
                else None
            )

        hips = mean_joint(
            [11, 12]
        )

        shoulders = mean_joint(
            [5, 6]
        )

        center = (
            hips
            if hips is not None
            else shoulders
            if shoulders is not None
            else mean_joint(
                list(range(V))
            )
        )

        if center is None:
            center = np.array(
                [np.nan, np.nan],
                dtype=np.float32
            )

        valid_center.append(
            center
        )

        sw = 0.0

        if (
            np.isfinite(
                xy[t, 5]
            ).all()
            and np.isfinite(
                xy[t, 6]
            ).all()
        ):
            sw = float(
                np.linalg.norm(
                    xy[t, 5]
                    - xy[t, 6]
                )
            )

        torso = []

        if (
            hips is not None
            and shoulders is not None
        ):
            torso.append(
                float(
                    np.linalg.norm(
                        hips
                        - shoulders
                    )
                )
            )

        scale = max(
            [
                sw,
                *torso,
                1.0,
            ]
        ) if (
            sw
            or torso
        ) else 1.0

        valid_scale.append(
            scale
        )

    centers = np.asarray(
        valid_center,
        dtype=np.float32
    )

    scales = np.asarray(
        valid_scale,
        dtype=np.float32
    )

    for j in range(2):

        vals = centers[:, j]
        good = np.isfinite(
            vals
        )

        if good.any():

            centers[:, j] = np.interp(
                np.arange(T),
                np.where(good)[0],
                vals[good]
            )

        else:
            centers[:, j] = 0.0

    good = (
        np.isfinite(scales)
        & (scales > 0)
    )

    if good.any():

        scales = np.interp(
            np.arange(T),
            np.where(good)[0],
            scales[good]
        )

    else:

        scales = np.ones(
            T,
            dtype=np.float32
        )

    for t in range(T):

        good = np.isfinite(
            xy[t]
        ).all(axis=-1)

        out[
            t,
            good
        ] = (
            xy[t, good]
            - centers[t]
        ) / max(
            float(scales[t]),
            1e-3
        )

    return out, c


def add_motion_features(
    xy,
    conf,
    times
):
    dt = np.gradient(
        times
    ).astype(
        np.float32
    )

    dt[
        dt < 1e-3
    ] = 1 / 30

    vel = (
        np.gradient(
            xy,
            axis=0
        )
        / dt[:, None, None]
    )

    acc = (
        np.gradient(
            vel,
            axis=0
        )
        / dt[:, None, None]
    )

    speed = np.linalg.norm(
        vel,
        axis=-1,
        keepdims=True
    )

    feat = np.concatenate(
        [
            xy,
            conf[..., None],
            vel,
            acc,
            speed,
        ],
        axis=-1
    )

    feat = np.nan_to_num(
        feat,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    ).astype(
        np.float32
    )

    return feat


def pose_cache_path(row):
    key = (
        f"{row.dataset}|"
        f"{Path(row.video_path).as_posix()}|"
        f"{row.start_s:.4f}|"
        f"{row.end_s if pd.notna(row.end_s) else 'END'}|"
        f"{row.label}|"
        f"{row.activity}"
    )

    digest = hashlib.sha1(
        key.encode()
    ).hexdigest()[:20]

    return (
        POSE_DIR
        / f"{row.dataset}_{digest}.npz"
    )


def extract_one_pose(row):

    out_path = pose_cache_path(
        row
    )

    if out_path.exists():

        try:

            with np.load(
                out_path,
                allow_pickle=False
            ) as z:

                if (
                    z["features"].shape
                    == (32, 17, 8)
                    and int(z["label"])
                    == int(row.label)
                    and str(
                        z["cache_version"]
                    )
                    == CACHE_VERSION
                ):
                    return (
                        out_path,
                        True
                    )

        except Exception:
            pass

        out_path.unlink(
            missing_ok=True
        )

    frames, times, fps = (
        read_sampled_frames(
            row.video_path,
            float(row.start_s),
            row.end_s
            if pd.notna(row.end_s)
            else None,
            32,
        )
    )

    valid = [
        f
        for f in frames
        if f is not None
    ]

    if len(valid) < 8:

        raise ValueError(
            "Insufficient readable sampled "
            f"frames ({len(valid)}/32)"
        )

    # Failed frame decoding is represented by a zero image.
    images = [
        f
        if f is not None
        else np.zeros_like(
            valid[0]
        )
        for f in frames
    ]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Note:
    # A single call contains the entire 32-frame segment and explicitly
    # requests batched inference + FP16 on CUDA/T4.
    results = pose_model.predict(
        source=images,
        imgsz=POSE_IMAGE_SIZE,
        batch=POSE_INFER_BATCH,
        conf=0.15,
        iou=0.70,
        max_det=10,
        half=POSE_FP16,
        device=(
            0
            if torch.cuda.is_available()
            else "cpu"
        ),
        verbose=False,
        stream=False,
    )

    assert_vram_budget(
        "pose inference"
    )

    kpts = []
    confs = []
    prev_center = None

    for result in results:

        kp, kc, center = (
            choose_primary_person(
                result,
                prev_center
            )
        )

        if kp is None:

            kpts.append(
                np.full(
                    (17, 2),
                    np.nan,
                    dtype=np.float32
                )
            )

            confs.append(
                np.zeros(
                    17,
                    dtype=np.float32
                )
            )

        else:

            kpts.append(kp)
            confs.append(kc)
            prev_center = center

    xy = np.stack(
        kpts
    )

    cf = np.stack(
        confs
    )

    xy, cf = normalize_skeleton(
        xy,
        cf
    )

    features = add_motion_features(
        xy,
        cf,
        times
    )

    tmp = out_path.with_suffix(
        ".tmp.npz"
    )

    np.savez_compressed(
        tmp,
        features=features,
        label=np.int64(row.label),
        dataset=str(row.dataset),
        subject=str(row.subject),
        camera=np.int64(
            row.camera
            if pd.notna(row.camera)
            else -1
        ),
        activity=str(row.activity),
        scenario=str(row.scenario),
        video_path=str(row.video_path),
        start_s=np.float32(
            row.start_s
        ),
        end_s=np.float32(
            row.end_s
            if pd.notna(row.end_s)
            else times[-1]
        ),
        fps=np.float32(fps),
        cache_version=CACHE_VERSION,
    )

    os.replace(
        tmp,
        out_path
    )

    return (
        out_path,
        False
    )


def ensure_pose_manifest(
    input_df,
    manifest_path,
    desc,
    essential=True,
    checkpoint_every=20,
):
    """Extract pose features for the requested rows with resumable caching."""

    if input_df.empty:
        return pd.DataFrame()

    input_df = (
        input_df
        .copy()
        .reset_index(drop=True)
    )

    rows = []
    errors = []

    for i, row in tqdm(
        input_df.iterrows(),
        total=len(input_df),
        desc=desc,
    ):

        BUDGET.guard(
            "pose_extraction",
            essential=essential
        )

        try:

            p, cache_hit = extract_one_pose(
                row
            )

            rows.append({
                **row.to_dict(),
                "pose_path": str(p),
                "pose_cache_hit": cache_hit,
                "pose_cache_version": CACHE_VERSION,
            })

        except BudgetStop:
            raise

        except Exception as e:

            errors.append({
                **row.to_dict(),
                "error": repr(e),
            })

        if (
            (i + 1)
            % checkpoint_every
            == 0
        ):

            BUDGET.checkpoint(
                f"{desc}_checkpoint_{i+1}"
            )

    out = pd.DataFrame(
        rows
    )

    out.to_csv(
        manifest_path,
        index=False
    )

    if errors:

        pd.DataFrame(
            errors
        ).to_csv(
            DIRS["logs"]
            / (
                Path(manifest_path).stem
                + "_errors.csv"
            ),
            index=False
        )

        print(
            f"[WARN] {desc}: "
            f"{len(errors)} pose failures logged."
        )

    return out


In [26]:
# ============================================================
# Train + validation pose cache only
# ============================================================
#
# Test and external CAUCAFall pose features are deliberately deferred.
# This avoids spending the first expensive preprocessing stage on data
# that is not needed to fit the model.
#
# Note:
# This cell does not modify the canonical pose-extraction functions.
# It adds a local, resumable/retry execution layer around the already
# defined `extract_one_pose()` and `ensure_pose_manifest()` machinery.
#
# The retry layer handles:
#   - OpenCV random-seek / decoding failures,
#   - temporarily unreadable sampled frames,
#   - CUDA/VRAM-related failures by retrying with smaller batches,
#   - stale/incomplete cache files,
#   - previously failed rows.
#
# The original global reader and batch size are restored after this cell.
# ============================================================


POSE_MANIFEST_PATH = (
    DIRS["metadata"]
    / "pose_manifest.csv"
)


# ------------------------------------------------------------
# Basic dependency checks.
# ------------------------------------------------------------

_required_globals = [
    "FULL_MANIFEST",
    "extract_one_pose",
    "pose_cache_path",
    "pose_model",
    "POSE_DIR",
    "CACHE_VERSION",
]

_missing_globals = [
    name
    for name in _required_globals
    if name not in globals()
]

if _missing_globals:

    raise RuntimeError(
        "Train+validation pose cache requires the following "
        f"definitions, which are currently missing: {_missing_globals}"
    )


# ------------------------------------------------------------
# Build the requested train+validation subset.
# ------------------------------------------------------------

trainval_df = FULL_MANIFEST[
    FULL_MANIFEST["split"].isin(
        ["train", "val"]
    )
].copy()


if trainval_df.empty:

    raise RuntimeError(
        "The train+validation manifest is empty. "
        "No pose features can be extracted."
    )


trainval_df = (
    trainval_df
    .reset_index(
        drop=True
    )
)


print(
    "Train+val manifest rows:",
    len(trainval_df),
)


# ------------------------------------------------------------
# Preserve the original global state.
#
# The canonical extraction cell defines `read_sampled_frames()` and
# `POSE_INFER_BATCH`. We temporarily override them only while this
# cell is running and restore them before returning.
# ------------------------------------------------------------

_original_read_sampled_frames = globals().get(
    "read_sampled_frames"
)

_original_pose_infer_batch = globals().get(
    "POSE_INFER_BATCH"
)

_original_pose_device = (
    getattr(
        pose_model,
        "device",
        None,
    )
)


# ------------------------------------------------------------
# Robust benchmark/extraction reader.
#
# The canonical reader is good for normal operation, but OpenCV's
# random frame seeking can fail on some compressed videos/backends.
#
# This fallback:
#   1. opens the file,
#   2. computes the requested frame indices,
#   3. tries direct seek,
#   4. falls back to sequential decoding from frame zero,
#   5. returns real decoded frames only,
#   6. requires >= 8 valid frames.
#
# It never invents a synthetic frame.
# Missing positions are later filled from other REAL decoded frames
# by `extract_one_pose()`.
# ------------------------------------------------------------

def _robust_read_sampled_frames(
    video_path,
    start_s=0,
    end_s=None,
    T=32,
):
    video_path = str(
        video_path
    )

    cap = cv2.VideoCapture(
        video_path
    )

    if not cap.isOpened():

        raise IOError(
            f"Cannot open video: {video_path}"
        )

    try:

        fps = float(
            cap.get(
                cv2.CAP_PROP_FPS
            )
            or 0
        )

        frame_count = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
            or 0
        )

        if fps <= 0:
            fps = 30.0

        if frame_count <= 0:

            raise ValueError(
                f"Video has no readable frames: {video_path}"
            )

        start_s = float(
            0.0
            if start_s is None
            else start_s
        )

        if not np.isfinite(
            start_s
        ):
            start_s = 0.0

        start_s = max(
            start_s,
            0.0,
        )

        if (
            end_s is None
            or not np.isfinite(
                float(end_s)
            )
        ):

            end_s = (
                frame_count - 1
            ) / fps

        else:

            end_s = max(
                float(end_s),
                start_s,
            )

        start_idx = min(
            frame_count - 1,
            max(
                0,
                int(
                    round(
                        start_s * fps
                    )
                ),
            ),
        )

        end_idx = min(
            frame_count - 1,
            max(
                start_idx,
                int(
                    round(
                        end_s * fps
                    )
                ),
            ),
        )

        target_indices = (
            np.linspace(
                start_idx,
                end_idx,
                T,
            )
            .round()
            .astype(int)
        )

        target_indices = np.clip(
            target_indices,
            start_idx,
            end_idx,
        )

        wanted = {
            int(i)
            for i in target_indices
        }


        # ========================================================
        # Attempt 1: direct seek
        # ========================================================

        frame_map = {}

        seek_result = cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            start_idx,
        )

        if seek_result:

            current_idx = int(
                cap.get(
                    cv2.CAP_PROP_POS_FRAMES
                )
                or start_idx
            )

            # Some backends report an invalid post-seek position.
            if current_idx < start_idx:
                current_idx = start_idx

            if current_idx <= end_idx:

                while (
                    current_idx <= end_idx
                    and len(frame_map)
                    < len(wanted)
                ):

                    ok, frame = cap.read()

                    if not ok:
                        break

                    if current_idx in wanted:

                        frame_map[
                            current_idx
                        ] = cv2.cvtColor(
                            frame,
                            cv2.COLOR_BGR2RGB,
                        )

                    current_idx += 1


        frames = [
            frame_map.get(
                int(idx)
            )
            for idx in target_indices
        ]

        valid_count = sum(
            frame is not None
            for frame in frames
        )


        # ========================================================
        # Attempt 2: sequential fallback
        # ========================================================

        if valid_count < min(
            8,
            T,
        ):

            cap.release()

            cap = cv2.VideoCapture(
                video_path
            )

            if not cap.isOpened():

                raise IOError(
                    f"Could not reopen video for sequential decoding: "
                    f"{video_path}"
                )

            frame_map = {}

            current_idx = 0

            while (
                current_idx <= end_idx
                and len(frame_map)
                < len(wanted)
            ):

                ok, frame = cap.read()

                if not ok:
                    break

                if (
                    current_idx >= start_idx
                    and current_idx in wanted
                ):

                    frame_map[
                        current_idx
                    ] = cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB,
                    )

                current_idx += 1


            frames = [
                frame_map.get(
                    int(idx)
                )
                for idx in target_indices
            ]

            valid_count = sum(
                frame is not None
                for frame in frames
            )


        if valid_count < min(
            8,
            T,
        ):

            raise ValueError(
                "Insufficient readable sampled frames "
                f"({valid_count}/{T}) for video={video_path}, "
                f"interval=[{start_s}, {end_s}]"
            )


        # --------------------------------------------------------
        # Return exactly T entries.
        #
        # Any missing sampled location is replaced by another
        # REAL decoded frame from this same interval.
        # --------------------------------------------------------

        valid_frames = [
            frame
            for frame in frames
            if frame is not None
        ]

        completed = []

        for i, frame in enumerate(
            frames
        ):

            if frame is None:

                frame = valid_frames[
                    i % len(valid_frames)
                ]

            completed.append(
                frame
            )


        times = np.asarray(
            [
                float(
                    idx / fps
                )
                for idx in target_indices
            ],
            dtype=np.float32,
        )

        return (
            completed,
            times,
            float(fps),
        )

    finally:

        cap.release()


# ------------------------------------------------------------
# Temporarily use robust reader.
# ------------------------------------------------------------

if (
    _original_read_sampled_frames
    is None
):

    raise RuntimeError(
        "The canonical `read_sampled_frames()` function is missing. "
        "Run the pose-extraction definitions cell first."
    )


read_sampled_frames = (
    _robust_read_sampled_frames
)


# ------------------------------------------------------------
# Cache/extraction helper.
#
# We intentionally process rows ourselves rather than calling
# `ensure_pose_manifest()` once, because its implementation catches
# every extraction exception and only reports the number of failures.
# That behavior is useful for ordinary logging, but not for recovery.
# ------------------------------------------------------------

def _cache_is_valid_for_row(
    row,
):
    """
    Check whether a previously generated NPZ is valid for this exact row.
    """
    try:

        cache_path = pose_cache_path(
            row
        )

    except Exception:

        return False


    if not cache_path.exists():

        return False


    try:

        with np.load(
            cache_path,
            allow_pickle=False,
        ) as z:

            if (
                "features" not in z
                or "label" not in z
                or "cache_version" not in z
            ):
                return False

            if tuple(
                z["features"].shape
            ) != (
                32,
                17,
                8,
            ):
                return False

            if int(
                z["label"]
            ) != int(
                row.label
            ):
                return False

            if str(
                z["cache_version"]
            ) != str(
                CACHE_VERSION
            ):
                return False

        return True

    except Exception:

        return False


def _is_cuda_memory_error(
    exc,
):
    text = str(
        exc
    ).lower()

    name = type(
        exc
    ).__name__.lower()

    return (
        "out of memory" in text
        or "cuda" in text
        and "memory" in text
        or "outofmemory" in name
    )


# ------------------------------------------------------------
# Extraction configuration.
#
# Start with the original batch size. Failed rows can be retried with
# progressively smaller batches without altering the notebook's
# permanent configuration.
# ------------------------------------------------------------

base_batch = int(
    _original_pose_infer_batch
    if _original_pose_infer_batch is not None
    else 32
)

retry_batches = []

for candidate in [
    base_batch,
    16,
    8,
    4,
]:

    candidate = max(
        1,
        int(candidate),
    )

    if candidate not in retry_batches:
        retry_batches.append(
            candidate
        )


# ------------------------------------------------------------
# Process each requested row with resumability + targeted retries.
# ------------------------------------------------------------

successful_rows = []
failed_rows = []

cache_hits = 0
fresh_extractions = 0


for row_idx, row in tqdm(
    trainval_df.iterrows(),
    total=len(
        trainval_df
    ),
    desc="Train+val pose cache",
):

    BUDGET.guard(
        "pose_extraction",
        essential=True,
    )


    # --------------------------------------------------------
    # Existing valid cache.
    # --------------------------------------------------------

    if _cache_is_valid_for_row(
        row
    ):

        successful_rows.append({
            **row.to_dict(),
            "pose_path": str(
                pose_cache_path(
                    row
                )
            ),
            "pose_cache_hit": True,
            "pose_cache_version":
                CACHE_VERSION,
        })

        cache_hits += 1

        continue


    # --------------------------------------------------------
    # Remove stale cache if present.
    # --------------------------------------------------------

    try:

        stale_path = pose_cache_path(
            row
        )

        stale_path.unlink(
            missing_ok=True
        )

    except Exception:
        pass


    row_success = False
    last_error = None


    # ========================================================
    # Retry loop
    # ========================================================

    for attempt, batch_size in enumerate(
        retry_batches
    ):

        BUDGET.guard(
            "pose_extraction_retry",
            essential=True,
        )


        # ----------------------------------------------------
        # Use smaller batch on later attempts.
        # ----------------------------------------------------

        POSE_INFER_BATCH = (
            batch_size
        )


        # ----------------------------------------------------
        # Clear CUDA allocator between retries.
        # ----------------------------------------------------

        if torch.cuda.is_available():

            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

            try:
                torch.cuda.synchronize()
            except Exception:
                pass


        try:

            out_path, cache_hit = (
                extract_one_pose(
                    row
                )
            )


            # Validate the produced cache immediately.
            if not _cache_is_valid_for_row(
                row
            ):

                raise RuntimeError(
                    "extract_one_pose() returned successfully, "
                    "but the resulting pose cache failed validation."
                )


            successful_rows.append({
                **row.to_dict(),
                "pose_path": str(
                    out_path
                ),
                "pose_cache_hit": bool(
                    cache_hit
                ),
                "pose_cache_version":
                    CACHE_VERSION,
            })


            if cache_hit:
                cache_hits += 1
            else:
                fresh_extractions += 1


            row_success = True
            break


        except BudgetStop:

            raise


        except Exception as exc:

            last_error = exc


            # ------------------------------------------------
            # Clean up an incomplete temporary cache.
            # ------------------------------------------------

            try:

                stale_path = pose_cache_path(
                    row
                )

                stale_path.unlink(
                    missing_ok=True
                )

                tmp_path = (
                    stale_path.with_suffix(
                        ".tmp.npz"
                    )
                )

                tmp_path.unlink(
                    missing_ok=True
                )

            except Exception:
                pass


            # ------------------------------------------------
            # CUDA/OOM:
            # continue with smaller batch.
            #
            # Other errors are also retried once or more because
            # OpenCV/video backends can fail transiently.
            # ------------------------------------------------

            if torch.cuda.is_available():

                try:
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                except Exception:
                    pass


            # Short diagnostic for the first failure.
            if attempt == 0:

                print(
                    "\n[POSE RETRY]",
                    f"row={row_idx}",
                    f"dataset={row.dataset}",
                    f"batch={batch_size}",
                    f"error={type(exc).__name__}: {exc}",
                )


    # ========================================================
    # Record unrecoverable row.
    # ========================================================

    if not row_success:

        failed_rows.append({
            **row.to_dict(),

            "error":
                repr(
                    last_error
                )
                if last_error is not None
                else "unknown_error",

            "error_type":
                type(
                    last_error
                ).__name__
                if last_error is not None
                else "unknown",

            "pose_cache_version":
                CACHE_VERSION,
        })


# ------------------------------------------------------------
# Restore the original global extraction configuration.
# ------------------------------------------------------------

POSE_INFER_BATCH = (
    _original_pose_infer_batch
    if _original_pose_infer_batch is not None
    else 32
)

if (
    _original_read_sampled_frames
    is not None
):

    read_sampled_frames = (
        _original_read_sampled_frames
    )


# ------------------------------------------------------------
# Build final manifest.
# ------------------------------------------------------------

POSE_MANIFEST = pd.DataFrame(
    successful_rows
)


# ------------------------------------------------------------
# Persist final manifest.
# ------------------------------------------------------------

POSE_MANIFEST.to_csv(
    POSE_MANIFEST_PATH,
    index=False,
)


# ------------------------------------------------------------
# Persist detailed failure log.
#
# The normal ensure_pose_manifest() function also writes an error
# log, but this cell's retry layer keeps richer information including
# row index and final exception type.
# ------------------------------------------------------------

pose_retry_error_path = (
    DIRS["logs"]
    / "pose_manifest_retry_errors.csv"
)

if failed_rows:

    pd.DataFrame(
        failed_rows
    ).to_csv(
        pose_retry_error_path,
        index=False,
    )

else:

    try:
        pose_retry_error_path.unlink(
            missing_ok=True
        )
    except Exception:
        pass


# ------------------------------------------------------------
# Report.
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRAIN + VALIDATION POSE CACHE")
print("=" * 70)

print(
    "Expected train+val rows     :",
    len(trainval_df),
)

print(
    "Cache hits                  :",
    cache_hits,
)

print(
    "Fresh successful extractions:",
    fresh_extractions,
)

print(
    "Successful pose rows        :",
    len(POSE_MANIFEST),
)

print(
    "Remaining failed rows       :",
    len(failed_rows),
)

print(
    "Pose cache version          :",
    CACHE_VERSION,
)

print(
    "Pose cache directory        :",
    POSE_DIR,
)

print(
    "Manifest path               :",
    POSE_MANIFEST_PATH,
)

print(
    "Retry error log             :",
    pose_retry_error_path,
)

print("=" * 70)


# ------------------------------------------------------------
# Hard validation.
# ------------------------------------------------------------

if POSE_MANIFEST.empty:

    raise RuntimeError(
        "No train/validation pose features were successfully cached. "
        "The retry layer could not recover any usable rows."
    )


if len(POSE_MANIFEST) != len(
    trainval_df
):

    # Show the first few concrete failures instead of only saying
    # "inspect the error log".
    examples = []

    for failure in failed_rows[:8]:

        examples.append({
            "dataset":
                failure.get("dataset"),

            "video_path":
                failure.get("video_path"),

            "start_s":
                failure.get("start_s"),

            "end_s":
                failure.get("end_s"),

            "error_type":
                failure.get("error_type"),

            "error":
                failure.get("error"),
        })


    raise RuntimeError(
        "Train/validation pose extraction is still incomplete after "
        "all local retries. "
        f"Cached={len(POSE_MANIFEST)} | "
        f"Expected={len(trainval_df)} | "
        f"Failed={len(failed_rows)}. "
        f"First failures: {examples}. "
        f"Full retry log: {pose_retry_error_path}"
    )


# ------------------------------------------------------------
# Final integrity check:
# every requested row must have a valid cache file.
# ------------------------------------------------------------

missing_cache_files = []

for row in (
    trainval_df.itertuples(
        index=False
    )
):

    if not _cache_is_valid_for_row(
        row
    ):

        missing_cache_files.append({
            "dataset":
                row.dataset,

            "video_path":
                row.video_path,

            "start_s":
                row.start_s,

            "end_s":
                row.end_s,
        })


if missing_cache_files:

    raise RuntimeError(
        "Pose manifest contains rows whose underlying NPZ cache is "
        "missing or invalid: "
        f"{len(missing_cache_files)}"
    )


print(
    "Train+val pose rows:",
    len(POSE_MANIFEST)
)

print(
    "Expected train+val rows:",
    len(trainval_df)
)

print(
    "[OK] Train+validation pose cache is complete."
)

Train+val manifest rows: 703


Train+val pose cache:   0%|          | 0/703 [00:00<?, ?it/s]


TRAIN + VALIDATION POSE CACHE
Expected train+val rows     : 703
Cache hits                  : 365
Fresh successful extractions: 338
Successful pose rows        : 703
Remaining failed rows       : 0
Pose cache version          : pose_v2_yolo11n_T32_C8_batch32_fp16
Pose cache directory        : /content/drive/MyDrive/fall_detection_project/pose_cache/pose_v2_yolo11n_T32_C8_batch32_fp16
Manifest path               : /content/drive/MyDrive/fall_detection_project/metadata/pose_manifest.csv
Retry error log             : /content/drive/MyDrive/fall_detection_project/logs/pose_manifest_retry_errors.csv
Train+val pose rows: 703
Expected train+val rows: 703
[OK] Train+validation pose cache is complete.


## 8. Train-only feature standardization

Coordinates are already body-centered/scaled per sequence. Derivatives can still have dataset-specific numeric ranges. A robust train-only standardizer is fit on the training split and then applied unchanged to validation/test/external data. Confidence stays on its natural `[0,1]` scale.

In [27]:
POSE_MANIFEST=pd.read_csv(POSE_MANIFEST_PATH)

# Exclude confidence channel (index 2) from z-scoring so confidence retains probabilistic meaning.
# Compute mean/std in streaming form to avoid materializing all training pose rows in RAM.
TRAIN_POSE=POSE_MANIFEST[POSE_MANIFEST.split=='train']
means=np.zeros(8,dtype=np.float32); stds=np.ones(8,dtype=np.float32)
sum_x=np.zeros(8,dtype=np.float64)
sum_x2=np.zeros(8,dtype=np.float64)
count=0
for p in TRAIN_POSE.pose_path:
    with np.load(p,allow_pickle=False) as z:
        x=z['features'].astype(np.float32).reshape(-1,8)
    sum_x += x.sum(axis=0,dtype=np.float64)
    sum_x2 += np.square(x,dtype=np.float32).sum(axis=0,dtype=np.float64)
    count += x.shape[0]
if count<=0:
    raise RuntimeError('Training pose cache is empty; cannot fit the train-only standardizer.')
for c in [0,1,3,4,5,6,7]:
    mu=sum_x[c]/count
    var=max(sum_x2[c]/count - mu*mu, 0.0)
    means[c]=np.float32(mu)
    stds[c]=np.float32(max(math.sqrt(var),1e-4))
STANDARDIZER={'means':means.tolist(),'stds':stds.tolist(),'confidence_channel':2}
(DIRS['metadata']/'pose_standardizer.json').write_text(json.dumps(STANDARDIZER,indent=2))
print(STANDARDIZER)

{'means': [-0.1446986198425293, -0.04387594759464264, 0.0, -0.033528730273246765, 0.044633422046899796, -0.44834792613983154, -3.2852373123168945, 5.280444622039795], 'stds': [7.34926700592041, 6.454782485961914, 1.0, 61.98717498779297, 89.68860626220703, 1180.80224609375, 2053.38330078125, 108.89708709716797], 'confidence_channel': 2}


## 9. PyTorch datasets and lightweight models

The proposed network has three small components: spatial graph reasoning, temporal convolutions, and a 2-head temporal attention block. Confidence statistics drive the fusion gate rather than simply being concatenated once.

In [28]:
class PoseSequenceDataset(Dataset):
    def __init__(self, df, means, stds, channel_mask=None, augment=False):
        self.df=df.reset_index(drop=True); self.means=np.asarray(means,np.float32); self.stds=np.asarray(stds,np.float32)
        self.mask=np.ones(8,dtype=np.float32) if channel_mask is None else np.asarray(channel_mask,np.float32)
        self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r=self.df.iloc[idx]
        with np.load(r.pose_path,allow_pickle=False) as z:
            x=z['features'].astype(np.float32)
        idx=np.asarray([0,1,3,4,5,6,7],dtype=np.int64)
        x[:,:,idx]=(x[:,:,idx]-self.means[idx])/self.stds[idx]
        x*=self.mask.reshape(1,1,-1)
        if self.augment:
            # Cheap skeleton-only augmentation: small coordinate noise and temporal jitter; labels stay unchanged.
            if random.random()<0.5: x[:,:,:2]+=np.random.normal(0,0.01,size=x[:,:,:2].shape).astype(np.float32)
            if random.random()<0.25: x=x[::-1].copy()
        return torch.from_numpy(x), torch.tensor(int(r.label),dtype=torch.long)

A=np.zeros((17,17),dtype=np.float32)
for i,j in COCO_EDGES: A[i,j]=A[j,i]=1
np.fill_diagonal(A,1)
d=A.sum(axis=1,keepdims=True)
A_norm=(A/np.sqrt(np.maximum(d,1e-6)*np.maximum(d.T,1e-6))).astype(np.float32)
A_tensor=torch.tensor(A_norm,dtype=torch.float32)

class GraphConv(nn.Module):
    def __init__(self,c_in,c_out,A):
        super().__init__(); self.register_buffer('A',A); self.proj=nn.Conv2d(c_in,c_out,1,bias=False); self.bn=nn.BatchNorm2d(c_out)
    def forward(self,x):
        # x: B,C,T,V
        x=torch.einsum('nctv,vw->nctw',x,self.A)
        return F.relu(self.bn(self.proj(x)), inplace=True)

class STGCNBranch(nn.Module):
    def __init__(self,in_ch,hidden=48):
        super().__init__()
        self.g1=GraphConv(in_ch,hidden,A_tensor)
        self.t1=nn.Conv2d(hidden,hidden,kernel_size=(5,1),padding=(2,0))
        self.g2=GraphConv(hidden,hidden,A_tensor)
        self.t2=nn.Conv2d(hidden,hidden,kernel_size=(3,1),padding=(1,0),dilation=(1,1))
        self.bn=nn.BatchNorm2d(hidden)
    def forward(self,x):
        x=x.permute(0,3,1,2).contiguous() # B,C,T,V
        x=self.g1(x); x=F.relu(self.bn(self.t1(x)),inplace=True)
        x=self.g2(x); x=F.relu(self.bn(self.t2(x)),inplace=True)
        conf=x.new_zeros((x.size(0),x.size(2),x.size(3)))
        # Confidence is passed into the branch separately through a lightweight side channel.
        # The caller supplies masked confidence in input channel 2; use it for confidence-weighted node pooling.
        return x.mean(dim=-1).transpose(1,2).contiguous() # B,T,H

class TCNBranch(nn.Module):
    def __init__(self,in_ch,hidden=48):
        super().__init__(); self.in_proj=nn.Linear(17*in_ch,hidden)
        self.net=nn.Sequential(
            nn.Conv1d(hidden,hidden,3,padding=1),nn.GELU(),nn.Dropout(0.2),
            nn.Conv1d(hidden,hidden,3,padding=2,dilation=2),nn.GELU(),nn.Dropout(0.2)
        )
    def forward(self,x):
        B,T,V,C=x.shape
        h=self.in_proj(x.reshape(B,T,V*C)).transpose(1,2)
        h=self.net(h).transpose(1,2)
        return h

class TemporalAttention(nn.Module):
    def __init__(self,dim=48,heads=2):
        super().__init__(); self.attn=nn.MultiheadAttention(dim,heads,batch_first=True,dropout=0.1); self.ln=nn.LayerNorm(dim)
    def forward(self,x):
        y,_=self.attn(x,x,x,need_weights=False)
        return self.ln(x+y)

class ProposedFallNet(nn.Module):
    def __init__(self,in_ch=8,hidden=48,use_stgcn=True,use_attention=True):
        super().__init__(); self.use_stgcn=use_stgcn; self.use_attention=use_attention
        self.spatial=STGCNBranch(in_ch,hidden) if use_stgcn else None
        self.temporal=TCNBranch(in_ch,hidden)
        self.gate=nn.Sequential(nn.Linear(hidden*2+2,32),nn.GELU(),nn.Linear(32,hidden),nn.Sigmoid())
        self.attn=TemporalAttention(hidden,heads=2) if use_attention else None
        self.head=nn.Sequential(nn.LayerNorm(hidden),nn.Dropout(0.2),nn.Linear(hidden,1))
    def forward(self,x):
        c=x[:,:,:,2]
        conf_stats=torch.stack([c.mean(dim=(2)), c.std(dim=(2))+1e-5],dim=-1) # B,T,2
        t=self.temporal(x)
        if self.use_stgcn: s=self.spatial(x)
        else: s=torch.zeros_like(t)
        g=self.gate(torch.cat([s,t,conf_stats],dim=-1))
        fused=g*s+(1-g)*t
        if self.attn is not None: fused=self.attn(fused)
        pooled=fused.mean(dim=1)
        return self.head(pooled).squeeze(-1)

class TinyMLP(nn.Module):
    def __init__(self,in_ch=8):
        super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(32*17*in_ch,64),nn.GELU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x): return self.net(x).squeeze(-1)

class TinyGRU(nn.Module):
    def __init__(self,in_ch=8,hidden=48):
        super().__init__(); self.proj=nn.Linear(17*in_ch,48); self.gru=nn.GRU(48,hidden,batch_first=True); self.head=nn.Linear(hidden,1)
    def forward(self,x):
        B,T,V,C=x.shape; h=self.proj(x.reshape(B,T,V*C)); y,_=self.gru(h); return self.head(y[:,-1]).squeeze(-1)

# Shape sanity check before any training.
with torch.no_grad():
    dummy=torch.zeros(2,32,17,8,device=torch_device)
    print('Proposed output:', ProposedFallNet().to(torch_device)(dummy).shape)
    print('MLP output:', TinyMLP().to(torch_device)(dummy).shape)
    print('GRU output:', TinyGRU().to(torch_device)(dummy).shape)

Proposed output: torch.Size([2])
MLP output: torch.Size([2])
GRU output: torch.Size([2])


## 10. Training utilities, checkpointing and metrics

The optimized training path matches the fitting objective to the primary goal: valid classification Accuracy. Because the training set is only moderately imbalanced, the default path avoids using class-weighted loss and weighted sampling at the same time, since that can intentionally shift the operating point toward recall. Validation-only threshold calibration is then used to choose an operating threshold for Accuracy; the held-out test sets stay untouched until the final evaluation.

In [ ]:
RESULT_DIR=DIRS['results']; MODEL_DIR=DIRS['models']

CONFIG={
    'T':32,'batch_size':64,'epochs':18,'patience':4,'lr':3e-4,'weight_decay':1e-4,
    'dropout':0.2,'grad_clip':1.0,'num_workers':2,
    # Accuracy-oriented defaults: the observed training split is only moderately
    # imbalanced, so do not combine weighted sampling with positive loss weighting.
    'use_weighted_sampler':False,
    'use_pos_weight':False,
    # Coarse validation-only threshold grid; the test sets are never consulted.
    'threshold_grid':tuple(np.round(np.arange(0.20,0.801,0.025),3)),
    'checkpoint_metric':'val_accuracy',
}

# Free-tier execution policy: keep the full proposed model as the essential run;
# baselines/ablations remain optional and are skipped automatically when the
# runtime target is exhausted.
RUN_OPTIONAL_ABLATIONS = True


def _worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)

def _loader_generator():
    g=torch.Generator(); g.manual_seed(SEED); return g

def make_loader(df, channel_mask=None, augment=False, shuffle=False):
    ds=PoseSequenceDataset(df,means,stds,channel_mask=channel_mask,augment=augment)
    common=dict(batch_size=CONFIG['batch_size'],
                num_workers=CONFIG['num_workers'],
                pin_memory=torch.cuda.is_available(),
                persistent_workers=CONFIG['num_workers']>0,
                worker_init_fn=_worker_init_fn,
                generator=_loader_generator())
    if shuffle and CONFIG.get('use_weighted_sampler',False):
        counts=df.label.value_counts().to_dict()
        weights=df.label.map({k:1.0/max(v,1) for k,v in counts.items()}).values
        sampler=WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),len(weights),replacement=True)
        return DataLoader(ds,sampler=sampler,**common)
    return DataLoader(ds,shuffle=shuffle,**common)


def select_accuracy_threshold(y_true, y_prob, grid=None):
    """Choose a coarse validation-only operating threshold for maximum Accuracy."""
    y_true=np.asarray(y_true).astype(int)
    y_prob=np.asarray(y_prob).astype(float)
    grid=np.asarray(CONFIG.get('threshold_grid',grid if grid is not None else np.arange(0.20,0.801,0.025)),dtype=float)
    best=None
    for threshold in grid:
        pred=(y_prob>=float(threshold)).astype(int)
        acc=float(accuracy_score(y_true,pred))
        f1=float(f1_score(y_true,pred,zero_division=0))
        cm=confusion_matrix(y_true,pred,labels=[0,1])
        fpr=float(cm[0,1]/max(cm[0].sum(),1))
        key=(acc,f1,-fpr,-abs(float(threshold)-0.5))
        if best is None or key>best[0]:
            best=(key,float(threshold),acc,f1,fpr)
    return {'threshold':best[1],'accuracy':best[2],'f1':best[3],'false_positive_rate':best[4]}

def _amp_enabled():
    return torch.cuda.is_available()

def _autocast_context():
    return torch.autocast(device_type='cuda',dtype=torch.float16,enabled=_amp_enabled())

def _make_grad_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=_amp_enabled())
    except (AttributeError,TypeError):
        return torch.cuda.amp.GradScaler(enabled=_amp_enabled())

def _torch_load(path):
    try:
        return torch.load(path,map_location=torch_device,weights_only=False)
    except TypeError:
        # Compatibility fallback for older PyTorch releases without weights_only.
        return torch.load(path,map_location=torch_device)

def _data_signature(df):
    cols=['dataset','video_path','subject','camera','scenario','label','start_s','end_s']
    use=[c for c in cols if c in df.columns]
    payload=df[use].fillna('').astype(str).sort_values(use).to_csv(index=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def _experiment_signature(model_name, train_df, val_df, channel_mask):
    payload={'schema':3,'model_name':model_name,'seed':SEED,'cache_version':CACHE_VERSION,
             'config':CONFIG,
             'channel_mask':None if channel_mask is None else [float(x) for x in np.asarray(channel_mask).ravel()],
             'train_signature':_data_signature(train_df),'val_signature':_data_signature(val_df)}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode('utf-8')).hexdigest()

def metric_dict(y_true,y_prob,threshold=0.5):
    y_true=np.asarray(y_true).astype(int); y_prob=np.asarray(y_prob).astype(float); y_pred=(y_prob>=float(threshold)).astype(int)
    cm=confusion_matrix(y_true,y_pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    d={
        'accuracy':accuracy_score(y_true,y_pred),
        'precision':precision_score(y_true,y_pred,zero_division=0),
        'recall_sensitivity':recall_score(y_true,y_pred,zero_division=0),
        'f1':f1_score(y_true,y_pred,zero_division=0),
        'specificity':tn/max(tn+fp,1),
        'false_positive_rate':fp/max(fp+tn,1),
        'false_negative_rate':fn/max(fn+tp,1),
        'roc_auc':roc_auc_score(y_true,y_prob) if len(np.unique(y_true))==2 else np.nan,
        'pr_auc':average_precision_score(y_true,y_prob) if len(np.unique(y_true))==2 else np.nan,
        'tn':tn,'fp':fp,'fn':fn,'tp':tp,'n':len(y_true)
    }
    return d

def model_params(model): return sum(p.numel() for p in model.parameters())

def eval_loader(model,loader,threshold=0.5):
    model.eval(); ys=[]; ps=[]
    t0=time.time()
    with torch.no_grad():
        for x,y in loader:
            x=x.to(torch_device,non_blocking=True); y=y.to(torch_device,non_blocking=True)
            with _autocast_context():
                logits=model(x); prob=torch.sigmoid(logits)
            ys.append(y.cpu().numpy()); ps.append(prob.float().cpu().numpy())
    elapsed=time.time()-t0
    y=np.concatenate(ys); p=np.concatenate(ps)
    d=metric_dict(y,p,threshold=threshold); d['eval_threshold']=float(threshold); d['eval_seconds']=elapsed; d['examples_per_second']=len(y)/max(elapsed,1e-6)
    return d,y,p

def train_model(model_name, model, train_df, val_df, channel_mask=None, epochs=None):
    epochs=epochs or CONFIG['epochs']
    train_loader=make_loader(train_df,channel_mask,augment=True,shuffle=True)
    val_loader=make_loader(val_df,channel_mask,augment=False,shuffle=False)
    counts=train_df.label.value_counts().to_dict(); pos=counts.get(1,1); neg=counts.get(0,1)
    pos_weight=torch.tensor([neg/max(pos,1)],dtype=torch.float32,device=torch_device)
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight) if CONFIG.get('use_pos_weight',False) else nn.BCEWithLogitsLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['lr'],weight_decay=CONFIG['weight_decay'])
    scaler=_make_grad_scaler()
    ckpt=MODEL_DIR/f'{model_name}.pt'
    signature=_experiment_signature(model_name,train_df,val_df,channel_mask)
    history=[]; best=-float('inf'); best_f1=-float('inf'); bad=0; start_epoch=0
    if ckpt.exists():
        try:
            blob=_torch_load(ckpt)
            if blob.get('experiment_signature')==signature and blob.get('schema_version')==3:
                model.load_state_dict(blob['model'])
                try:
                    opt.load_state_dict(blob['optimizer'])
                except Exception as e:
                    print(f'[WARN] {model_name}: optimizer state could not be resumed ({e!r}); discarding incompatible checkpoint and restarting from epoch 0.')
                    ckpt.unlink(missing_ok=True)
                    start_epoch=0; best=-float('inf'); best_f1=-float('inf'); history=[]; bad=0
                else:
                    start_epoch=int(blob.get('epoch',0)); best=float(blob.get('best_val_accuracy',blob.get('best_val_loss',best))); best_f1=float(blob.get('best_val_f1',best_f1))
                    history=blob.get('history',[]); bad=int(blob.get('bad_epochs',0))
                print(f'[RESUME] {model_name} from epoch {start_epoch} (matching experiment signature)')
            else:
                print(f'[STALE CHECKPOINT] {model_name}: signature/config/data mismatch; starting this experiment fresh.')
        except Exception as e:
            print(f'[WARN] {model_name}: checkpoint unreadable; starting fresh. Error: {e!r}')
    model.to(torch_device)
    essential_run = model_name in {'E_full_proposed'}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(start_epoch,epochs):
        if not BUDGET.guard(f'train_{model_name}_epoch_{epoch+1}',essential=essential_run):
            break
        model.train(); losses=[]; t0=time.time()
        for x,y in train_loader:
            x=x.to(torch_device,non_blocking=True); y=y.float().to(torch_device,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with _autocast_context():
                logits=model(x); loss=loss_fn(logits,y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),CONFIG['grad_clip'])
            scaler.step(opt); scaler.update()
            losses.append(float(loss.detach().cpu()))
        if not losses:
            print(f'[BUDGET] {model_name}: no training batches completed; stopping.')
            break
        # Use validation-only threshold calibration to evaluate the current
        # operating point; test data are never touched during training/model selection.
        val_d_raw,y_val_epoch,p_val_epoch=eval_loader(model,val_loader,threshold=0.5)
        cal=select_accuracy_threshold(y_val_epoch,p_val_epoch)
        rec={'model':model_name,'epoch':epoch+1,'train_loss':float(np.mean(losses)),
             'val_f1':val_d_raw['f1'],'val_loss_proxy':1-val_d_raw['f1'],'epoch_seconds':time.time()-t0,
             'val_calibrated_accuracy':cal['accuracy'],'val_best_threshold':cal['threshold'],
             'val_calibrated_f1':cal['f1'],'val_calibrated_false_positive_rate':cal['false_positive_rate'],
             **{f'val_{k}':v for k,v in val_d_raw.items() if k not in ['tn','fp','fn','tp','eval_threshold']}}
        history.append(rec)
        improved=(rec['val_calibrated_accuracy']>best+1e-4 or
                   (abs(rec['val_calibrated_accuracy']-best)<=1e-4 and rec['val_calibrated_f1']>best_f1+1e-4))
        if improved:
            best=rec['val_calibrated_accuracy']; best_f1=rec['val_calibrated_f1']; bad=0
            torch.save({
                'schema_version':3,
                'experiment_signature':signature,
                'model':model.state_dict(),
                'optimizer':opt.state_dict(),
                'epoch':epoch+1,
                'best_val_accuracy':best,
                'best_val_f1':best_f1,
                'best_val_threshold':cal['threshold'],
                'history':history,
                'bad_epochs':bad,
                'config':CONFIG,
                'model_name':model_name,
            },ckpt)
        else:
            bad+=1
        assert_vram_budget(f'training {model_name} epoch {epoch+1}')
        print(f"{model_name} epoch {epoch+1:02d}: loss={rec['train_loss']:.4f} valAcc={rec['val_calibrated_accuracy']:.3f} thr={rec['val_best_threshold']:.3f} ({rec['epoch_seconds']:.1f}s)")
        BUDGET.checkpoint(f'train_{model_name}_epoch_{epoch+1}')
        if bad>=CONFIG['patience']:
            print(f'[EARLY STOP] {model_name}')
            break
    if not ckpt.exists():
        raise FileNotFoundError(f'No checkpoint exists for {model_name} after training attempt: {ckpt}')
    blob=_torch_load(ckpt)
    if blob.get('experiment_signature')!=signature:
        raise RuntimeError(f'Checkpoint signature mismatch for {model_name}; refusing to evaluate stale weights.')
    model.load_state_dict(blob['model'])
    val_d,y,p=eval_loader(model,val_loader,threshold=0.5)
    return model,val_d,y,p,history


## 11. Validation policy (no separate learning-rate sweep)

The original notebook section was titled as a learning-rate screen, but the executed implementation used the fixed `3e-4` learning rate and did not run a separate sweep. To avoid implying an experiment that was not performed, this section documents the fixed, budgeted learning-rate policy instead.

In [31]:
def stratified_subset(
    df,
    fraction=0.35,
    seed=SEED,
):
    parts = []

    for label, g in df.groupby(
        "label"
    ):

        n = max(
            2,
            int(
                round(
                    len(g)
                    * fraction
                )
            ),
        )

        parts.append(
            g.sample(
                min(
                    n,
                    len(g)
                ),
                random_state=seed + int(label),
            )
        )

    return (
        pd.concat(parts)
        .sample(
            frac=1,
            random_state=seed
        )
        .reset_index(drop=True)
    )


# POSE_MANIFEST currently contains train + validation only.
train_df = POSE_MANIFEST[
    POSE_MANIFEST.split == "train"
].reset_index(drop=True)

val_df = POSE_MANIFEST[
    POSE_MANIFEST.split == "val"
].reset_index(drop=True)

if train_df.empty:
    raise RuntimeError(
        "Training split is empty."
    )

if train_df["label"].nunique() < 2:
    raise RuntimeError(
        "Training split contains fewer than two classes; "
        "refusing to train a binary classifier."
    )

if (
    len(val_df)
    and val_df["label"].nunique() < 2
):
    print(
        "[WARN] Validation split contains only one class; "
        "ROC-AUC/PR-AUC will be reported as NaN."
    )

# ------------------------------------------------------------
# Note: free-tier change
# Do not spend GPU time on a learning-rate screen before the main
# scientific model is trained. The notebook keeps the documented
# default LR as the main run.
# ------------------------------------------------------------

CONFIG["lr"] = 3e-4

print(
    "Training rows:",
    len(train_df)
)

print(
    "Validation rows:",
    len(val_df)
)

print(
    "Learning rate:",
    CONFIG["lr"]
)


Training rows: 410
Validation rows: 293
Learning rate: 0.0003


## 12. Main training suite

The main run contains the two small baselines plus the proposed model. The ablation table is compact and budget-aware. A-D vary the input information; structural ablations remove ST-GCN or temporal attention while keeping the rest lightweight.

In [ ]:
RUNS = []


def run_and_record(
    model_name,
    model,
    train_df=train_df,
    val_df=val_df,
    test_df=None,
    mask=None,
    essential=False,
):
    if not BUDGET.guard(
        f"{model_name}_start",
        essential=essential
    ):
        return None

    t0 = time.time()

    model, val_d, vy, vp, hist = train_model(
        model_name,
        model,
        train_df,
        val_df,
        channel_mask=mask
    )

    calibrated = select_accuracy_threshold(vy,vp)

    entry = {
        "model": model_name,
        "parameters": model_params(
            model
        ),
        "train_seconds":
            time.time() - t0,
    }

    entry.update({
        f"val_{k}": v
        for k, v in val_d.items()
    })
    entry.update({
        "val_calibrated_accuracy":calibrated["accuracy"],
        "val_best_threshold":calibrated["threshold"],
        "val_calibrated_f1":calibrated["f1"],
        "val_calibrated_false_positive_rate":calibrated["false_positive_rate"],
    })

    # Test evaluation is deferred until all models have been trained and
    # the final model has been selected. This prevents repeated extraction
    # and evaluation on held-out data during the model-selection stage.

    RUNS.append(
        entry
    )

    pd.DataFrame(
        RUNS
    ).to_csv(
        RESULT_DIR
        / "validation_results.csv",
        index=False
    )

    return (
        model,
        entry
    )


# ============================================================
# ESSENTIAL MODEL: train this FIRST.
# ============================================================

# Earlier ablation results identified position + velocity + acceleration
# as the strongest feature set by validation Accuracy in the original run.
FULL_MASK = np.asarray([1,1,0,1,1,1,1,0],dtype=np.float32)

full_result = run_and_record(
    "E_full_proposed",
    ProposedFallNet().to(torch_device),
    mask=FULL_MASK,
    essential=True,
)

if full_result is None:
    raise RuntimeError(
        "The essential full proposed model did not complete."
    )

del full_result
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# Optional baselines / ablations
# ============================================================
#
# These are useful scientifically, but they cannot consume the
# hard runtime reserve needed for the core result.
# ============================================================

if RUN_OPTIONAL_ABLATIONS:

    # Small baselines.
    for name, ctor in [
        ("tiny_mlp", TinyMLP),
        ("tiny_gru", TinyGRU),
    ]:

        if not BUDGET.guard(
            f"optional_{name}",
            essential=False
        ):
            break

        result = run_and_record(
            name,
            ctor().to(torch_device),
            mask=np.ones(
                8,
                dtype=np.float32
            ),
            essential=False,
        )

        if result is not None:
            del result

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Informational channel ablations.
    ABLATIONS = {
        "A_position_only":
            [1,1,0,0,0,0,0,0],

        "B_position_velocity":
            [1,1,0,1,1,0,0,0],

        "D_position_velocity_acceleration_confidence":
            [1,1,1,1,1,1,1,0],
    }

    ab_results = []

    for name, mask in ABLATIONS.items():

        if not BUDGET.guard(
            f"optional_ablation_{name}",
            essential=False
        ):
            break

        result = run_and_record(
            name,
            ProposedFallNet().to(torch_device),
            mask=np.asarray(
                mask,
                dtype=np.float32
            ),
            essential=False,
        )

        if result is not None:
            ab_results.append(
                result[1]
            )
            del result

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Structural ablations are lower priority under Free-tier runtime.
    for name, use_stgcn, use_attention in [
        (
            "without_STGCN",
            False,
            True,
        ),
        (
            "without_temporal_attention",
            True,
            False,
        ),
    ]:

        if not BUDGET.guard(
            f"optional_structural_{name}",
            essential=False
        ):
            break

        result = run_and_record(
            name,
            ProposedFallNet(
                use_stgcn=use_stgcn,
                use_attention=use_attention,
            ).to(torch_device),
            mask=np.ones(
                8,
                dtype=np.float32
            ),
            essential=False,
        )

        if result is not None:
            ab_results.append(
                result[1]
            )
            del result

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(
        ab_results
    ).to_csv(
        RESULT_DIR
        / "ablation_results.csv",
        index=False
    )

pd.DataFrame(
    RUNS
).to_csv(
    RESULT_DIR
    / "validation_results.csv",
    index=False
)


## 13. Select and freeze the final model

Model selection is made only on validation data. CAUCAFall and the held-out test split remain untouched until after the final model and operating threshold are frozen. The primary criterion is validation Accuracy at a validation-only calibrated threshold; F1 and false-positive rate are secondary tie-breakers.

In [ ]:
# ============================================================
# Freeze the final model using validation-only model selection
# ============================================================

run_df = pd.DataFrame(
    RUNS
)

if run_df.empty:

    raise RuntimeError(
        "No model completed within the runtime budget."
    )


# ------------------------------------------------------------
# Validate the columns required by the documented selection rule.
# ------------------------------------------------------------

required_cols = {
    "model",
    "val_calibrated_accuracy",
    "val_f1",
    "val_false_positive_rate",
    "parameters",
}

missing_cols = (
    required_cols
    - set(
        run_df.columns
    )
)

if missing_cols:

    raise RuntimeError(
        "Model-selection table is missing required columns: "
        f"{sorted(missing_cols)}"
    )


# ------------------------------------------------------------
# Select only from completed main-suite models using the documented
# validation-only rule:
#
#   1. Find highest validation calibrated Accuracy.
#   2. Prefer higher validation F1.
#   3. Prefer lower validation false-positive rate.
#   4. Prefer fewer parameters as final deterministic tie-break.
#
# No held-out test data are used here.
# ------------------------------------------------------------

max_acc = float(run_df["val_calibrated_accuracy"].max())
cand = run_df[run_df["val_calibrated_accuracy"] >= max_acc - 0.002].copy()

cand = cand.sort_values(
    [
        "val_f1",
        "val_false_positive_rate",
        "parameters",
    ],
    ascending=[
        False,
        True,
        True,
    ],
    kind="mergesort",
)

if cand.empty:

    raise RuntimeError(
        "Model selection produced no candidates."
    )


chosen_name = str(
    cand.iloc[0]["model"]
)


print("Frozen final model:", chosen_name, "| candidates within 0.002 calibrated Accuracy:", cand["model"].tolist())


# ============================================================
# Recreate the selected model family exactly.
# ============================================================

if chosen_name == "tiny_mlp":

    final_model = TinyMLP()

elif chosen_name == "tiny_gru":

    final_model = TinyGRU()

elif chosen_name == "without_STGCN":

    final_model = ProposedFallNet(
        use_stgcn=False,
        use_attention=True,
    )

elif chosen_name == "without_temporal_attention":

    final_model = ProposedFallNet(
        use_stgcn=True,
        use_attention=False,
    )

else:

    # Full proposed model.
    final_model = ProposedFallNet(
        use_stgcn=True,
        use_attention=True,
    )


# ------------------------------------------------------------
# Final feature/channel mask.
# ------------------------------------------------------------

_mask_lookup = dict(ABLATIONS) if "ABLATIONS" in globals() else {}
_mask_lookup["E_full_proposed"] = FULL_MASK
final_mask = np.asarray(
    _mask_lookup.get(chosen_name,[1] * 8),
    dtype=np.float32,
)


# ============================================================
# Load the frozen checkpoint.
# ============================================================

final_ckpt = (
    MODEL_DIR
    / f"{chosen_name}.pt"
)


if not final_ckpt.exists():

    raise FileNotFoundError(
        "Expected checkpoint missing: "
        f"{final_ckpt}"
    )


# ------------------------------------------------------------
# IMPORTANT PyTorch compatibility fix.
#
# The training code saved a dictionary checkpoint containing:
#   - schema_version
#   - experiment_signature
#   - model = model.state_dict()
#   - optimizer = opt.state_dict()
#   - epoch
#
# PyTorch 2.6+ changed torch.load() so that weights_only=True is
# the default. That restricted unpickler rejects some metadata in
# this legacy/full checkpoint format.
#
# This checkpoint was produced locally by this trusted notebook,
# so explicitly use weights_only=False for loading the checkpoint
# dictionary. We still load ONLY the `model` state_dict into the
# freshly recreated architecture below.
# ------------------------------------------------------------

try:

    blob = torch.load(
        final_ckpt,
        map_location=torch_device,
        weights_only=False,
    )

except TypeError:
    # Compatibility with older PyTorch versions that do not expose
    # the `weights_only` keyword.
    blob = torch.load(
        final_ckpt,
        map_location=torch_device,
    )


# ------------------------------------------------------------
# Validate checkpoint structure before loading.
# ------------------------------------------------------------

if not isinstance(
    blob,
    dict,
):

    raise RuntimeError(
        "Final checkpoint has an unexpected format. "
        f"Expected dict, got {type(blob).__name__}: "
        f"{final_ckpt}"
    )


if "model" not in blob:

    raise RuntimeError(
        "Final checkpoint does not contain the required "
        "'model' state_dict: "
        f"{final_ckpt}. "
        f"Available keys: {sorted(blob.keys())}"
    )


model_state = blob[
    "model"
]


if not isinstance(
    model_state,
    dict,
):

    raise RuntimeError(
        "Checkpoint key 'model' is not a state_dict dictionary. "
        f"Got: {type(model_state).__name__}"
    )


# ------------------------------------------------------------
# Load the trained parameters into the freshly recreated model.
# ------------------------------------------------------------

try:

    incompatible = final_model.load_state_dict(
        model_state,
        strict=True,
    )

except RuntimeError as exc:

    raise RuntimeError(
        "The selected checkpoint is incompatible with the "
        "recreated model architecture. "
        f"Model={chosen_name} | "
        f"Checkpoint={final_ckpt} | "
        f"Original error: {exc}"
    ) from exc


# ------------------------------------------------------------
# `strict=True` should already guarantee an exact match.
# Keep an explicit sanity check for clarity.
# ------------------------------------------------------------

if (
    getattr(
        incompatible,
        "missing_keys",
        None,
    )
    or getattr(
        incompatible,
        "unexpected_keys",
        None,
    )
):

    raise RuntimeError(
        "Final checkpoint/model state_dict mismatch detected. "
        f"Missing keys={getattr(incompatible, 'missing_keys', [])} | "
        f"Unexpected keys={getattr(incompatible, 'unexpected_keys', [])}"
    )


# ------------------------------------------------------------
# Freeze model for final evaluation.
# ------------------------------------------------------------

final_model.to(
    torch_device
)

final_model.eval()


# ------------------------------------------------------------
# Final checkpoint sanity information.
# ------------------------------------------------------------

checkpoint_schema = blob.get(
    "schema_version",
    "unknown",
)

checkpoint_epoch = blob.get(
    "epoch",
    "unknown",
)

print(
    "Final checkpoint:",
    final_ckpt,
)

print(
    "Checkpoint schema:",
    checkpoint_schema,
)

print(
    "Checkpoint epoch:",
    checkpoint_epoch,
)

print("Parameters:", model_params(final_model))

# Freeze the final operating threshold using validation data only.
_final_val_loader=make_loader(val_df,final_mask,augment=False,shuffle=False)
_, _final_val_y, _final_val_p=eval_loader(final_model,_final_val_loader,threshold=0.5)
_final_cal=select_accuracy_threshold(_final_val_y,_final_val_p)
final_threshold=float(_final_cal["threshold"])
print("Validation-calibrated threshold:", final_threshold)
print("Validation calibrated Accuracy:", _final_cal["accuracy"])

## 14. MCFD cross-view evaluation

This is a separate scientific test: training cameras 1–4, validation camera 5, held-out test cameras 6–8. Because the same scenario/event may exist across cameras, the correct interpretation is **view shift**, not a subject- or event-independent generalization claim.

In [ ]:
# ============================================================
# Held-out test evaluation (final validation-frozen threshold)
# ============================================================
#
# Test pose features are generated only now, after model selection.
# No training/tuning uses these held-out examples.
#
# Note:
# The current MCFD manifest contains 1,014 held-out test segments, but
# the physical extracted archive can contain repacked / duplicated clip
# files whose duration does not cover the OmniFall absolute annotation
# end time.
#
# Therefore this cell:
#
#   1. Runs the canonical ensure_pose_manifest() first.
#   2. Keeps all successfully cached rows unchanged.
#   3. Detects only the remaining failed rows.
#   4. Retries failed MCFD rows using the physical clip itself when the
#      selected physical file is shorter than the absolute annotation
#      interval. This is treated as a pre-segmented physical clip.
#   5. Retries inference with smaller batches if necessary.
#   6. Never changes labels, split assignments, or the final model.
#
# GMDCSA24 keeps the original annotation-time extraction semantics.
# MCFD receives only a targeted fallback for the rows that failed.
# ============================================================


cross_rows = []


# ============================================================
# 1. Build held-out test manifest
# ============================================================

heldout_test_df = FULL_MANIFEST[
    FULL_MANIFEST["split"] == "test"
].copy()


if heldout_test_df.empty:

    raise RuntimeError(
        "Held-out test manifest is empty."
    )


TEST_POSE_PATH = (
    DIRS["metadata"]
    / "test_pose_manifest.csv"
)


# ============================================================
# 2. Canonical extraction pass
#
# This preserves the original notebook behavior first.
# Existing valid pose caches are reused by extract_one_pose().
# ============================================================

heldout_pose = ensure_pose_manifest(
    heldout_test_df,
    TEST_POSE_PATH,
    desc="Held-out test pose cache",
    essential=True,
    checkpoint_every=20,
)


if heldout_pose.empty:

    raise RuntimeError(
        "Held-out test pose extraction produced no data."
    )


# ============================================================
# 3. Build a deterministic row identity so successful rows are
#    retained and only failed rows are retried.
# ============================================================

_POSE_ID_COLUMNS = [
    "dataset",
    "video_path",
    "start_s",
    "end_s",
    "label",
    "activity",
    "subject",
    "camera",
    "scenario",
]


def _pose_row_key(row):

    values = []

    for column in _POSE_ID_COLUMNS:

        value = getattr(
            row,
            column,
            None,
        )

        if pd.isna(value):

            values.append(
                "<NA>"
            )

        elif isinstance(
            value,
            (float, np.floating),
        ):

            values.append(
                f"{float(value):.6f}"
            )

        else:

            values.append(
                str(value)
            )

    return tuple(
        values
    )


successful_keys = {
    _pose_row_key(row)
    for row in heldout_pose.itertuples(
        index=False
    )
}


remaining_test_df = heldout_test_df[
    [
        _pose_row_key(row)
        not in successful_keys
        for row in heldout_test_df.itertuples(
            index=False
        )
    ]
].copy()


print(
    "Initial successful held-out pose rows:",
    len(heldout_pose),
)

print(
    "Rows still requiring retry:",
    len(remaining_test_df),
)


if remaining_test_df.empty:

    print(
        "[OK] All held-out test pose features were already available."
    )

else:

    # ========================================================
    # 4. MCFD-specific physical-video fallback
    # ========================================================
    #
    # The official OmniFall MCFD protocol is segment-level:
    # each annotation row has start/end times while the underlying
    # camera stream is a physical AVI.
    #
    # In the current extracted archive, some physical paths can be
    # shorter than the annotation's absolute end time. When this
    # happens, retry using [0, full_clip] because that physical file
    # behaves like an already-segmented clip.
    # ========================================================

    mcfd_retry_df = remaining_test_df[
        remaining_test_df["dataset"].astype(str)
        == "MCFD"
    ].copy()

    non_mcfd_retry_df = remaining_test_df[
        remaining_test_df["dataset"].astype(str)
        != "MCFD"
    ].copy()


    # --------------------------------------------------------
    # Discover all physical MCFD video files.
    # --------------------------------------------------------

    mcfd_root = (
        EXTRACTED.get("MCFD")
        if "EXTRACTED" in globals()
        else None
    )


    if (
        mcfd_root is None
        or not Path(mcfd_root).exists()
    ):

        raise RuntimeError(
            "MCFD extracted root is unavailable; cannot perform "
            "held-out MCFD retry."
        )


    mcfd_physical_videos = sorted(
        find_videos(
            mcfd_root
        ),
        key=lambda p: str(p).lower(),
    )


    if not mcfd_physical_videos:

        raise RuntimeError(
            f"No physical MCFD videos found under {mcfd_root}."
        )


    # --------------------------------------------------------
    # Canonical physical MCFD identity.
    #
    # The manifest builder uses:
    #     chute<N>/cam<M>
    #
    # We reconstruct the same identity locally so that this cell
    # does not depend on a private nested function from another cell.
    # --------------------------------------------------------

    def _heldout_mcfd_key(value):

        raw = str(
            value
        ).replace(
            "\\",
            "/",
        )

        parts = [
            part.strip()
            for part in raw.split("/")
            if part.strip()
        ]


        chute_id = None

        for part in parts:

            stem = Path(
                part
            ).stem

            match = re.search(
                r"(?:^|[^a-z0-9])chute[-_ ]?0*(\d+)(?:[^0-9]|$)",
                stem.lower(),
            )

            if match:

                chute_id = int(
                    match.group(1)
                )

                break


        camera_id = None

        # Search filename first.
        terminal = (
            Path(
                parts[-1]
            ).stem
            if parts
            else ""
        )

        match = re.search(
            r"(?:^|[^a-z0-9])cam(?:era)?[-_ ]?0*(\d+)",
            terminal.lower(),
        )

        if match:

            camera_id = int(
                match.group(1)
            )


        # Fall back to directory names.
        if camera_id is None:

            for part in reversed(
                parts[:-1]
            ):

                match = re.search(
                    r"(?:^|[^a-z0-9])cam(?:era)?[-_ ]?0*(\d+)(?:[^0-9]|$)",
                    part.lower(),
                )

                if match:

                    camera_id = int(
                        match.group(1)
                    )

                    break


        if (
            chute_id is None
            or camera_id is None
        ):

            return ""

        return (
            f"chute{chute_id}/cam{camera_id}"
        )


    # --------------------------------------------------------
    # Group physical MCFD files by canonical camera identity.
    # --------------------------------------------------------

    mcfd_groups = {}


    for video_path in mcfd_physical_videos:

        key = _heldout_mcfd_key(
            video_path
        )

        if not key:
            continue

        mcfd_groups.setdefault(
            key,
            [],
        ).append(
            video_path
        )


    if not mcfd_groups:

        raise RuntimeError(
            "Could not construct any canonical MCFD physical-video "
            "keys during held-out retry."
        )


    # --------------------------------------------------------
    # Cache physical durations.
    #
    # Do this once per physical file rather than
    # once per failed annotation.
    # --------------------------------------------------------

    mcfd_duration_cache = {}


    def _video_duration_seconds(
        video_path
    ):

        video_path = Path(
            video_path
        )

        cached = mcfd_duration_cache.get(
            str(video_path)
        )

        if cached is not None:
            return cached


        cap = cv2.VideoCapture(
            str(video_path)
        )

        if not cap.isOpened():

            mcfd_duration_cache[
                str(video_path)
            ] = None

            return None


        try:

            fps = float(
                cap.get(
                    cv2.CAP_PROP_FPS
                )
                or 0.0
            )

            n = int(
                cap.get(
                    cv2.CAP_PROP_FRAME_COUNT
                )
                or 0
            )

        finally:

            cap.release()


        if (
            fps <= 0
            or n <= 0
        ):

            duration = None

        else:

            duration = (
                float(n)
                / float(fps)
            )


        mcfd_duration_cache[
            str(video_path)
        ] = duration

        return duration


    # --------------------------------------------------------
    # Resolve the best physical MCFD candidate for one failed row.
    #
    # Ranking:
    #
    #   A. Candidate large enough to contain annotation end time:
    #      prefer the shortest sufficient video.
    #
    #   B. No candidate contains absolute end time:
    #      interpret the archive copy as a pre-segmented clip and
    #      prefer a duration closest to the annotation interval length.
    #
    # The function returns:
    #      (physical_path, use_full_clip, diagnostics)
    # --------------------------------------------------------

    def _resolve_mcfd_retry_video(
        row
    ):

        logical_key = _heldout_mcfd_key(
            row.video_path
        )


        candidates = list(
            mcfd_groups.get(
                logical_key,
                [],
            )
        )


        if not candidates:

            # Fall back to the exact representative video.
            exact = Path(
                row.video_path
            )

            if exact.exists():

                candidates = [
                    exact
                ]

            else:

                return (
                    None,
                    False,
                    {
                        "reason":
                            "no_physical_candidates",
                        "logical_key":
                            logical_key,
                    },
                )


        try:

            start_s = float(
                row.start_s
            )

        except Exception:

            start_s = 0.0


        if not np.isfinite(
            start_s
        ):

            start_s = 0.0


        try:

            end_s = float(
                row.end_s
            )

        except Exception:

            end_s = None


        if (
            end_s is not None
            and not np.isfinite(
                end_s
            )
        ):

            end_s = None


        interval_length = (
            max(
                end_s - start_s,
                0.0,
            )
            if end_s is not None
            else None
        )


        candidate_info = []


        for candidate in sorted(
            candidates,
            key=lambda p: str(p).lower(),
        ):

            duration = (
                _video_duration_seconds(
                    candidate
                )
            )

            if duration is None:
                continue


            candidate_info.append({
                "path":
                    candidate,

                "duration":
                    duration,
            })


        if not candidate_info:

            return (
                None,
                False,
                {
                    "reason":
                        "all_candidates_unreadable",
                    "logical_key":
                        logical_key,
                },
            )


        # ====================================================
        # Case A:
        # At least one physical file contains the annotated
        # absolute end time.
        # ====================================================

        sufficient = []

        if end_s is not None:

            for info in candidate_info:

                if (
                    info["duration"]
                    >= end_s
                    - 0.25
                ):

                    sufficient.append(
                        info
                    )


        if sufficient:

            selected = min(
                sufficient,
                key=lambda info: (
                    info["duration"],
                    str(
                        info["path"]
                    ).lower(),
                ),
            )

            return (
                selected["path"],
                False,
                {
                    "reason":
                        "sufficient_full_stream",
                    "logical_key":
                        logical_key,
                    "duration":
                        selected["duration"],
                    "annotation_start":
                        start_s,
                    "annotation_end":
                        end_s,
                },
            )


        # ====================================================
        # Case B:
        # No physical candidate reaches the absolute annotation
        # end. This strongly indicates that the extracted physical
        # file is already a clip rather than the original full stream.
        #
        # Select the physical clip whose duration most closely matches
        # the requested segment length.
        # ====================================================

        if interval_length is None:
            interval_length = 0.0


        selected = min(
            candidate_info,
            key=lambda info: (
                abs(
                    info["duration"]
                    - interval_length
                ),
                str(
                    info["path"]
                ).lower(),
            ),
        )


        return (
            selected["path"],
            True,
            {
                "reason":
                    "pre_segmented_clip_fallback",
                "logical_key":
                    logical_key,
                "duration":
                    selected["duration"],
                "annotation_start":
                    start_s,
                "annotation_end":
                    end_s,
                "interval_length":
                    interval_length,
            },
        )


    # ========================================================
    # 5. Retry failed MCFD rows only.
    # ========================================================

    recovered_rows = []
    unrecovered_rows = []


    for row_idx, row in tqdm(
        mcfd_retry_df.iterrows(),
        total=len(
            mcfd_retry_df
        ),
        desc="Retry MCFD held-out pose cache",
    ):

        BUDGET.guard(
            "heldout_mcfd_pose_retry",
            essential=True,
        )


        (
            retry_video,
            use_full_clip,
            resolution_info,
        ) = _resolve_mcfd_retry_video(
            row
        )


        if retry_video is None:

            unrecovered_rows.append({
                **row.to_dict(),

                "error":
                    repr(
                        RuntimeError(
                            "No readable physical MCFD "
                            "candidate was found."
                        )
                    ),

                "retry_reason":
                    resolution_info.get(
                        "reason"
                    ),

                "resolution_info":
                    str(
                        resolution_info
                    ),
            })

            continue


        # ----------------------------------------------------
        # Build a temporary extraction row.
        #
        # The final held-out manifest keeps the ORIGINAL row
        # metadata. Only the temporary extraction interval/path
        # is changed for this retry.
        # ----------------------------------------------------

        extraction_row = row.copy()

        extraction_row["video_path"] = (
            str(
                retry_video
            )
        )


        if use_full_clip:

            extraction_row["start_s"] = (
                0.0
            )

            extraction_row["end_s"] = (
                np.nan
            )


        # ----------------------------------------------------
        # Targeted inference retries.
        # ----------------------------------------------------

        original_batch = (
            globals().get(
                "POSE_INFER_BATCH",
                32,
            )
        )

        retry_batch_values = []

        for batch_value in [
            original_batch,
            16,
            8,
            4,
        ]:

            try:

                batch_value = int(
                    batch_value
                )

            except Exception:

                continue


            batch_value = max(
                1,
                batch_value,
            )


            if batch_value not in retry_batch_values:

                retry_batch_values.append(
                    batch_value
                )


        success = False
        last_error = None


        for batch_value in retry_batch_values:

            BUDGET.guard(
                "heldout_mcfd_pose_retry",
                essential=True,
            )


            POSE_INFER_BATCH = (
                batch_value
            )


            if torch.cuda.is_available():

                try:
                    torch.cuda.empty_cache()
                    torch.cuda.synchronize()
                except Exception:
                    pass


            try:

                out_path, cache_hit = (
                    extract_one_pose(
                        extraction_row
                    )
                )


                # ------------------------------------------------
                # Verify cache integrity immediately.
                # ------------------------------------------------

                cache_ok = False


                try:

                    with np.load(
                        out_path,
                        allow_pickle=False,
                    ) as z:

                        cache_ok = (
                            "features" in z
                            and tuple(
                                z["features"].shape
                            )
                            == (
                                32,
                                17,
                                8,
                            )
                            and int(
                                z["label"]
                            )
                            == int(
                                row.label
                            )
                            and str(
                                z["cache_version"]
                            )
                            == str(
                                CACHE_VERSION
                            )
                        )

                except Exception:

                    cache_ok = False


                if not cache_ok:

                    raise RuntimeError(
                        "Pose cache was produced but failed "
                        "post-extraction integrity validation."
                    )


                recovered_rows.append({
                    **row.to_dict(),

                    "pose_path":
                        str(
                            out_path
                        ),

                    "pose_cache_hit":
                        bool(
                            cache_hit
                        ),

                    "pose_cache_version":
                        CACHE_VERSION,

                    "pose_retry":
                        True,

                    "pose_retry_batch":
                        batch_value,

                    "pose_retry_reason":
                        resolution_info.get(
                            "reason"
                        ),

                    "pose_retry_video":
                        str(
                            retry_video
                        ),

                    "pose_retry_full_clip":
                        bool(
                            use_full_clip
                        ),
                })


                success = True
                break


            except BudgetStop:

                raise


            except Exception as exc:

                last_error = exc


                try:

                    bad_path = pose_cache_path(
                        extraction_row
                    )

                    bad_path.unlink(
                        missing_ok=True
                    )

                    bad_tmp = (
                        bad_path.with_suffix(
                            ".tmp.npz"
                        )
                    )

                    bad_tmp.unlink(
                        missing_ok=True
                    )

                except Exception:
                    pass


                if torch.cuda.is_available():

                    try:
                        torch.cuda.empty_cache()
                        torch.cuda.synchronize()
                    except Exception:
                        pass


        POSE_INFER_BATCH = (
            original_batch
        )


        if not success:

            unrecovered_rows.append({
                **row.to_dict(),

                "error":
                    repr(
                        last_error
                    )
                    if last_error is not None
                    else "unknown_error",

                "error_type":
                    type(
                        last_error
                    ).__name__
                    if last_error is not None
                    else "unknown",

                "retry_reason":
                    resolution_info.get(
                        "reason"
                    ),

                "resolution_info":
                    str(
                        resolution_info
                    ),

                "retry_video":
                    str(
                        retry_video
                    ),

                "retry_full_clip":
                    bool(
                        use_full_clip
                    ),
            })


    # ========================================================
    # 6. Retry any non-MCFD rows conservatively.
    #
    # In the current failure pattern these should normally be zero.
    # We retry them using the canonical extractor without changing
    # their annotation interval.
    # ========================================================

    if not non_mcfd_retry_df.empty:

        for row_idx, row in tqdm(
            non_mcfd_retry_df.iterrows(),
            total=len(
                non_mcfd_retry_df
            ),
            desc="Retry remaining held-out pose cache",
        ):

            BUDGET.guard(
                "heldout_pose_retry",
                essential=True,
            )


            try:

                out_path, cache_hit = (
                    extract_one_pose(
                        row
                    )
                )


                recovered_rows.append({
                    **row.to_dict(),

                    "pose_path":
                        str(
                            out_path
                        ),

                    "pose_cache_hit":
                        bool(
                            cache_hit
                        ),

                    "pose_cache_version":
                        CACHE_VERSION,

                    "pose_retry":
                        True,

                    "pose_retry_batch":
                        globals().get(
                            "POSE_INFER_BATCH",
                            32,
                        ),

                    "pose_retry_reason":
                        "canonical_retry",

                    "pose_retry_video":
                        str(
                            row.video_path
                        ),

                    "pose_retry_full_clip":
                        False,
                })


            except BudgetStop:

                raise


            except Exception as exc:

                unrecovered_rows.append({
                    **row.to_dict(),

                    "error":
                        repr(
                            exc
                        ),

                    "error_type":
                        type(
                            exc
                        ).__name__,

                    "retry_reason":
                        "canonical_retry_failed",
                })


# ============================================================
# 7. Restore the original batch setting.
# ============================================================

if (
    "_original_pose_infer_batch"
    in globals()
):

    POSE_INFER_BATCH = (
        _original_pose_infer_batch
    )


# ============================================================
# 8. Merge original successful rows with recovered rows.
# ============================================================

successful_manifest_rows = (
    heldout_pose.to_dict(
        orient="records"
    )
)

successful_manifest_rows.extend(
    recovered_rows
)


heldout_pose_final = pd.DataFrame(
    successful_manifest_rows
)


# ------------------------------------------------------------
# Remove accidental duplicate logical rows deterministically.
# ------------------------------------------------------------

if not heldout_pose_final.empty:

    pose_identity_columns = [
        column
        for column
        in _POSE_ID_COLUMNS
        if column
        in heldout_pose_final.columns
    ]

    heldout_pose_final = (
        heldout_pose_final
        .drop_duplicates(
            subset=pose_identity_columns,
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 9. Persist the complete held-out pose manifest.
# ============================================================

heldout_pose_final.to_csv(
    TEST_POSE_PATH,
    index=False,
)


# ============================================================
# 10. Persist retry diagnostics.
# ============================================================

heldout_retry_error_path = (
    DIRS["logs"]
    / "heldout_test_pose_retry_errors.csv"
)

if unrecovered_rows:

    pd.DataFrame(
        unrecovered_rows
    ).to_csv(
        heldout_retry_error_path,
        index=False,
    )

else:

    try:

        heldout_retry_error_path.unlink(
            missing_ok=True
        )

    except Exception:
        pass


heldout_retry_success_path = (
    DIRS["logs"]
    / "heldout_test_pose_retry_success.csv"
)

if recovered_rows:

    pd.DataFrame(
        recovered_rows
    ).to_csv(
        heldout_retry_success_path,
        index=False,
    )

else:

    try:

        heldout_retry_success_path.unlink(
            missing_ok=True
        )

    except Exception:
        pass


# ============================================================
# 11. Final held-out validation.
# ============================================================

if heldout_pose_final.empty:

    raise RuntimeError(
        "Held-out test pose extraction produced no usable rows "
        "after canonical extraction and targeted retry."
    )


if len(
    heldout_pose_final
) != len(
    heldout_test_df
):

    failed_examples = (
        unrecovered_rows[:8]
    )

    raise RuntimeError(
        "Held-out test pose extraction is incomplete after retry. "
        f"Cached={len(heldout_pose_final)} | "
        f"Expected={len(heldout_test_df)} | "
        f"Unrecovered={len(unrecovered_rows)}. "
        f"First unresolved examples={failed_examples}. "
        f"Full retry log={heldout_retry_error_path}"
    )


# ------------------------------------------------------------
# Make the downstream variable use the validated final manifest.
# ------------------------------------------------------------

heldout_pose = (
    heldout_pose_final
)


print()
print("=" * 70)
print("HELD-OUT TEST POSE CACHE")
print("=" * 70)

print(
    "Expected held-out rows       :",
    len(heldout_test_df),
)

print(
    "Final cached pose rows       :",
    len(heldout_pose),
)

print(
    "Canonical extraction rows    :",
    len(
        heldout_pose_final
    )
    - len(
        recovered_rows
    ),
)

print(
    "Recovered retry rows         :",
    len(
        recovered_rows
    ),
)

print(
    "Remaining failed rows        :",
    len(
        unrecovered_rows
    ),
)

print(
    "Retry success log            :",
    heldout_retry_success_path,
)

print(
    "Retry error log              :",
    heldout_retry_error_path,
)

print("=" * 70)


# ============================================================
# 12. MCFD: official cross-view test split.
# ============================================================

mcfd_test = heldout_pose[
    (
        heldout_pose["dataset"]
        == "MCFD"
    )
    & (
        heldout_pose["split"]
        == "test"
    )
].copy()


if len(
    mcfd_test
) > 0:

    loader = make_loader(
        mcfd_test,
        final_mask,
        augment=False,
        shuffle=False,
    )

    mcfd_d, _, _ = eval_loader(final_model,loader,threshold=final_threshold)

    cross_rows.append({
        "dataset":
            "MCFD",

        "evaluation":
            "cross_view_test",

        "interpretation":
            "view_shift_not_subject_independent",

        "parameters":
            model_params(
                final_model
            ),

        **mcfd_d,
    })


# ============================================================
# 13. GMDCSA24: held-out subject test.
# ============================================================

gmd_holdout = heldout_pose[
    (
        heldout_pose["dataset"]
        == "GMDCSA24"
    )
    & (
        heldout_pose["split"]
        == "test"
    )
].copy()


if len(
    gmd_holdout
) > 0:

    loader = make_loader(
        gmd_holdout,
        final_mask,
        augment=False,
        shuffle=False,
    )

    gmd_d, _, _ = eval_loader(final_model,loader,threshold=final_threshold)

    cross_rows.append({
        "dataset":
            "GMDCSA24",

        "evaluation":
            "held_out_subject_test",

        "interpretation":
            "subject_disjoint_within_dataset",

        "parameters":
            model_params(
                final_model
            ),

        **gmd_d,
    })


# ============================================================
# 14. Save final cross-domain results.
# ============================================================

cross_results_df = pd.DataFrame(
    cross_rows
)


if cross_results_df.empty:

    raise RuntimeError(
        "No held-out test evaluation results were produced."
    )


cross_results_df.to_csv(
    RESULT_DIR
    / "cross_domain_results.csv",
    index=False,
)


print()
print("=" * 70)
print("HELD-OUT TEST EVALUATION COMPLETE")
print("=" * 70)
print(
    cross_results_df[
        [
            "dataset",
            "evaluation",
            "accuracy",
            "precision",
            "recall",
            "f1",
        ]
    ]
    if all(
        column in cross_results_df.columns
        for column in [
            "dataset",
            "evaluation",
            "accuracy",
            "precision",
            "recall",
            "f1",
        ]
    )
    else cross_results_df
)
print("=" * 70)

## 15. External CAUCAFall test - no tuning, no training

The final model is frozen before CAUCAFall is touched. CAUCAFall is evaluated only as an external dataset; the decision threshold is the threshold selected on the internal validation split and then frozen, so no CAUCAFall tuning occurs.

In [ ]:
# ============================================================
# External CAUCAFall evaluation
# ============================================================
#
# CAUCAFall remains completely external. Its pose features are generated
# only after the final model has been frozen.
# ============================================================

def extract_external_caucafall(
    force=False
):
    ext = (
        MANIFESTS
        .get(
            "CAUCAFall",
            pd.DataFrame()
        )
        .copy()
    )

    if ext.empty:
        return pd.DataFrame()

    manifest_path = (
        DIRS["metadata"]
        / "caucafall_pose_manifest.csv"
    )

    # Reuse a verified complete cached manifest.
    if (
        not force
        and manifest_path.exists()
    ):

        try:

            cached = pd.read_csv(
                manifest_path
            )

            required = {
                "pose_path",
                "video_path",
                "start_s",
                "end_s",
                "label",
                "subject",
                "camera",
            }

            if (
                required.issubset(
                    cached.columns
                )
                and len(cached) == len(ext)
                and cached["pose_path"].map(
                    lambda p: Path(str(p)).exists()
                ).all()
            ):
                print(
                    "[CACHE HIT] CAUCAFall pose manifest:",
                    len(cached)
                )

                return cached

        except Exception:
            pass

    out = ensure_pose_manifest(
        ext,
        manifest_path,
        desc="CAUCAFall pose cache",
        essential=True,
        checkpoint_every=20,
    )

    return out


if (
    ENABLE_CAUCAFALL
    and BUDGET.guard(
        "external_CAUCAFall_pose",
        essential=True
    )
):

    cauc_pose = extract_external_caucafall()

    if not cauc_pose.empty:

        loader = make_loader(
            cauc_pose,
            final_mask,
            augment=False,
            shuffle=False,
        )

        cauc_d, y_c, p_c = eval_loader(final_model,loader,threshold=final_threshold)

        cauca_results = {
            "dataset":
                "CAUCAFall",

            "evaluation":
                "external_cross_dataset",

            "interpretation":
                "unseen_dataset_environment",

            "parameters":
                model_params(
                    final_model
                ),

            **cauc_d,
        }

        previous_cross = (
            pd.read_csv(
                RESULT_DIR
                / "cross_domain_results.csv"
            )
            if (
                RESULT_DIR
                / "cross_domain_results.csv"
            ).exists()
            else pd.DataFrame()
        )

        cross = pd.concat(
            [
                previous_cross,
                pd.DataFrame(
                    [cauca_results]
                ),
            ],
            ignore_index=True,
        )

        cross.to_csv(
            RESULT_DIR
            / "cross_domain_results.csv",
            index=False
        )


## 16. Hard-negative analysis

The key diagnostic is not overall accuracy. For each non-fall activity, the notebook reports the number of examples, false alarms, false-alarm rate, and mean predicted fall probability. This table is computed on the external CAUCAFall test where possible, and on the subject-held-out GMDCSA24 test as a secondary diagnostic.

In [ ]:
# ============================================================
# Hard-negative activity breakdown
# ============================================================
#
# The analysis reports false alarms on non-fall activities.
#
# Note:
# `HARD_NEGATIVE_RULES` was referenced by the original function but
# was not defined anywhere in the notebook. To keep this repair local
# and avoid modifying unrelated cells, the rules are defined here.
#
# Only label==0 examples go into the final table.
# ============================================================


# ------------------------------------------------------------
# Local hard-negative taxonomy.
#
# The rules use the manifest's human-readable `activity`
# field. The ordering is deterministic, with specific rules checked before generic ones.
# ------------------------------------------------------------

HARD_NEGATIVE_RULES = {
    "walk":
        re.compile(
            r"(?<![a-z])walk(?:ing)?(?![a-z])",
            flags=re.I,
        ),

    "sit_down":
        re.compile(
            r"(?<![a-z])sit[_ -]?down(?![a-z])",
            flags=re.I,
        ),

    "sitting":
        re.compile(
            r"(?<![a-z])sitt(?:ing)?(?![a-z])",
            flags=re.I,
        ),

    "lie_down":
        re.compile(
            r"(?<![a-z])lie[_ -]?down(?![a-z])",
            flags=re.I,
        ),

    "lying":
        re.compile(
            r"(?<![a-z])lying(?![a-z])",
            flags=re.I,
        ),

    "stand_up":
        re.compile(
            r"(?<![a-z])stand[_ -]?up(?![a-z])",
            flags=re.I,
        ),

    "standing":
        re.compile(
            r"(?<![a-z])standing(?![a-z])",
            flags=re.I,
        ),

    "kneel_down":
        re.compile(
            r"(?<![a-z])kneel[_ -]?down(?![a-z])",
            flags=re.I,
        ),

    "kneeling":
        re.compile(
            r"(?<![a-z])kneeling(?![a-z])",
            flags=re.I,
        ),

    "squat_down":
        re.compile(
            r"(?<![a-z])squat[_ -]?down(?![a-z])",
            flags=re.I,
        ),

    "squatting":
        re.compile(
            r"(?<![a-z])squatting(?![a-z])",
            flags=re.I,
        ),

    "crawl":
        re.compile(
            r"(?<![a-z])crawl(?:ing)?(?![a-z])",
            flags=re.I,
        ),

    "jump":
        re.compile(
            r"(?<![a-z])jump(?:ing)?(?![a-z])",
            flags=re.I,
        ),

    "other":
        re.compile(
            r"(?<![a-z])other(?![a-z])",
            flags=re.I,
        ),
}


# ------------------------------------------------------------
# Empty-result schema.
# ------------------------------------------------------------

HARD_NEGATIVE_COLUMNS = [
    "activity",
    "number_of_examples",
    "false_alarms",
    "false_alarm_rate",
    "mean_predicted_fall_probability",
]


# ------------------------------------------------------------
# Hard-negative aggregation.
# ------------------------------------------------------------

def hard_negative_table(
    df,
    probs,
):
    d = (
        df
        .reset_index(
            drop=True
        )
        .copy()
    )

    probs = np.asarray(
        probs
    ).reshape(
        -1
    )


    # --------------------------------------------------------
    # Strict length validation.
    # --------------------------------------------------------

    if len(d) != len(
        probs
    ):

        raise ValueError(
            "Prediction/row length mismatch in hard-negative analysis: "
            f"rows={len(d)} | probabilities={len(probs)}"
        )


    if "label" not in d.columns:

        raise RuntimeError(
            "Hard-negative analysis requires a 'label' column."
        )


    if "activity" not in d.columns:

        raise RuntimeError(
            "Hard-negative analysis requires an 'activity' column."
        )


    # --------------------------------------------------------
    # Prediction columns.
    # --------------------------------------------------------

    d["pred_fall_prob"] = (
        probs.astype(
            np.float32,
            copy=False,
        )
    )

    d["pred"] = (
        d["pred_fall_prob"]
        >= 0.5
    )

    d["false_alarm"] = (
        (
            d["label"]
            == 0
        )
        & d["pred"]
    )


    # --------------------------------------------------------
    # Canonicalize activity names.
    # --------------------------------------------------------

    def canon(activity):

        text = str(
            activity
        ).strip()

        for (
            category,
            regex,
        ) in HARD_NEGATIVE_RULES.items():

            if regex.search(
                text
            ):

                return category

        return None


    d["hard_category"] = (
        d["activity"]
        .map(
            canon
        )
    )


    # --------------------------------------------------------
    # Keep only non-fall hard negatives with a recognized activity.
    # --------------------------------------------------------

    d = d[
        (
            d["label"]
            == 0
        )
        & d["hard_category"].notna()
    ].copy()


    if d.empty:

        return pd.DataFrame(
            columns=HARD_NEGATIVE_COLUMNS
        )


    # --------------------------------------------------------
    # Aggregate.
    # --------------------------------------------------------

    out = (
        d
        .groupby(
            "hard_category",
            sort=True,
        )
        .agg(
            number_of_examples=(
                "label",
                "size",
            ),

            false_alarms=(
                "false_alarm",
                "sum",
            ),

            mean_predicted_fall_probability=(
                "pred_fall_prob",
                "mean",
            ),
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # False-alarm rate.
    # --------------------------------------------------------

    out["false_alarm_rate"] = (
        out["false_alarms"]
        / out["number_of_examples"]
    )


    out = out.rename(
        columns={
            "hard_category":
                "activity",
        }
    )


    # --------------------------------------------------------
    # Stable output order.
    # --------------------------------------------------------

    out = out[
        HARD_NEGATIVE_COLUMNS
    ].copy()


    return out


# ============================================================
# Compute hard-negative analysis
# ============================================================

hard = []


# ------------------------------------------------------------
# CAUCAFall external hard negatives.
# ------------------------------------------------------------

if (
    ENABLE_CAUCAFALL
    and "cauc_pose" in globals()
    and cauc_pose is not None
    and len(cauc_pose) > 0
):

    loader = make_loader(
        cauc_pose,
        final_mask,
        augment=False,
        shuffle=False,
    )

    _, _, probs = eval_loader(final_model,loader,threshold=final_threshold)

    hard_table = hard_negative_table(
        cauc_pose,
        probs,
    )

    if not hard_table.empty:

        hard_table.insert(
            0,
            "dataset",
            "CAUCAFall",
        )

        hard.append(
            hard_table
        )


# ------------------------------------------------------------
# GMDCSA24 held-out subject hard negatives.
# ------------------------------------------------------------

if (
    "heldout_pose" in globals()
    and heldout_pose is not None
    and len(heldout_pose) > 0
):

    gmd_test = heldout_pose[
        (
            heldout_pose["dataset"]
            == "GMDCSA24"
        )
        & (
            heldout_pose["split"]
            == "test"
        )
    ].copy()

else:

    gmd_test = pd.DataFrame()


if not gmd_test.empty:

    loader = make_loader(
        gmd_test,
        final_mask,
        augment=False,
        shuffle=False,
    )

    _, _, probs = eval_loader(final_model,loader,threshold=final_threshold)

    hard_table = hard_negative_table(
        gmd_test,
        probs,
    )

    if not hard_table.empty:

        hard_table.insert(
            0,
            "dataset",
            "GMDCSA24",
        )

        hard.append(
            hard_table
        )


# ============================================================
# Final table
# ============================================================

if hard:

    hard_df = pd.concat(
        hard,
        ignore_index=True,
    )

else:

    hard_df = pd.DataFrame(
        columns=[
            "dataset",
            *HARD_NEGATIVE_COLUMNS,
        ]
    )


# ------------------------------------------------------------
# Persist results.
# ------------------------------------------------------------

hard_output_path = (
    RESULT_DIR
    / "hard_negative_activity_breakdown.csv"
)


hard_df.to_csv(
    hard_output_path,
    index=False,
)


# ------------------------------------------------------------
# Print results.
# ------------------------------------------------------------

print(
    "=" * 70
)

print(
    "HARD-NEGATIVE ACTIVITY BREAKDOWN"
)

print(
    "=" * 70
)

print(
    hard_df
    if not hard_df.empty
    else "No recognized non-fall hard-negative activities were available."
)

print(
    "=" * 70
)

print(
    "Saved:",
    hard_output_path,
)

## 17. Plots and confusion matrix

In [ ]:
# Validation comparison.
vr=pd.DataFrame(RUNS)
if not vr.empty:
    plt.figure(figsize=(10,5)); x=np.arange(len(vr));
    plt.bar(x,vr['val_calibrated_accuracy'].values); plt.xticks(x,vr['model'].values,rotation=45,ha='right'); plt.ylabel('Validation Accuracy (calibrated threshold)'); plt.title('Validation model comparison - primary selection metric'); plt.tight_layout(); plt.savefig(RESULT_DIR/'validation_comparison.png',dpi=180); plt.show()

# External confusion matrix.
if ENABLE_CAUCAFALL and 'cauc_pose' in globals() and len(cauc_pose):
    loader=make_loader(cauc_pose,final_mask,augment=False,shuffle=False); _,yy,pp=eval_loader(final_model,loader,threshold=final_threshold); yp=(pp>=final_threshold).astype(int); cm=confusion_matrix(yy,yp,labels=[0,1])
    plt.figure(figsize=(5,4)); plt.imshow(cm); plt.xticks([0,1],['Non-fall','Fall']); plt.yticks([0,1],['Non-fall','Fall']);
    for (i,j),v in np.ndenumerate(cm): plt.text(j,i,str(v),ha='center',va='center')
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('CAUCAFall confusion matrix'); plt.tight_layout(); plt.savefig(RESULT_DIR/'confusion_matrix_caucafall.png',dpi=180); plt.show()

if not hard_df.empty:
    plt.figure(figsize=(9,5)); plt.bar(np.arange(len(hard_df)),hard_df['false_alarm_rate']); plt.xticks(np.arange(len(hard_df)),hard_df['activity']); plt.ylabel('False-alarm rate'); plt.title('Hard-negative false alarms'); plt.tight_layout(); plt.savefig(RESULT_DIR/'hard_negative_false_alarm.png',dpi=180); plt.show()


## 18. Final metrics and reproducibility metadata

No number is inserted into the report unless it came from a completed evaluation. Runtime is measured from the persistent budget state. Dataset archive checksums, model configuration, split policy, package versions, GPU information, counts, and errors are all captured.

In [ ]:
# Gather reproducibility metadata.
run_meta={
    'seed':SEED,
    'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    'runtime_seconds_total_measured':BUDGET.elapsed_total,
    'runtime_target_seconds':3*3600,
    'runtime_hard_ceiling_seconds':3.83*3600,
    'gpu_operational_vram_limit_gib':MAX_OPERATIONAL_VRAM_GIB,
    'project_root':str(PROJECT_ROOT),
    'dataset_configs':DATASETS,
    'train_datasets':train_dataset_names,
    'external_test_datasets':external_dataset_names,
    'architecture':{
        'pose_model':POSE_MODEL_NAME,
        'pose_image_size': POSE_IMAGE_SIZE,
        'pose_infer_batch': POSE_INFER_BATCH,
        'pose_fp16': POSE_FP16,
        'pose_frozen':True,
        'keypoints':17,
        'sequence_length':32,
        'channels':['x','y','confidence','velocity_x','velocity_y','acceleration_x','acceleration_y','speed_magnitude'],
        'stgcn_hidden':48,'tcn_hidden':48,'attention_heads':2,
    },
    'training':CONFIG,
    'split_protocol':{
        'GMDCSA24':'Subject_1/2 train, Subject_3 validation, Subject_4 test',
        'MCFD':'versioned OmniFall cross-view split (cv); report strictly as view-shift, not subject-independent',
        'CAUCAFall':'external test only; no tuning',
    },
    'counts':{
        'manifest_rows':int(len(FULL_MANIFEST)),
        'trainval_pose_rows':int(len(POSE_MANIFEST)),
        'train':int((POSE_MANIFEST.split=='train').sum()),
        'val':int((POSE_MANIFEST.split=='val').sum()),
        'heldout_test_pose_rows':int(len(heldout_pose)) if 'heldout_pose' in globals() else 0,
        'test':int((heldout_pose.split=='test').sum()) if 'heldout_pose' in globals() else 0,
        'external_caucafall_pose_rows':int(len(cauc_pose)) if 'cauc_pose' in globals() else 0,
    },
    'system':{
        'python':sys.version,
        'pytorch':torch.__version__,'cuda':torch.version.cuda,
        'ultralytics':ultralytics.__version__,'opencv':cv2.__version__,
        'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'gpu_peak_allocated_gib':round(torch.cuda.max_memory_allocated()/2**30,3) if torch.cuda.is_available() else 0.0,
        'gpu_peak_reserved_gib':round(torch.cuda.max_memory_reserved()/2**30,3) if torch.cuda.is_available() else 0.0,
    },
    'chosen_model':chosen_name,
    'selection_rule':'max validation calibrated Accuracy; among models within 0.002 Accuracy points of the maximum, prefer higher validation F1, then lower validation false-positive rate, then fewer parameters',
    'final_threshold':float(final_threshold) if 'final_threshold' in globals() else None,
    'budget_file':str(BUDGET_FILE),
    'budget_decisions_file':str(BUDGET_DECISIONS_FILE),
}
(DIRS['metadata']/'run_metadata.json').write_text(json.dumps(run_meta,indent=2,default=str))
BUDGET.checkpoint('final_metadata')

all_metrics=RESULT_DIR/'all_metrics.csv'
frames=[]
if (RESULT_DIR/'validation_results.csv').exists():
    x=pd.read_csv(RESULT_DIR/'validation_results.csv'); x['evaluation']='within_domain_validation'; frames.append(x)
if (RESULT_DIR/'cross_domain_results.csv').exists():
    x=pd.read_csv(RESULT_DIR/'cross_domain_results.csv'); x['evaluation']=x.get('evaluation','cross_domain'); frames.append(x)
if frames: pd.concat(frames,ignore_index=True).to_csv(all_metrics,index=False)

print('Measured cumulative runtime:',round(BUDGET.elapsed_total/60,2),'minutes')
print('Target exceeded:',BUDGET.target_exceeded,'| hard remaining:',round(BUDGET.remaining_hard/60,2),'min')

## 19. Final lightweight results bundle

The ZIP contains the results, metadata, and checkpoints needed to reproduce and inspect this run. It intentionally excludes `datasets_zips/` and the raw extracted datasets.

In [43]:
BUNDLE=PROJECT_ROOT/'fall_detection_results_bundle.zip'
tmp_bundle=TEMP_ROOT/'bundle_contents'
if tmp_bundle.exists(): shutil.rmtree(tmp_bundle)
tmp_bundle.mkdir(parents=True)

for src_dir in [DIRS['results'],DIRS['metadata'],DIRS['models'],DIRS['logs']]:
    if src_dir.exists():
        shutil.copytree(src_dir,tmp_bundle/src_dir.name,dirs_exist_ok=True)

with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in tmp_bundle.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(tmp_bundle))
print('Bundle:',BUNDLE,'size GiB:',round(BUNDLE.stat().st_size/2**30,3))
print('Bundle members:',sum(1 for p in tmp_bundle.rglob('*') if p.is_file()))

Bundle: /content/drive/MyDrive/fall_detection_project/fall_detection_results_bundle.zip size GiB: 0.007
Bundle members: 62


## 20. Final scientific interpretation guardrails

- **Do not report a “SOTA” claim.** This notebook is an engineered, reproducible confidence-aware skeleton pipeline.
- **Do not call MCFD cross-view performance “subject-independent.”** MCFD has very limited subject diversity and is mainly evidence for view shift.
- **Do not tune on CAUCAFall.** It is the external cross-dataset test.
- **Accuracy is the primary optimization target here, but do not interpret it without false-positive/false-negative rates, hard negatives, and PR-AUC.**
- **Do not claim the 3-hour target was met unless `run_metadata.json` shows measured runtime <= 10,800 seconds. The hard engineering ceiling is 4 hours.**
- **When a budget guard skips an optional experiment, report the skip in the run log rather than treating the missing result as a negative result.**

### Required output files

`results/validation_results.csv`  
`results/cross_domain_results.csv`  
`results/ablation_results.csv`  
`results/all_metrics.csv`  
`results/hard_negative_activity_breakdown.csv`  
`results/validation_comparison.png`  
`results/confusion_matrix_caucafall.png` (when applicable)  
`results/hard_negative_false_alarm.png` (when applicable)  
`metadata/full_manifest.csv`  
`metadata/pose_manifest.csv`  
`metadata/run_metadata.json`  
`fall_detection_results_bundle.zip`

In [44]:
# ============================================================
# Final cell — check, package, and automatically download
# ============================================================

from pathlib import Path

EXPECTED_OUTPUTS = [
    RESULT_DIR/'validation_results.csv',
    RESULT_DIR/'cross_domain_results.csv',
    RESULT_DIR/'ablation_results.csv',
    RESULT_DIR/'all_metrics.csv',
    RESULT_DIR/'hard_negative_activity_breakdown.csv',
    RESULT_DIR/'validation_comparison.png',
    DIRS['metadata']/'full_manifest.csv',
    DIRS['metadata']/'pose_manifest.csv',
    DIRS['metadata']/'test_pose_manifest.csv',
    DIRS['metadata']/'run_metadata.json',
]

# Optional outputs are allowed to be absent; required core outputs are not.
OPTIONAL_OUTPUTS = {
    RESULT_DIR/'confusion_matrix_caucafall.png',
    RESULT_DIR/'hard_negative_false_alarm.png',
}

def build_and_verify_final_bundle():
    BUNDLE=PROJECT_ROOT/'fall_detection_results_bundle.zip'
    tmp_bundle=TEMP_ROOT/'bundle_contents_final'
    if tmp_bundle.exists():
        shutil.rmtree(tmp_bundle)
    tmp_bundle.mkdir(parents=True,exist_ok=True)

    core_required=EXPECTED_OUTPUTS
    missing=[str(p) for p in core_required if not Path(p).exists()]
    if missing:
        raise FileNotFoundError('Required outputs are missing: ' + '; '.join(missing))

    for src_dir in [DIRS['results'],DIRS['metadata'],DIRS['models'],DIRS['logs']]:
        if src_dir.exists():
            shutil.copytree(src_dir,tmp_bundle/src_dir.name,dirs_exist_ok=True)

    with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(tmp_bundle.rglob('*')):
            if p.is_file():
                z.write(p,p.relative_to(tmp_bundle))

    if not BUNDLE.exists() or BUNDLE.stat().st_size<=0:
        raise IOError(f'Final bundle was not created correctly: {BUNDLE}')

    verify_missing=[str(p) for p in core_required if not Path(p).exists()]
    optional_missing=[str(p) for p in sorted(OPTIONAL_OUTPUTS) if not p.exists()]

    summary={
        'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
        'bundle':str(BUNDLE),
        'bundle_bytes':BUNDLE.stat().st_size,
        'required_missing':verify_missing,
        'optional_missing':optional_missing,
        'core_output_count':len([p for p in core_required if Path(p).exists()]),
        'optional_output_count':len(OPTIONAL_OUTPUTS)-len(optional_missing),
        'measured_runtime_seconds':BUDGET.elapsed_total,
        'hard_ceiling_seconds':4*3600,
    }
    (DIRS['metadata']/'final_output_verification.json').write_text(json.dumps(summary,indent=2,default=str))

    if verify_missing:
        raise RuntimeError('Final verification failed: required outputs disappeared before packaging.')
    return BUNDLE,summary

BUNDLE,FINAL_SUMMARY=build_and_verify_final_bundle()

# Mark the run complete only after artifact verification succeeds.
RUN_COMPLETE_FILE.write_text(json.dumps({
    'timestamp_utc':FINAL_SUMMARY['timestamp_utc'],
    'bundle':str(BUNDLE),
    'measured_runtime_seconds':FINAL_SUMMARY['measured_runtime_seconds'],
},indent=2))

print('=== FINAL OUTPUT VERIFICATION ===')
print('Required outputs verified:',FINAL_SUMMARY['core_output_count'])
print('Optional outputs present:',FINAL_SUMMARY['optional_output_count'])
print('Missing optional outputs:',FINAL_SUMMARY['optional_missing'])
print('Measured runtime (min):',round(FINAL_SUMMARY['measured_runtime_seconds']/60,2))
print('4-hour ceiling remaining (min):',round((4*3600-FINAL_SUMMARY['measured_runtime_seconds'])/60,2))
print('Bundle:',BUNDLE)

# Automatic Colab download. Outside Colab, the artifact is still saved and reported.
try:
    from google.colab import files as colab_files
    colab_files.download(str(BUNDLE))
    print('Colab download initiated.')
except ImportError:
    print('[INFO] google.colab.files is unavailable in this environment; bundle remains saved at:',BUNDLE)


=== FINAL OUTPUT VERIFICATION ===
Required outputs verified: 10
Optional outputs present: 2
Missing optional outputs: []
Measured runtime (min): 29.92
4-hour ceiling remaining (min): 210.08
Bundle: /content/drive/MyDrive/fall_detection_project/fall_detection_results_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Colab download initiated.
